# AS-HMM-BO: Adaptive-State HMM-Guided Bayesian Optimization

**Author:** Sourav Guha  
**Project:** Deep-learning-based image segmentation on the Warwick GlaS dataset  

This notebook contains the end-to-end experimental workflow: dataset inspection, preprocessing, U-Net segmentation, validation and cross-validation, HMM-guided adaptive Bayesian optimization, optimizer baselines, ablations, and result analysis.

> **Before running:** Obtain the Warwick GlaS dataset separately and place it under `data/raw/Warwick_QU_Dataset/`. See the repository README. Outputs and execution counts have been cleared so the notebook opens cleanly and does not publish machine-specific results.


# Cell 1 — Imports and environment inspection

In [ ]:

# Cell 1: Imports and Environment Inspection

import os
import sys
import platform
import random
import importlib.metadata as metadata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Project paths
PROJECT_ROOT = Path.cwd().resolve()
DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / "Warwick_QU_Dataset"

# Reproducibility defaults
SEED = 42

print("=" * 65)
print("AS-HMM-BO: ENVIRONMENT INSPECTION")
print("=" * 65)

print(f"Python       : {sys.version.split()[0]}")
print(f"Platform     : {platform.platform()}")
print(f"Working dir  : {PROJECT_ROOT}")
print(f"Dataset path : {DATASET_ROOT}")
print(f"PyTorch      : {torch.__version__}")
print(f"NumPy        : {np.__version__}")
print(f"Pandas       : {pd.__version__}")
print(f"Pillow       : {Image.__version__}")

print(f"\nCUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(
        f"GPU memory    : "
        f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB"
    )
else:
    print("Device        : CPU")

print("\nOptional research dependencies:")

for package in [
    "torchvision",
    "optuna",
    "scikit-learn",
    "scipy",
    "tqdm",
    "PyYAML",
    "hmmlearn",
]:
    try:
        print(f"{package:15}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package:15}: NOT INSTALLED")

print("\nDataset directory exists:", DATASET_ROOT.is_dir())

if not DATASET_ROOT.is_dir():
    print(
        "\nWARNING: Dataset path not found. "
        "Check the notebook's working directory before proceeding."
    )


# Cell 2 — Resolve the project root and dataset path

In [ ]:

# Cell 2: Project Root and Dataset Path Resolution

CURRENT_DIR = Path.cwd().resolve()

# If the notebook is inside "notebooks", its parent is the project root.
if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / "Warwick_QU_Dataset"

print("Current directory :", CURRENT_DIR)
print("Project root      :", PROJECT_ROOT)
print("Expected dataset  :", DATASET_ROOT)

if DATASET_ROOT.is_dir():
    print("\nSUCCESS: Dataset directory found.")
else:
    print("\nDataset not found at the expected location.")

    alternatives = [
        CURRENT_DIR / "data" / "raw" / "Warwick_QU_Dataset",
        PROJECT_ROOT / "Warwick_QU_Dataset",
        PROJECT_ROOT / "data" / "Warwick_QU_Dataset",
    ]

    found = [path for path in alternatives if path.is_dir()]

    if found:
        print("\nPossible dataset locations:")
        for path in found:
            print(" -", path)
    else:
        print(
            "\nNo dataset found in the checked locations. "
            "Verify the folder location in File Explorer."
        )

assert PROJECT_ROOT.exists(), "Project root does not exist."

print("\nPath verification complete.")


# Cell 3 — Reproducibility configuration

In [ ]:

# Cell 3: Reproducibility and Experiment Configuration

import json
import random

SEED = 42

# Initial compute configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0  # Conservative default for Windows notebooks
PIN_MEMORY = DEVICE.type == "cuda"

# Initial dataset and training configuration
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 2
INITIAL_EPOCHS = 2  # Smoke tests only; not the final research budget

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Request deterministic behavior where supported.
torch.use_deterministic_algorithms(True, warn_only=True)

# Record the initial configuration for later experiments.
EXPERIMENT_CONFIG = {
    "project": "AS-HMM-BO",
    "version": "0.1.0-dev",
    "seed": SEED,
    "device": str(DEVICE),
    "dataset": "Warwick_QU_Dataset",
    "image_size": list(IMAGE_SIZE),
    "batch_size": BATCH_SIZE,
    "initial_epochs": INITIAL_EPOCHS,
    "num_workers": NUM_WORKERS,
    "pin_memory": PIN_MEMORY,
    "test_set_used_for_optimization": False,
}

print(json.dumps(EXPERIMENT_CONFIG, indent=2))


# Cell 4 — Dataset file inventory

In [ ]:

# Cell 4: Dataset File Inventory

assert DATASET_ROOT.is_dir(), f"Dataset directory not found: {DATASET_ROOT}"

all_files = sorted(
    path for path in DATASET_ROOT.rglob("*")
    if path.is_file()
)

print(f"Dataset root: {DATASET_ROOT}")
print(f"Total files : {len(all_files)}")

# Summarize file extensions
extension_counts = {}

for path in all_files:
    extension = path.suffix.lower() or "<no extension>"
    extension_counts[extension] = extension_counts.get(extension, 0) + 1

extension_df = pd.DataFrame(
    [
        {"extension": ext, "file_count": count}
        for ext, count in sorted(extension_counts.items())
    ]
).sort_values("file_count", ascending=False)

display(extension_df)

print("\nTop-level contents:")

for path in sorted(DATASET_ROOT.iterdir()):
    if path.is_dir():
        count = sum(1 for p in path.rglob("*") if p.is_file())
        print(f"[DIR]  {path.name}/ ({count} files)")
    else:
        print(f"[FILE] {path.name}")

print("\nFirst 40 files:")

for path in all_files[:40]:
    print(path.relative_to(DATASET_ROOT))


# Cell 5 — Detect likely images, masks, and metadata

In [ ]:

# Cell 5: Preliminary Image, Mask, and Metadata Discovery

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

image_files = [
    path for path in all_files
    if path.suffix.lower() in IMAGE_EXTENSIONS
]

metadata_files = [
    path for path in all_files
    if path.suffix.lower() in {".csv", ".tsv", ".json", ".xlsx"}
]

def classify_filename(path):
    name = path.stem.lower()

    if any(token in name for token in ["mask", "annotation", "groundtruth", "ground_truth", "_gt"]):
        return "possible_mask"
    if "grade" in name:
        return "possible_metadata"
    return "unclassified_image_or_file"

classification_df = pd.DataFrame([
    {
        "relative_path": str(path.relative_to(DATASET_ROOT)),
        "filename": path.name,
        "classification": classify_filename(path),
        "extension": path.suffix.lower(),
        "size_kb": round(path.stat().st_size / 1024, 2),
    }
    for path in all_files
])

print("Image-format files:", len(image_files))
print("Metadata candidates:", len(metadata_files))

display(
    classification_df[
        classification_df["classification"] != "unclassified_image_or_file"
    ].head(50)
)

print("\nMetadata files:")
for path in metadata_files:
    print(" -", path.relative_to(DATASET_ROOT))

print("\nRepresentative image-format filenames:")
for path in image_files[:30]:
    print(" -", path.relative_to(DATASET_ROOT))


# Cell 6 — Build and validate the image-mask manifest

In [ ]:

# Cell 6: Build and Validate the GlaS Image-Mask Manifest

import re

IMAGE_PATTERN = re.compile(r"^(train|testA|testB)_(\d+)\.bmp$", re.IGNORECASE)

records = []
unrecognized_images = []

for path in sorted(DATASET_ROOT.glob("*.bmp")):
    match = IMAGE_PATTERN.match(path.name)

    if match:
        split_name = match.group(1)
        sample_id = int(match.group(2))

        annotation_path = DATASET_ROOT / (
            f"{split_name}_{sample_id}_anno.bmp"
        )

        records.append({
            "sample_id": f"{split_name}_{sample_id}",
            "source_split": split_name,
            "index": sample_id,
            "image_path": str(path),
            "mask_path": (
                str(annotation_path) if annotation_path.is_file() else None
            ),
            "image_exists": path.is_file(),
            "mask_exists": annotation_path.is_file(),
        })

    elif not path.stem.lower().endswith("_anno"):
        unrecognized_images.append(path.name)

manifest = pd.DataFrame(records)

if manifest.empty:
    raise RuntimeError(
        "No original images matched the expected GlaS naming convention."
    )

manifest = manifest.sort_values(
    ["source_split", "index"],
    key=lambda col: col.str.lower() if col.name == "source_split" else col,
).reset_index(drop=True)

print("Image counts by original partition:")
display(
    manifest.groupby("source_split")
    .agg(
        images=("sample_id", "count"),
        existing_images=("image_exists", "sum"),
        existing_masks=("mask_exists", "sum"),
    )
)

missing_masks = manifest.loc[~manifest["mask_exists"]]
missing_images = manifest.loc[~manifest["image_exists"]]

print("\nTotal original images:", len(manifest))
print("Missing annotations :", len(missing_masks))
print("Missing images      :", len(missing_images))
print("Unrecognized files  :", unrecognized_images)

duplicate_ids = manifest["sample_id"].duplicated().sum()
print("Duplicate sample IDs:", duplicate_ids)

if not missing_masks.empty:
    display(missing_masks[["sample_id", "image_path", "mask_path"]])

assert duplicate_ids == 0, "Duplicate sample IDs detected."
assert missing_images.empty, "One or more image files are missing."
assert missing_masks.empty, "One or more image annotations are missing."
assert not unrecognized_images, "Unexpected original image filenames detected."

expected_counts = {"train": 85, "testA": 60, "testB": 20}

actual_counts = manifest.groupby("source_split").size().to_dict()

print("\nExpected partition counts:", expected_counts)
print("Actual partition counts  :", actual_counts)

if actual_counts == expected_counts:
    print("PARTITION COUNTS MATCH THE HISTORICAL RECORD.")
else:
    print("WARNING: Partition counts differ from the historical record.")

print("\nManifest validation completed.")


# Cell 7 — Inspect image and annotation formats

In [ ]:

# Cell 7: Inspect Training Images and Annotation Encoding

train_manifest = manifest.loc[
    manifest["source_split"].str.lower() == "train"
].copy()

if train_manifest.empty:
    raise RuntimeError("No training images found in the manifest.")

sample_count = min(5, len(train_manifest))
sample_rows = train_manifest.head(sample_count)

inspection_records = []

for _, row in sample_rows.iterrows():
    with Image.open(row["image_path"]) as image:
        image_info = {
            "sample_id": row["sample_id"],
            "image_size": image.size,
            "image_mode": image.mode,
            "image_format": image.format,
        }

    with Image.open(row["mask_path"]) as mask:
        mask_array = np.array(mask)
        unique_values = np.unique(mask_array.reshape(-1, mask_array.shape[-1])
                                  if mask_array.ndim == 3
                                  else mask_array)

        mask_info = {
            "mask_size": mask.size,
            "mask_mode": mask.mode,
            "mask_format": mask.format,
            "mask_shape": tuple(mask_array.shape),
            "mask_dtype": str(mask_array.dtype),
            "unique_values_count": len(unique_values),
            "unique_values_sample": unique_values[:12].tolist(),
        }

    inspection_records.append({**image_info, **mask_info})

display(pd.DataFrame(inspection_records))

# Visualize one training image and its corresponding annotation.
sample = sample_rows.iloc[0]

with Image.open(sample["image_path"]) as image:
    image_array = np.array(image.convert("RGB"))

with Image.open(sample["mask_path"]) as mask:
    mask_array = np.array(mask)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image_array)
axes[0].set_title(f"Original image: {sample['sample_id']}")
axes[0].axis("off")

if mask_array.ndim == 2:
    axes[1].imshow(mask_array, cmap="gray")
else:
    axes[1].imshow(mask_array)
axes[1].set_title("Original annotation")
axes[1].axis("off")

axes[2].imshow(image_array)
if mask_array.ndim == 2:
    axes[2].imshow(mask_array > 0, cmap="Reds", alpha=0.4)
else:
    axes[2].imshow(mask_array, alpha=0.4)
axes[2].set_title("Overlay preview")
axes[2].axis("off")

plt.tight_layout()
plt.show()


# Cell 8 — Inspect Grade.csv and dataset metadata

In [ ]:

# Cell 8: Inspect Dataset Metadata

GRADE_PATH = DATASET_ROOT / "Grade.csv"

if not GRADE_PATH.is_file():
    raise FileNotFoundError(f"Metadata file not found: {GRADE_PATH}")

grades_df = pd.read_csv(GRADE_PATH)

print("Metadata file:", GRADE_PATH.name)
print("Rows        :", len(grades_df))
print("Columns     :", grades_df.columns.tolist())
print("\nData types:")
print(grades_df.dtypes)

print("\nFirst 10 rows:")
display(grades_df.head(10))

print("\nMissing values per column:")
display(grades_df.isna().sum().rename("missing_count").to_frame())

print("\nMetadata summary:")
display(grades_df.describe(include="all").T)

print("\nDataset metadata inspection complete.")


# Cell 9 — Quantify annotation encoding on training images

In [ ]:

# Cell 9: Training Annotation Encoding Audit

from collections import Counter

mask_audit_records = []
global_value_counts = Counter()

for _, row in train_manifest.iterrows():
    with Image.open(row["mask_path"]) as mask_image:
        mask = np.asarray(mask_image.convert("L"), dtype=np.uint8)

    values, counts = np.unique(mask, return_counts=True)

    global_value_counts.update({
        int(value): int(count)
        for value, count in zip(values, counts)
    })

    mask_audit_records.append({
        "sample_id": row["sample_id"],
        "height": mask.shape[0],
        "width": mask.shape[1],
        "min_value": int(mask.min()),
        "max_value": int(mask.max()),
        "unique_value_count": len(values),
        "unique_values": values.tolist(),
        "nonzero_fraction": float(np.mean(mask != 0)),
        "fraction_value_1": float(np.mean(mask == 1)),
        "fraction_value_255": float(np.mean(mask == 255)),
    })

mask_audit_df = pd.DataFrame(mask_audit_records)

print("Training annotation count:", len(mask_audit_df))
print("\nAnnotation dimensions:")
display(mask_audit_df[["height", "width"]].value_counts().rename("count").to_frame())

print("\nMask encoding summary:")
display(
    mask_audit_df[
        [
            "sample_id",
            "min_value",
            "max_value",
            "unique_value_count",
            "nonzero_fraction",
        ]
    ].head(15)
)

print("\nDistinct grayscale values observed in training masks:")
print(sorted(global_value_counts.keys()))

print("\nMost frequent grayscale values:")
display(
    pd.DataFrame([
        {"pixel_value": value, "pixel_count": count}
        for value, count in global_value_counts.most_common(20)
    ])
)

print("\nNonzero pixel fraction across training annotations:")
display(mask_audit_df["nonzero_fraction"].describe().to_frame())

# Visualize a few training annotations with their original pixel values.
n_show = min(4, len(train_manifest))
fig, axes = plt.subplots(n_show, 2, figsize=(10, 3.5 * n_show))

if n_show == 1:
    axes = np.array([axes])

for i, (_, row) in enumerate(train_manifest.head(n_show).iterrows()):
    with Image.open(row["image_path"]) as image:
        image_array = np.asarray(image.convert("RGB"))

    with Image.open(row["mask_path"]) as mask_image:
        mask_array = np.asarray(mask_image.convert("L"))

    axes[i, 0].imshow(image_array)
    axes[i, 0].set_title(f"Image: {row['sample_id']}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(mask_array, cmap="nipy_spectral", vmin=0)
    axes[i, 1].set_title(
        f"Annotation: {row['sample_id']} "
        f"(max={mask_array.max()}, unique={len(np.unique(mask_array))})"
    )
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()


# Cell 10 — Verify instance-label semantics and foreground masks

In [ ]:

# Cell 10: Verify Instance Labels and Binary Foreground Masks

from collections import Counter

instance_audit_records = []
instance_count_distribution = Counter()

for _, row in train_manifest.iterrows():
    with Image.open(row["mask_path"]) as mask_image:
        instance_map = np.asarray(mask_image.convert("L"), dtype=np.uint8)

    instance_ids, instance_pixel_counts = np.unique(
        instance_map, return_counts=True
    )

    gland_ids = instance_ids[instance_ids != 0]
    binary_mask = instance_map > 0

    instance_count_distribution[len(gland_ids)] += 1

    instance_audit_records.append({
        "sample_id": row["sample_id"],
        "instance_count": len(gland_ids),
        "background_fraction": float(np.mean(~binary_mask)),
        "gland_fraction": float(np.mean(binary_mask)),
        "smallest_instance_pixels": (
            int(instance_pixel_counts[instance_ids != 0].min())
            if len(gland_ids) else 0
        ),
        "largest_instance_pixels": (
            int(instance_pixel_counts[instance_ids != 0].max())
            if len(gland_ids) else 0
        ),
        "instance_ids": gland_ids.tolist(),
    })

instance_audit_df = pd.DataFrame(instance_audit_records)

print("Training images audited:", len(instance_audit_df))
print("Total annotated gland instances:",
      int(instance_audit_df["instance_count"].sum()))
print("\nInstance-count summary:")
display(instance_audit_df["instance_count"].describe().to_frame())
print("\nImages grouped by number of gland instances:")
display(
    instance_audit_df["instance_count"]
    .value_counts()
    .sort_index()
    .rename_axis("instance_count")
    .to_frame("image_count")
)
print("\nExample instance labels:")
display(
    instance_audit_df[
        ["sample_id", "instance_count", "gland_fraction", "instance_ids"]
    ].head(10)
)


# Cell 11 — Visualize instance labels and binary masks

In [ ]:

# Cell 11: Compare Original Instance Maps and Binary Masks

sample_ids = ["train_1", "train_3", "train_9", "train_14"]

fig, axes = plt.subplots(len(sample_ids), 3, figsize=(15, 5 * len(sample_ids)))

for row_index, sample_id in enumerate(sample_ids):
    sample = train_manifest.loc[
        train_manifest["sample_id"] == sample_id
    ].iloc[0]

    with Image.open(sample["image_path"]) as image_file:
        image = np.asarray(image_file.convert("RGB"))

    with Image.open(sample["mask_path"]) as mask_file:
        instance_map = np.asarray(mask_file.convert("L"), dtype=np.uint8)

    binary_mask = instance_map > 0
    overlay = image.copy()
    overlay[binary_mask] = (
        0.5 * overlay[binary_mask] + 0.5 * np.array([0, 255, 0])
    ).astype(np.uint8)

    axes[row_index, 0].imshow(image)
    axes[row_index, 0].set_title(f"{sample_id}: Original image")

    axes[row_index, 1].imshow(instance_map, cmap="nipy_spectral")
    axes[row_index, 1].set_title("Instance annotation")

    axes[row_index, 2].imshow(overlay)
    axes[row_index, 2].set_title("Binary foreground overlay")

    for ax in axes[row_index]:
        ax.axis("off")

plt.tight_layout()
plt.show()


# Cell 12 — Create reproducible train/validation splits

In [ ]:

# Cell 12: Reproducible Train/Validation Split

from sklearn.model_selection import train_test_split

train_ids, val_ids = train_test_split(
    train_manifest["sample_id"].tolist(),
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
)

train_ids = sorted(train_ids, key=lambda x: int(x.split("_")[1]))
val_ids = sorted(val_ids, key=lambda x: int(x.split("_")[1]))

fit_manifest = (
    train_manifest[
        train_manifest["sample_id"].isin(train_ids)
    ]
    .sort_values("sample_id", key=lambda s: s.str.extract(r"_(\d+)")[0].astype(int))
    .reset_index(drop=True)
)

val_manifest = (
    train_manifest[
        train_manifest["sample_id"].isin(val_ids)
    ]
    .sort_values("sample_id", key=lambda s: s.str.extract(r"_(\d+)")[0].astype(int))
    .reset_index(drop=True)
)

assert len(fit_manifest) == 68
assert len(val_manifest) == 17
assert set(fit_manifest["sample_id"]).isdisjoint(
    set(val_manifest["sample_id"])
)
assert set(fit_manifest["sample_id"]) | set(val_manifest["sample_id"]) == set(
    train_manifest["sample_id"]
)

print("Optimization-training images:", len(fit_manifest))
print("Validation images:", len(val_manifest))
print("TestA images reserved:", int((manifest["source_split"] == "testA").sum()))
print("TestB images reserved:", int((manifest["source_split"] == "testB").sum()))
print("Training IDs:", fit_manifest["sample_id"].tolist())
print("Validation IDs:", val_manifest["sample_id"].tolist())


# Cell 13 — Implement the segmentation dataset

In [ ]:

# Cell 13: GlaS Segmentation Dataset

import torch.nn.functional as F

class GlaSSegmentationDataset(Dataset):
    def __init__(self, dataframe, image_size=IMAGE_SIZE):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.image_size = tuple(image_size)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(row["image_path"]) as image_file:
            image = image_file.convert("RGB").resize(
                (self.image_size[1], self.image_size[0]),
                resample=Image.Resampling.BILINEAR,
            )
            image_array = np.asarray(image, dtype=np.float32) / 255.0

        with Image.open(row["mask_path"]) as mask_file:
            instance_map = mask_file.convert("L").resize(
                (self.image_size[1], self.image_size[0]),
                resample=Image.Resampling.NEAREST,
            )
            instance_array = np.asarray(instance_map, dtype=np.uint8)

        binary_array = (instance_array > 0).astype(np.float32)

        image_tensor = torch.from_numpy(
            np.ascontiguousarray(image_array.transpose(2, 0, 1))
        )
        binary_mask_tensor = torch.from_numpy(
            np.ascontiguousarray(binary_array[None, ...])
        )
        instance_mask_tensor = torch.from_numpy(
            np.ascontiguousarray(instance_array.astype(np.int64))
        )

        return {
            "image": image_tensor,
            "mask": binary_mask_tensor,
            "instance_mask": instance_mask_tensor,
            "sample_id": row["sample_id"],
        }


train_dataset = GlaSSegmentationDataset(fit_manifest)
val_dataset = GlaSSegmentationDataset(val_manifest)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))

sample = train_dataset[0]
for key, value in sample.items():
    if torch.is_tensor(value):
        print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}")
    else:
        print(f"{key}: {value}")

assert sample["image"].shape == (3, *IMAGE_SIZE)
assert sample["mask"].shape == (1, *IMAGE_SIZE)
assert sample["instance_mask"].shape == IMAGE_SIZE
assert set(torch.unique(sample["mask"]).tolist()).issubset({0.0, 1.0})


# Cell 14 — Validate the preprocessing visually

In [ ]:

# Cell 14: Inspect Resized Samples

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for i in range(3):
    sample = train_dataset[i]

    image = sample["image"].permute(1, 2, 0).numpy()
    binary_mask = sample["mask"][0].numpy()
    instance_mask = sample["instance_mask"].numpy()

    axes[i, 0].imshow(image)
    axes[i, 0].set_title(f'{sample["sample_id"]}: Image')

    axes[i, 1].imshow(binary_mask, cmap="gray", vmin=0, vmax=1)
    axes[i, 1].set_title("Binary gland mask")

    axes[i, 2].imshow(instance_mask, cmap="nipy_spectral")
    axes[i, 2].set_title("Preserved instance labels")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()


# Cell 15 — Create DataLoaders

In [ ]:

# Cell 15: Create Training and Validation DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

train_batch = next(iter(train_loader))
val_batch = next(iter(val_loader))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

for name, batch in [("Training", train_batch), ("Validation", val_batch)]:
    print(f"\n{name} batch")
    print("Images:", tuple(batch["image"].shape))
    print("Masks:", tuple(batch["mask"].shape))
    print("Instance masks:", tuple(batch["instance_mask"].shape))
    print("Sample IDs:", batch["sample_id"])

    assert batch["image"].shape[1:] == (3, *IMAGE_SIZE)
    assert batch["mask"].shape[1:] == (1, *IMAGE_SIZE)

print("\nDataLoader checks passed.")


# Cell 16 — Implement a compact U-Net

In [ ]:

# Cell 16: Compact U-Net Architecture

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.layers(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, base_channels=16):
        super().__init__()

        self.enc1 = DoubleConv(in_channels, base_channels)
        self.enc2 = DoubleConv(base_channels, base_channels * 2)
        self.enc3 = DoubleConv(base_channels * 2, base_channels * 4)
        self.enc4 = DoubleConv(base_channels * 4, base_channels * 8)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = DoubleConv(base_channels * 8, base_channels * 16)

        self.up4 = nn.ConvTranspose2d(
            base_channels * 16, base_channels * 8, kernel_size=2, stride=2
        )
        self.dec4 = DoubleConv(base_channels * 16, base_channels * 8)

        self.up3 = nn.ConvTranspose2d(
            base_channels * 8, base_channels * 4, kernel_size=2, stride=2
        )
        self.dec3 = DoubleConv(base_channels * 8, base_channels * 4)

        self.up2 = nn.ConvTranspose2d(
            base_channels * 4, base_channels * 2, kernel_size=2, stride=2
        )
        self.dec2 = DoubleConv(base_channels * 4, base_channels * 2)

        self.up1 = nn.ConvTranspose2d(
            base_channels * 2, base_channels, kernel_size=2, stride=2
        )
        self.dec1 = DoubleConv(base_channels * 2, base_channels)

        self.output_conv = nn.Conv2d(base_channels, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        bottleneck = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(bottleneck), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))

        return self.output_conv(d1)


model = UNet(base_channels=16).to(DEVICE)

parameter_count = sum(parameter.numel() for parameter in model.parameters())
trainable_count = sum(
    parameter.numel() for parameter in model.parameters()
    if parameter.requires_grad
)

print("Model:", model.__class__.__name__)
print("Device:", DEVICE)
print(f"Total parameters: {parameter_count:,}")
print(f"Trainable parameters: {trainable_count:,}")


# Cell 17 — Verify the forward and backward passes

In [ ]:

# Cell 17: Smoke-Test Forward and Backward Passes

model.train()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

images = train_batch["image"].to(DEVICE)
masks = train_batch["mask"].to(DEVICE)

optimizer.zero_grad(set_to_none=True)

logits = model(images)
loss = criterion(logits, masks)
loss.backward()

gradient_tensors = sum(
    parameter.grad is not None
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Input shape:", tuple(images.shape))
print("Logits shape:", tuple(logits.shape))
print("Target shape:", tuple(masks.shape))
print("Smoke-test loss:", float(loss.detach()))
print("Parameters with gradients:", gradient_tensors)
print("All logits finite:", bool(torch.isfinite(logits).all()))
print("Loss finite:", bool(torch.isfinite(loss)))

assert logits.shape == masks.shape
assert torch.isfinite(logits).all()
assert torch.isfinite(loss)
assert gradient_tensors > 0

print("\nForward/backward smoke test passed.")


# Cell 18 — Implement binary segmentation metrics

In [ ]:

# Cell 18: Binary Segmentation Metrics

@torch.no_grad()
def calculate_segmentation_metrics(logits, targets, threshold=0.5, epsilon=1e-7):
    probabilities = torch.sigmoid(logits)
    predictions = probabilities >= threshold
    targets = targets >= 0.5

    predictions = predictions.bool()
    targets = targets.bool()

    dims = (1, 2, 3)

    true_positive = (predictions & targets).sum(dim=dims).float()
    false_positive = (predictions & ~targets).sum(dim=dims).float()
    false_negative = (~predictions & targets).sum(dim=dims).float()

    dice = (
        (2.0 * true_positive + epsilon)
        / (2.0 * true_positive + false_positive + false_negative + epsilon)
    )

    iou = (
        (true_positive + epsilon)
        / (true_positive + false_positive + false_negative + epsilon)
    )

    precision = (
        (true_positive + epsilon)
        / (true_positive + false_positive + epsilon)
    )

    recall = (
        (true_positive + epsilon)
        / (true_positive + false_negative + epsilon)
    )

    return {
        "dice": dice.mean().item(),
        "iou": iou.mean().item(),
        "precision": precision.mean().item(),
        "recall": recall.mean().item(),
    }


# Sanity-check the metrics on a perfect prediction.
perfect_targets = torch.tensor(
    [[[[0.0, 1.0], [1.0, 0.0]]]]
)
perfect_logits = torch.where(
    perfect_targets.bool(),
    torch.tensor(10.0),
    torch.tensor(-10.0),
)

perfect_metrics = calculate_segmentation_metrics(
    perfect_logits, perfect_targets
)

print("Perfect-prediction metric check:", perfect_metrics)

assert all(abs(value - 1.0) < 1e-5 for value in perfect_metrics.values())
print("Metric sanity check passed.")


# Cell 19 — Define the training and validation epoch functions

In [ ]:

# Cell 19: Loss Function and Epoch Routines

def combined_segmentation_loss(logits, targets, dice_weight=0.5, epsilon=1e-7):
    bce_loss = F.binary_cross_entropy_with_logits(logits, targets)

    probabilities = torch.sigmoid(logits)
    dims = (1, 2, 3)

    intersection = (probabilities * targets).sum(dim=dims)
    denominator = probabilities.sum(dim=dims) + targets.sum(dim=dims)

    soft_dice = (
        2.0 * intersection + epsilon
    ) / (
        denominator + epsilon
    )

    dice_loss = 1.0 - soft_dice.mean()

    return (1.0 - dice_weight) * bce_loss + dice_weight * dice_loss


def run_epoch(model, loader, optimizer=None, threshold=0.5):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_samples = 0
    metric_sums = {
        "dice": 0.0,
        "iou": 0.0,
        "precision": 0.0,
        "recall": 0.0,
    }

    context = torch.enable_grad() if training else torch.no_grad()

    with context:
        for batch in loader:
            images = batch["image"].to(DEVICE)
            masks = batch["mask"].to(DEVICE)

            if training:
                optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            loss = combined_segmentation_loss(logits, masks)

            if training:
                loss.backward()
                optimizer.step()

            batch_size = images.size(0)
            total_loss += loss.detach().item() * batch_size
            total_samples += batch_size

            batch_metrics = calculate_segmentation_metrics(
                logits.detach(), masks, threshold=threshold
            )

            for key in metric_sums:
                metric_sums[key] += batch_metrics[key] * batch_size

    if total_samples == 0:
        raise ValueError("Cannot evaluate an empty DataLoader.")

    results = {
        "loss": total_loss / total_samples,
        **{
            key: value / total_samples
            for key, value in metric_sums.items()
        },
    }

    return results


# Cell 20 — Run a short baseline experiment

In [ ]:

# Cell 20: Initial Baseline Training

import copy
import time

OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_CONFIG = {
    "model": "UNet",
    "base_channels": 16,
    "optimizer": "Adam",
    "learning_rate": 1e-3,
    "weight_decay": 1e-5,
    "loss": "BCE + Soft Dice",
    "epochs": INITIAL_EPOCHS,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
}

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=BASELINE_CONFIG["learning_rate"],
    weight_decay=BASELINE_CONFIG["weight_decay"],
)

history = []
best_val_dice = -1.0
best_epoch = None
best_state_dict = None

for epoch in range(1, INITIAL_EPOCHS + 1):
    epoch_start = time.time()

    train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
    val_metrics = run_epoch(model, val_loader, optimizer=None)

    record = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
        "seconds": time.time() - epoch_start,
    }
    history.append(record)

    print(
        f"Epoch {epoch}/{INITIAL_EPOCHS} | "
        f"train loss={train_metrics['loss']:.4f}, "
        f"train Dice={train_metrics['dice']:.4f} | "
        f"val loss={val_metrics['loss']:.4f}, "
        f"val Dice={val_metrics['dice']:.4f}, "
        f"val IoU={val_metrics['iou']:.4f} | "
        f"{record['seconds']:.1f}s"
    )

    if val_metrics["dice"] > best_val_dice:
        best_val_dice = val_metrics["dice"]
        best_epoch = epoch
        best_state_dict = copy.deepcopy(model.state_dict())

        checkpoint_path = CHECKPOINT_DIR / "unet_baseline_best.pt"
        torch.save(
            {
                "epoch": best_epoch,
                "model_state_dict": best_state_dict,
                "model_config": {"base_channels": 16},
                "baseline_config": BASELINE_CONFIG,
                "validation_metrics": val_metrics,
                "seed": SEED,
            },
            checkpoint_path,
        )

history_df = pd.DataFrame(history)

print("\nBest validation epoch:", best_epoch)
print(f"Best validation Dice: {best_val_dice:.4f}")
print("Checkpoint:", checkpoint_path.resolve())

display(history_df)


# Cell 21 — Restore and verify the best checkpoint

In [ ]:

# Cell 21: Restore Best Checkpoint and Verify Validation Metrics

checkpoint = torch.load(
    CHECKPOINT_DIR / "unet_baseline_best.pt",
    map_location=DEVICE,
    weights_only=True,
)

best_model = UNet(
    base_channels=checkpoint["model_config"]["base_channels"]
).to(DEVICE)

best_model.load_state_dict(checkpoint["model_state_dict"])

restored_metrics = run_epoch(
    best_model,
    val_loader,
    optimizer=None,
)

print("Checkpoint epoch:", checkpoint["epoch"])
print("Recorded validation metrics:", checkpoint["validation_metrics"])
print("Recomputed validation metrics:", restored_metrics)

for metric_name in ("loss", "dice", "iou", "precision", "recall"):
    recorded = checkpoint["validation_metrics"][metric_name]
    recomputed = restored_metrics[metric_name]
    assert abs(recorded - recomputed) < 1e-5, (
        f"{metric_name} mismatch: {recorded} vs {recomputed}"
    )

print("\nCheckpoint restoration verified.")


# Cell 22 — Diagnose prediction coverage on validation images

In [ ]:

# Cell 22: Per-Image Validation Diagnostics

best_model.eval()
diagnostic_records = []

with torch.no_grad():
    for sample in val_dataset:
        image = sample["image"].unsqueeze(0).to(DEVICE)
        target = sample["mask"].unsqueeze(0).to(DEVICE)

        logits = best_model(image)
        probabilities = torch.sigmoid(logits)
        prediction = probabilities >= 0.5
        target_binary = target >= 0.5

        metrics = calculate_segmentation_metrics(logits, target)

        diagnostic_records.append({
            "sample_id": sample["sample_id"],
            "target_foreground_fraction": target_binary.float().mean().item(),
            "predicted_foreground_fraction": prediction.float().mean().item(),
            "mean_probability": probabilities.mean().item(),
            **metrics,
        })

val_diagnostics_df = pd.DataFrame(diagnostic_records)

print("Validation images:", len(val_diagnostics_df))
print("\nAggregate foreground coverage:")
display(
    val_diagnostics_df[
        ["target_foreground_fraction", "predicted_foreground_fraction"]
    ].describe().T
)

print("\nLowest-Dice validation samples:")
display(
    val_diagnostics_df.sort_values("dice").head(8).round(4)
)

print("\nHighest-Dice validation samples:")
display(
    val_diagnostics_df.sort_values("dice", ascending=False).head(5).round(4)
)

print("\nMean per-image validation metrics:")
display(
    val_diagnostics_df[
        ["dice", "iou", "precision", "recall"]
    ].mean().to_frame("mean")
)


# Cell 23 — Visualize predictions and probability maps

In [ ]:

# Cell 23: Visual Prediction Diagnostics

best_model.eval()

sample_ids_to_show = ["train_67", "train_55", "train_1", "train_11"]
fig, axes = plt.subplots(len(sample_ids_to_show), 4, figsize=(16, 4 * len(sample_ids_to_show)))

for i, sample_id in enumerate(sample_ids_to_show):
    sample = next(
        item for item in val_dataset
        if item["sample_id"] == sample_id
    )

    image = sample["image"].unsqueeze(0).to(DEVICE)
    target = sample["mask"][0].numpy()

    with torch.no_grad():
        probabilities = torch.sigmoid(best_model(image))[0, 0].cpu().numpy()

    prediction = probabilities >= 0.5

    axes[i, 0].imshow(sample["image"].permute(1, 2, 0).numpy())
    axes[i, 0].set_title(f"{sample_id}: Image")

    axes[i, 1].imshow(target, cmap="gray", vmin=0, vmax=1)
    axes[i, 1].set_title("Ground truth")

    axes[i, 2].imshow(probabilities, cmap="viridis", vmin=0, vmax=1)
    axes[i, 2].set_title("Predicted probability")

    axes[i, 3].imshow(prediction, cmap="gray", vmin=0, vmax=1)
    axes[i, 3].set_title("Prediction at 0.5")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.show()


# Cell 24 — Measure probability calibration and threshold sensitivity

In [ ]:

# Cell 24: Validation Threshold Sensitivity

@torch.no_grad()
def collect_validation_probabilities(model, dataset):
    model.eval()
    records = []

    for sample in dataset:
        image = sample["image"].unsqueeze(0).to(DEVICE)
        target = sample["mask"].unsqueeze(0).to(DEVICE)

        probability = torch.sigmoid(model(image)).cpu()
        records.append({
            "sample_id": sample["sample_id"],
            "probabilities": probability,
            "target": target.cpu(),
        })

    return records


validation_predictions = collect_validation_probabilities(best_model, val_dataset)

threshold_records = []

for threshold in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95]:
    sample_metrics = []
    predicted_fractions = []

    for record in validation_predictions:
        probabilities = record["probabilities"]
        target = record["target"]

        predictions = probabilities >= threshold
        predicted_fractions.append(predictions.float().mean().item())

        # Convert thresholded predictions back to logits so the common
        # metric function can be reused without changing its interface.
        threshold_logits = torch.where(
            predictions,
            torch.tensor(10.0),
            torch.tensor(-10.0),
        )

        sample_metrics.append(
            calculate_segmentation_metrics(threshold_logits, target)
        )

    threshold_records.append({
        "threshold": threshold,
        "mean_dice": np.mean([m["dice"] for m in sample_metrics]),
        "mean_iou": np.mean([m["iou"] for m in sample_metrics]),
        "mean_precision": np.mean([m["precision"] for m in sample_metrics]),
        "mean_recall": np.mean([m["recall"] for m in sample_metrics]),
        "mean_predicted_foreground_fraction": np.mean(predicted_fractions),
    })

threshold_df = pd.DataFrame(threshold_records)
display(threshold_df.round(4))


# Cell 25 — Inspect probability distributions

In [ ]:

# Cell 25: Inspect Predicted Probability Distributions

all_probabilities = torch.cat([
    record["probabilities"].reshape(-1)
    for record in validation_predictions
]).numpy()

print("Probability quantiles:")
display(
    pd.Series(all_probabilities)
    .quantile([0, 0.01, 0.10, 0.25, 0.50, 0.75, 0.90, 0.99, 1.0])
    .rename("probability")
    .to_frame()
)

plt.figure(figsize=(9, 4))
plt.hist(all_probabilities, bins=50)
plt.xlabel("Predicted foreground probability")
plt.ylabel("Pixel count")
plt.title("Validation prediction probability distribution")
plt.tight_layout()
plt.show()


# Cell 26 — Add a foreground-aware loss

In [ ]:

# Cell 26: Estimate Training-Only Foreground Weight

foreground_pixels = 0
total_pixels = 0

for sample in train_dataset:
    mask = sample["mask"]
    foreground_pixels += int(mask.sum().item())
    total_pixels += mask.numel()

background_pixels = total_pixels - foreground_pixels
foreground_fraction = foreground_pixels / total_pixels

# Bound the weight to avoid extreme gradients on a small dataset.
POSITIVE_WEIGHT = float(
    np.clip(background_pixels / max(foreground_pixels, 1), 1.0, 5.0)
)

print(f"Training foreground fraction: {foreground_fraction:.4f}")
print(f"Training background fraction: {1.0 - foreground_fraction:.4f}")
print(f"Positive BCE weight: {POSITIVE_WEIGHT:.4f}")

assert 1.0 <= POSITIVE_WEIGHT <= 5.0


# Cell 27 — Train a fresh controlled baseline

In [ ]:

# Cell 27: Controlled Baseline Training

def foreground_aware_loss(logits, targets, positive_weight, dice_weight=0.5, epsilon=1e-7):
    pos_weight = torch.tensor(
        [positive_weight], dtype=logits.dtype, device=logits.device
    )

    bce = F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=pos_weight
    )

    probabilities = torch.sigmoid(logits)
    dims = (1, 2, 3)
    intersection = (probabilities * targets).sum(dim=dims)
    denominator = probabilities.sum(dim=dims) + targets.sum(dim=dims)

    soft_dice = (
        2.0 * intersection + epsilon
    ) / (
        denominator + epsilon
    )

    return (1.0 - dice_weight) * bce + dice_weight * (1.0 - soft_dice.mean())


def run_epoch_with_loss(model, loader, loss_function, optimizer=None, threshold=0.5):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_samples = 0
    metric_sums = {key: 0.0 for key in ("dice", "iou", "precision", "recall")}

    with torch.set_grad_enabled(training):
        for batch in loader:
            images = batch["image"].to(DEVICE)
            masks = batch["mask"].to(DEVICE)

            if training:
                optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            loss = loss_function(logits, masks)

            if training:
                loss.backward()
                optimizer.step()

            batch_size = images.size(0)
            total_loss += loss.detach().item() * batch_size
            total_samples += batch_size

            metrics = calculate_segmentation_metrics(
                logits.detach(), masks, threshold=threshold
            )
            for key in metric_sums:
                metric_sums[key] += metrics[key] * batch_size

    return {
        "loss": total_loss / total_samples,
        **{key: value / total_samples for key, value in metric_sums.items()},
    }


CONTROLLED_EPOCHS = 8
CONTROLLED_LR = 2e-4
CONTROLLED_WEIGHT_DECAY = 1e-4

# Seed before model initialization for a reproducible fresh model.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

controlled_model = UNet(base_channels=16).to(DEVICE)
controlled_optimizer = torch.optim.Adam(
    controlled_model.parameters(),
    lr=CONTROLLED_LR,
    weight_decay=CONTROLLED_WEIGHT_DECAY,
)

loss_fn = lambda logits, targets: foreground_aware_loss(
    logits, targets, positive_weight=POSITIVE_WEIGHT
)

controlled_history = []
controlled_best_dice = -1.0
controlled_best_epoch = None
controlled_best_state = None

for epoch in range(1, CONTROLLED_EPOCHS + 1):
    started = time.time()

    train_metrics = run_epoch_with_loss(
        controlled_model, train_loader, loss_fn, optimizer=controlled_optimizer
    )
    val_metrics = run_epoch_with_loss(
        controlled_model, val_loader, loss_fn, optimizer=None
    )

    record = {
        "epoch": epoch,
        **{f"train_{key}": value for key, value in train_metrics.items()},
        **{f"val_{key}": value for key, value in val_metrics.items()},
        "seconds": time.time() - started,
    }
    controlled_history.append(record)

    print(
        f"Epoch {epoch}/{CONTROLLED_EPOCHS} | "
        f"train loss={train_metrics['loss']:.4f}, "
        f"train Dice={train_metrics['dice']:.4f} | "
        f"val loss={val_metrics['loss']:.4f}, "
        f"val Dice={val_metrics['dice']:.4f}, "
        f"val recall={val_metrics['recall']:.4f} | "
        f"{record['seconds']:.1f}s"
    )

    if val_metrics["dice"] > controlled_best_dice:
        controlled_best_dice = val_metrics["dice"]
        controlled_best_epoch = epoch
        controlled_best_state = copy.deepcopy(controlled_model.state_dict())

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": controlled_best_state,
                "model_config": {"base_channels": 16},
                "config": {
                    "learning_rate": CONTROLLED_LR,
                    "weight_decay": CONTROLLED_WEIGHT_DECAY,
                    "epochs": CONTROLLED_EPOCHS,
                    "positive_weight": POSITIVE_WEIGHT,
                    "loss": "weighted BCE + soft Dice",
                    "seed": SEED,
                },
                "validation_metrics": val_metrics,
            },
            CHECKPOINT_DIR / "unet_controlled_best.pt",
        )

controlled_history_df = pd.DataFrame(controlled_history)

print("\nBest controlled epoch:", controlled_best_epoch)
print(f"Best controlled validation Dice: {controlled_best_dice:.4f}")
display(controlled_history_df)


# Cell 28 — Patient-Level Train/Validation Leakage Audit

In [ ]:

# Cell 28: Audit Patient-Level Train/Validation Leakage

grade_path = DATASET_ROOT / "Grade.csv"
grade_data = pd.read_csv(grade_path)

def normalize_sample_id(value):
    return Path(str(value).strip()).stem.lower()

grade_data["normalized_id"] = grade_data["name"].map(normalize_sample_id)
grade_data["patient ID"] = grade_data["patient ID"].astype(str).str.strip()

if grade_data["normalized_id"].duplicated().any():
    raise ValueError("Duplicate image identifiers found in Grade.csv.")

patient_lookup = grade_data.set_index("normalized_id")["patient ID"]

# Recover the actual image IDs from the existing dataset objects.
train_ids = [
    train_dataset[i]["sample_id"]
    for i in range(len(train_dataset))
]

# The original split contains 68 training and 17 validation images.
# Recover validation IDs from the manifest by excluding the training IDs.
train_id_set = {normalize_sample_id(x) for x in train_ids}

train_rows = manifest[
    manifest["sample_id"].map(normalize_sample_id).isin(train_id_set)
]
val_rows = manifest[
    (manifest["source_split"].str.lower() == "train")
    & (~manifest["sample_id"].map(normalize_sample_id).isin(train_id_set))
]

val_ids = val_rows["sample_id"].tolist()

if len(train_ids) != 68 or len(val_ids) != 17:
    raise ValueError(
        f"Unexpected split sizes: {len(train_ids)} train, "
        f"{len(val_ids)} validation. Check the split variables."
    )

train_patients = {
    patient_lookup.get(normalize_sample_id(sample_id))
    for sample_id in train_ids
}
val_patients = {
    patient_lookup.get(normalize_sample_id(sample_id))
    for sample_id in val_ids
}

if None in train_patients or None in val_patients:
    raise ValueError("Some sample IDs could not be matched to Grade.csv.")

overlap = train_patients & val_patients

print(f"Training images: {len(train_ids)}")
print(f"Validation images: {len(val_ids)}")
print(f"Unique training patients: {len(train_patients)}")
print(f"Unique validation patients: {len(val_patients)}")
print(f"Patients appearing in BOTH sets: {len(overlap)}")
print("Overlapping patient IDs:", sorted(overlap, key=str))

if overlap:
    print(
        "\nWARNING: Patient overlap exists. Rebuild the split by patient "
        "before treating validation scores as generalization evidence."
    )
else:
    print("\nNo patient overlap detected in this split.")


# Cell 29 — Create a Patient-Level Train/Validation Split

In [ ]:

# Cell 29: Create a Patient-Level Train/Validation Split

from sklearn.model_selection import GroupShuffleSplit

grade_data = pd.read_csv(DATASET_ROOT / "Grade.csv").copy()

def normalize_sample_id(value):
    return Path(str(value).strip()).stem.lower()

grade_data["normalized_id"] = grade_data["name"].map(normalize_sample_id)

if grade_data["normalized_id"].duplicated().any():
    raise ValueError("Duplicate image identifiers found in Grade.csv.")

patient_lookup = grade_data.set_index("normalized_id")["patient ID"]

# Start from all 85 images in the original training partition.
glas_train_manifest = manifest.loc[
    manifest["source_split"].str.lower() == "train"
].copy()

glas_train_manifest["normalized_id"] = (
    glas_train_manifest["sample_id"].map(normalize_sample_id)
)
glas_train_manifest["patient_id"] = (
    glas_train_manifest["normalized_id"].map(patient_lookup)
)

if glas_train_manifest["patient_id"].isna().any():
    missing = glas_train_manifest.loc[
        glas_train_manifest["patient_id"].isna(), "sample_id"
    ].tolist()
    raise ValueError(f"Missing patient metadata for: {missing}")

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_indices, val_indices = next(
    splitter.split(
        glas_train_manifest,
        groups=glas_train_manifest["patient_id"]
    )
)

patient_train_manifest = (
    glas_train_manifest.iloc[train_indices]
    .drop(columns=["normalized_id"])
    .reset_index(drop=True)
)

patient_val_manifest = (
    glas_train_manifest.iloc[val_indices]
    .drop(columns=["normalized_id"])
    .reset_index(drop=True)
)

train_patient_ids = set(patient_train_manifest["patient_id"])
val_patient_ids = set(patient_val_manifest["patient_id"])
overlapping_patients = train_patient_ids & val_patient_ids

print("Patient-level split summary")
print("----------------------------")
print(f"Original training-partition images: {len(glas_train_manifest)}")
print(f"Optimization-training images:       {len(patient_train_manifest)}")
print(f"Validation images:                   {len(patient_val_manifest)}")
print(f"Training patients:                   {len(train_patient_ids)}")
print(f"Validation patients:                 {len(val_patient_ids)}")
print(f"Overlapping patients:                {len(overlapping_patients)}")
print(f"Training patient IDs:                {sorted(train_patient_ids)}")
print(f"Validation patient IDs:              {sorted(val_patient_ids)}")

print("\nImages per patient in each split:")
display(
    pd.concat([
        patient_train_manifest.assign(split="train"),
        patient_val_manifest.assign(split="validation")
    ])
    .groupby(["split", "patient_id"])
    .size()
    .rename("image_count")
    .reset_index()
)

assert len(patient_train_manifest) + len(patient_val_manifest) == 85
assert len(overlapping_patients) == 0
assert set(patient_train_manifest["sample_id"]).isdisjoint(
    set(patient_val_manifest["sample_id"])
)

print("\nPASS: no patient appears in both training and validation.")


# Cell 30 — Rebuild Datasets and DataLoaders

In [ ]:

# Cell 30: Rebuild Datasets and DataLoaders Using Patient-Level Split

train_dataset = GlaSSegmentationDataset(patient_train_manifest)
validation_dataset = GlaSSegmentationDataset(patient_val_manifest)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

print("Patient-separated data pipeline")
print("--------------------------------")
print(f"Training dataset:   {len(train_dataset)} images")
print(f"Validation dataset: {len(validation_dataset)} images")
print(f"Training batches:   {len(train_loader)}")
print(f"Validation batches: {len(validation_loader)}")

# Verify sample IDs emitted by the actual datasets.
train_sample_ids = {
    normalize_sample_id(train_dataset[i]["sample_id"])
    for i in range(len(train_dataset))
}
val_sample_ids = {
    normalize_sample_id(validation_dataset[i]["sample_id"])
    for i in range(len(validation_dataset))
}

train_patient_set = set(patient_train_manifest["patient_id"])
val_patient_set = set(patient_val_manifest["patient_id"])

assert len(train_dataset) == 57
assert len(validation_dataset) == 28
assert train_sample_ids.isdisjoint(val_sample_ids)
assert train_patient_set.isdisjoint(val_patient_set)

sample_batch = next(iter(train_loader))
print("\nBatch tensor shapes:")
print("Images:", tuple(sample_batch["image"].shape))
print("Masks:", tuple(sample_batch["mask"].shape))
print("Instance masks:", tuple(sample_batch["instance_mask"].shape))

print("\nPASS: loaders use the patient-separated split.")


# Cell 31 — Retrain the Controlled Baseline

In [ ]:

# Cell 31: Controlled Baseline on Patient-Separated Split

import copy
import time

# Estimate foreground weighting using training pixels only.
foreground_pixels = 0
total_pixels = 0

for sample in train_dataset:
    mask = sample["mask"]
    foreground_pixels += int(mask.sum().item())
    total_pixels += mask.numel()

background_pixels = total_pixels - foreground_pixels
foreground_fraction = foreground_pixels / total_pixels
POSITIVE_WEIGHT = float(
    np.clip(background_pixels / max(foreground_pixels, 1), 1.0, 5.0)
)

print(f"Training foreground fraction: {foreground_fraction:.4f}")
print(f"Training background fraction: {1 - foreground_fraction:.4f}")
print(f"Positive BCE weight: {POSITIVE_WEIGHT:.4f}")

# Reset seeds before creating the new model.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

controlled_model = UNet(base_channels=16).to(DEVICE)

controlled_optimizer = torch.optim.Adam(
    controlled_model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)

loss_fn = lambda logits, targets: foreground_aware_loss(
    logits,
    targets,
    positive_weight=POSITIVE_WEIGHT
)

CONTROLLED_EPOCHS = 8
controlled_history = []
controlled_best_dice = -1.0
controlled_best_epoch = None
controlled_best_state = None

patient_checkpoint_path = (
    PROJECT_ROOT / "outputs" / "checkpoints"
    / "unet_patient_split_best.pt"
)
patient_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, CONTROLLED_EPOCHS + 1):
    start_time = time.time()

    train_metrics = run_epoch_with_loss(
        controlled_model,
        train_loader,
        loss_fn,
        optimizer=controlled_optimizer
    )

    val_metrics = run_epoch_with_loss(
        controlled_model,
        validation_loader,
        loss_fn,
        optimizer=None
    )

    elapsed = time.time() - start_time

    record = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
        "seconds": elapsed
    }
    controlled_history.append(record)

    print(
        f"Epoch {epoch}/{CONTROLLED_EPOCHS} | "
        f"train loss={train_metrics['loss']:.4f}, "
        f"train Dice={train_metrics['dice']:.4f} | "
        f"val loss={val_metrics['loss']:.4f}, "
        f"val Dice={val_metrics['dice']:.4f}, "
        f"val IoU={val_metrics['iou']:.4f} | "
        f"{elapsed:.1f}s"
    )

    if val_metrics["dice"] > controlled_best_dice:
        controlled_best_dice = val_metrics["dice"]
        controlled_best_epoch = epoch
        controlled_best_state = copy.deepcopy(
            controlled_model.state_dict()
        )

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": controlled_best_state,
                "validation_metrics": val_metrics,
                "config": {
                    "seed": SEED,
                    "base_channels": 16,
                    "learning_rate": 2e-4,
                    "weight_decay": 1e-4,
                    "positive_weight": POSITIVE_WEIGHT,
                    "epochs": CONTROLLED_EPOCHS,
                    "training_images": len(train_dataset),
                    "validation_images": len(validation_dataset),
                    "training_patients": sorted(train_patient_set),
                    "validation_patients": sorted(val_patient_set),
                }
            },
            patient_checkpoint_path
        )

controlled_history_df = pd.DataFrame(controlled_history)

print(f"\nBest epoch: {controlled_best_epoch}")
print(f"Best patient-separated validation Dice: {controlled_best_dice:.4f}")
print(f"Checkpoint: {patient_checkpoint_path}")

display(controlled_history_df.round(4))


# Cell 32 — Evaluate the Best Checkpoint by Patient

In [ ]:

# Cell 32: Patient-Level Validation Diagnostics

checkpoint = torch.load(
    patient_checkpoint_path,
    map_location=DEVICE,
    weights_only=True
)

controlled_model.load_state_dict(checkpoint["model_state_dict"])
controlled_model.eval()

patient_lookup = grade_data.set_index("normalized_id")["patient ID"]

image_records = []

with torch.no_grad():
    for batch in validation_loader:
        images = batch["image"].to(DEVICE)
        masks = batch["mask"].to(DEVICE)
        logits = controlled_model(images)
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).float()

        for i, sample_id in enumerate(batch["sample_id"]):
            target = masks[i:i + 1]
            prediction = predictions[i:i + 1]
            probability = probabilities[i:i + 1]

            intersection = (prediction * target).sum().item()
            pred_pixels = prediction.sum().item()
            target_pixels = target.sum().item()
            union = (
                (prediction + target) > 0
            ).sum().item()

            dice = (
                2 * intersection / (pred_pixels + target_pixels)
                if pred_pixels + target_pixels > 0 else 1.0
            )
            iou = intersection / union if union > 0 else 1.0
            precision = (
                intersection / pred_pixels if pred_pixels > 0 else 1.0
            )
            recall = (
                intersection / target_pixels if target_pixels > 0 else 1.0
            )

            normalized_id = normalize_sample_id(sample_id)
            patient_id = patient_lookup.loc[normalized_id]

            image_records.append({
                "sample_id": sample_id,
                "patient_id": patient_id,
                "dice": dice,
                "iou": iou,
                "precision": precision,
                "recall": recall,
                "target_foreground_fraction": (
                    target_pixels / target.numel()
                ),
                "predicted_foreground_fraction": (
                    pred_pixels / prediction.numel()
                ),
                "mean_probability": probability.mean().item()
            })

image_diagnostics = pd.DataFrame(image_records)

print("Image-level validation summary")
display(
    image_diagnostics[
        ["dice", "iou", "precision", "recall",
         "target_foreground_fraction", "predicted_foreground_fraction"]
    ].describe().round(4)
)

# Mean of per-image metrics within each patient.
patient_mean_metrics = (
    image_diagnostics
    .groupby("patient_id")
    .agg(
        image_count=("sample_id", "count"),
        mean_image_dice=("dice", "mean"),
        std_image_dice=("dice", "std"),
        mean_image_iou=("iou", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_target_foreground=("target_foreground_fraction", "mean"),
        mean_predicted_foreground=("predicted_foreground_fraction", "mean")
    )
    .reset_index()
)

# Pooled pixel-level Dice for each patient.
pooled_patient_records = []

for patient_id, group in image_diagnostics.groupby("patient_id"):
    sample_ids = set(group["sample_id"])
    intersection_total = 0.0
    prediction_total = 0.0
    target_total = 0.0

    with torch.no_grad():
        for batch in validation_loader:
            selected = [
                i for i, sid in enumerate(batch["sample_id"])
                if sid in sample_ids
            ]
            if not selected:
                continue

            images = batch["image"][selected].to(DEVICE)
            masks = batch["mask"][selected].to(DEVICE)
            predictions = (
                torch.sigmoid(controlled_model(images)) >= 0.5
            ).float()

            intersection_total += (predictions * masks).sum().item()
            prediction_total += predictions.sum().item()
            target_total += masks.sum().item()

    pooled_dice = (
        2 * intersection_total / (prediction_total + target_total)
        if prediction_total + target_total > 0 else 1.0
    )

    pooled_patient_records.append({
        "patient_id": patient_id,
        "pooled_pixel_dice": pooled_dice
    })

pooled_patient_metrics = pd.DataFrame(pooled_patient_records)

patient_metrics = patient_mean_metrics.merge(
    pooled_patient_metrics,
    on="patient_id",
    how="left"
).sort_values("patient_id")

print("\nValidation metrics by held-out patient")
display(patient_metrics.round(4))

print("\nLowest-Dice validation images")
display(
    image_diagnostics
    .sort_values("dice")
    .head(10)
    .round(4)
)

print("\nHighest-Dice validation images")
display(
    image_diagnostics
    .sort_values("dice", ascending=False)
    .head(10)
    .round(4)
)

print("\nCheckpoint epoch:", checkpoint["epoch"])
print("Checkpoint validation Dice:", checkpoint["validation_metrics"]["dice"])


# Cell 33 — Visualize Best and Worst Predictions

In [ ]:

# Cell 33: Visualize Predictions on Held-Out Patients

import math

# Include difficult cases and one strong example from each held-out patient.
selected_ids = []

for patient_id, group in image_diagnostics.groupby("patient_id"):
    selected_ids.extend(
        group.sort_values("dice").head(2)["sample_id"].tolist()
    )
    selected_ids.extend(
        group.sort_values("dice", ascending=False).head(1)["sample_id"].tolist()
    )

# Preserve order while removing duplicates.
selected_ids = list(dict.fromkeys(selected_ids))

# Map validation sample IDs to dataset positions.
validation_positions = {
    validation_dataset[i]["sample_id"]: i
    for i in range(len(validation_dataset))
}

controlled_model.eval()
num_columns = 5

for sample_id in selected_ids:
    sample = validation_dataset[validation_positions[sample_id]]
    image_tensor = sample["image"]
    target = sample["mask"]

    with torch.no_grad():
        logits = controlled_model(
            image_tensor.unsqueeze(0).to(DEVICE)
        )
        probability = torch.sigmoid(logits)[0, 0].cpu().numpy()

    image = image_tensor.permute(1, 2, 0).cpu().numpy()
    target_np = target[0].cpu().numpy()
    predicted_np = probability >= 0.5

    row = image_diagnostics.loc[
        image_diagnostics["sample_id"] == sample_id
    ].iloc[0]

    fig, axes = plt.subplots(
        1, num_columns, figsize=(17, 4)
    )

    axes[0].imshow(np.clip(image, 0, 1))
    axes[0].set_title(f"{sample_id}\nOriginal")

    axes[1].imshow(target_np, cmap="gray", vmin=0, vmax=1)
    axes[1].set_title("Ground truth")

    axes[2].imshow(predicted_np, cmap="gray", vmin=0, vmax=1)
    axes[2].set_title("Prediction @ 0.5")

    im = axes[3].imshow(
        probability, cmap="viridis", vmin=0, vmax=1
    )
    axes[3].set_title("Foreground probability")
    fig.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04)

    axes[4].imshow(np.clip(image, 0, 1))
    axes[4].imshow(
        predicted_np,
        cmap="Reds",
        alpha=0.35,
        vmin=0,
        vmax=1
    )
    axes[4].set_title("Prediction overlay")

    for ax in axes:
        ax.axis("off")

    fig.suptitle(
        f"Patient {row['patient_id']} | "
        f"Dice={row['dice']:.3f} | "
        f"Precision={row['precision']:.3f} | "
        f"Recall={row['recall']:.3f}",
        y=1.03
    )
    plt.tight_layout()
    plt.show()


# Cell 34 — Build Patient-Grouped Cross-Validation Folds

In [ ]:

# Cell 34: Create and Audit Patient-Grouped Cross-Validation Folds

from sklearn.model_selection import GroupKFold

# Rebuild the original training partition independently of the
# single train/validation split used for the baseline.
cv_manifest = manifest.loc[
    manifest["source_split"].str.lower() == "train"
].copy()

# Reuse the patient-ID mapping established from Grade.csv.
cv_manifest["normalized_id"] = cv_manifest["sample_id"].map(
    normalize_sample_id
)

patient_lookup = grade_data.set_index(
    "normalized_id"
)["patient ID"]

cv_manifest["patient_id"] = cv_manifest["normalized_id"].map(
    patient_lookup
)

assert cv_manifest["patient_id"].notna().all(), (
    "Some training images could not be matched to patient IDs."
)

cv_manifest["patient_id"] = cv_manifest["patient_id"].astype(int)

N_SPLITS = 5
group_kfold = GroupKFold(n_splits=N_SPLITS)

cv_folds = []
fold_audit_rows = []

for fold_number, (train_indices, val_indices) in enumerate(
    group_kfold.split(
        cv_manifest,
        groups=cv_manifest["patient_id"]
    ),
    start=1
):
    fold_train = cv_manifest.iloc[train_indices].copy().reset_index(drop=True)
    fold_val = cv_manifest.iloc[val_indices].copy().reset_index(drop=True)

    train_patients = set(fold_train["patient_id"])
    val_patients = set(fold_val["patient_id"])
    overlap = train_patients.intersection(val_patients)

    assert not overlap, (
        f"Patient leakage detected in fold {fold_number}: {overlap}"
    )

    cv_folds.append({
        "fold": fold_number,
        "train_manifest": fold_train,
        "val_manifest": fold_val,
        "train_patients": sorted(train_patients),
        "val_patients": sorted(val_patients),
    })

    fold_audit_rows.append({
        "fold": fold_number,
        "train_images": len(fold_train),
        "validation_images": len(fold_val),
        "train_patient_count": len(train_patients),
        "validation_patient_count": len(val_patients),
        "patient_overlap_count": len(overlap),
        "validation_patients": sorted(val_patients),
    })

cv_fold_audit = pd.DataFrame(fold_audit_rows)

print("Grouped cross-validation audit")
display(cv_fold_audit)

print(f"Total training images: {len(cv_manifest)}")
print(f"Unique training patients: {cv_manifest['patient_id'].nunique()}")
print(f"Number of folds: {len(cv_folds)}")
print("TestA/TestB used for fold construction: No")

assert len(cv_folds) == N_SPLITS
assert cv_fold_audit["patient_overlap_count"].eq(0).all()
assert sum(
    len(fold["val_manifest"]) for fold in cv_folds
) == len(cv_manifest)

print("All fold integrity checks passed.")


# Cell 35 — Train the Five-Fold Baseline

In [ ]:

# Cell 35: Five-Fold Patient-Grouped Baseline Training

import copy
import time
import json
from pathlib import Path
from torch.utils.data import DataLoader

CV_EPOCHS = 8
CV_BASE_CHANNELS = 16
CV_LEARNING_RATE = 2e-4
CV_WEIGHT_DECAY = 1e-4

cv_output_dir = PROJECT_ROOT / "outputs" / "cross_validation"
cv_checkpoint_dir = cv_output_dir / "checkpoints"
cv_checkpoint_dir.mkdir(parents=True, exist_ok=True)

cv_epoch_records = []
cv_best_records = []
cv_image_records = []
cv_fold_checkpoints = {}

def cv_loss(logits, targets, positive_weight):
    """BCE-with-logits plus soft Dice loss."""
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits,
        targets,
        pos_weight=positive_weight
    )

    probabilities = torch.sigmoid(logits)
    intersection = (probabilities * targets).sum(dim=(1, 2, 3))
    denominator = (
        probabilities.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    dice = (2.0 * intersection + 1e-7) / (denominator + 1e-7)
    dice_loss = 1.0 - dice.mean()

    return 0.5 * bce + 0.5 * dice_loss


def cv_evaluate(model, loader, positive_weight):
    model.eval()
    total_loss = 0.0
    total_images = 0
    image_rows = []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            targets = batch["mask"].to(DEVICE)
            logits = model(images)

            loss = cv_loss(logits, targets, positive_weight)
            probabilities = torch.sigmoid(logits)
            predictions = probabilities >= 0.5
            targets_binary = targets >= 0.5

            total_loss += loss.item() * images.size(0)
            total_images += images.size(0)

            for i, sample_id in enumerate(batch["sample_id"]):
                pred = predictions[i]
                truth = targets_binary[i]

                intersection = (pred & truth).sum().item()
                pred_count = pred.sum().item()
                truth_count = truth.sum().item()

                dice = (
                    2.0 * intersection + 1e-7
                ) / (pred_count + truth_count + 1e-7)

                iou = (
                    intersection + 1e-7
                ) / (pred_count + truth_count - intersection + 1e-7)

                precision = (
                    intersection + 1e-7
                ) / (pred_count + 1e-7)

                recall = (
                    intersection + 1e-7
                ) / (truth_count + 1e-7)

                image_rows.append({
                    "sample_id": sample_id,
                    "dice": float(dice),
                    "iou": float(iou),
                    "precision": float(precision),
                    "recall": float(recall),
                    "target_foreground_fraction": float(truth.float().mean()),
                    "predicted_foreground_fraction": float(pred.float().mean()),
                })

    return total_loss / max(total_images, 1), image_rows


for fold_info in cv_folds:
    fold_number = fold_info["fold"]

    print(f"\n{'=' * 24} FOLD {fold_number}/{N_SPLITS} {'=' * 24}")

    # Reproducible initialization and shuffled training order per fold.
    fold_seed = SEED + fold_number
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(fold_seed)

    train_ds = GlaSSegmentationDataset(fold_info["train_manifest"])
    val_ds = GlaSSegmentationDataset(fold_info["val_manifest"])

    generator = torch.Generator()
    generator.manual_seed(fold_seed)

    train_dl = DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        generator=generator
    )

    val_dl = DataLoader(
        val_ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    # Calculate positive-pixel weighting from this fold's training images only.
    foreground_pixels = 0
    background_pixels = 0

    for batch in train_dl:
        mask = batch["mask"]
        foreground_pixels += int((mask > 0.5).sum().item())
        background_pixels += int((mask <= 0.5).sum().item())

    pos_weight_value = background_pixels / max(foreground_pixels, 1)
    positive_weight = torch.tensor(
        [pos_weight_value],
        dtype=torch.float32,
        device=DEVICE
    ).view(1, 1, 1, 1)

    # Every fold starts from a newly initialized model.
    model = UNet(base_channels=CV_BASE_CHANNELS).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CV_LEARNING_RATE,
        weight_decay=CV_WEIGHT_DECAY
    )

    best_dice = -1.0
    best_epoch = None
    best_state = None
    best_val_loss = None
    fold_start = time.time()

    for epoch in range(1, CV_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_images = 0
        train_dice_sum = 0.0

        epoch_start = time.time()

        for batch in train_dl:
            images = batch["image"].to(DEVICE)
            targets = batch["mask"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = cv_loss(logits, targets, positive_weight)

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite training loss in fold {fold_number}, "
                    f"epoch {epoch}"
                )

            loss.backward()
            optimizer.step()

            batch_size_actual = images.size(0)
            train_loss_sum += loss.item() * batch_size_actual
            train_images += batch_size_actual

            with torch.no_grad():
                predictions = torch.sigmoid(logits) >= 0.5
                truths = targets >= 0.5

                intersections = (predictions & truths).sum(dim=(1, 2, 3))
                pred_counts = predictions.sum(dim=(1, 2, 3))
                truth_counts = truths.sum(dim=(1, 2, 3))

                batch_dice = (
                    2.0 * intersections.float() + 1e-7
                ) / (
                    pred_counts.float() + truth_counts.float() + 1e-7
                )

                train_dice_sum += batch_dice.sum().item()

        train_loss = train_loss_sum / max(train_images, 1)
        train_dice = train_dice_sum / max(train_images, 1)

        val_loss, val_rows = cv_evaluate(
            model, val_dl, positive_weight
        )
        val_dice = float(np.mean([r["dice"] for r in val_rows]))

        cv_epoch_records.append({
            "fold": fold_number,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_dice": train_dice,
            "val_loss": val_loss,
            "val_image_mean_dice": val_dice,
            "epoch_seconds": time.time() - epoch_start
        })

        print(
            f"Epoch {epoch:02d}/{CV_EPOCHS} | "
            f"train loss={train_loss:.4f} | "
            f"train Dice={train_dice:.4f} | "
            f"val loss={val_loss:.4f} | "
            f"val Dice={val_dice:.4f}"
        )

        if val_dice > best_dice:
            best_dice = val_dice
            best_epoch = epoch
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

    checkpoint_path = (
        cv_checkpoint_dir / f"unet_cv_fold_{fold_number}_best.pt"
    )

    torch.save({
        "fold": fold_number,
        "epoch": best_epoch,
        "validation_image_mean_dice": best_dice,
        "validation_loss": best_val_loss,
        "model_state_dict": best_state,
        "seed": fold_seed,
        "base_channels": CV_BASE_CHANNELS,
        "learning_rate": CV_LEARNING_RATE,
        "weight_decay": CV_WEIGHT_DECAY,
    }, checkpoint_path)

    # Re-evaluate the selected checkpoint, not the final epoch by default.
    model.load_state_dict(best_state)
    _, best_rows = cv_evaluate(model, val_dl, positive_weight)

    for row in best_rows:
        normalized_id = normalize_sample_id(row["sample_id"])
        row["fold"] = fold_number
        row["patient_id"] = int(patient_lookup.loc[normalized_id])
        row["best_epoch"] = best_epoch
        cv_image_records.append(row)

    cv_fold_checkpoints[fold_number] = str(checkpoint_path)

    cv_best_records.append({
        "fold": fold_number,
        "train_images": len(train_ds),
        "validation_images": len(val_ds),
        "train_patients": len(fold_info["train_patients"]),
        "validation_patients": len(fold_info["val_patients"]),
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "best_val_image_mean_dice": best_dice,
        "elapsed_minutes": (time.time() - fold_start) / 60.0,
        "checkpoint": str(checkpoint_path)
    })

    print(
        f"Fold {fold_number} complete: best epoch={best_epoch}, "
        f"validation Dice={best_dice:.4f}, "
        f"elapsed={(time.time() - fold_start) / 60.0:.1f} min"
    )

cv_epoch_history = pd.DataFrame(cv_epoch_records)
cv_fold_results = pd.DataFrame(cv_best_records)
cv_image_results = pd.DataFrame(cv_image_records)

cv_image_results.to_csv(
    cv_output_dir / "baseline_cv_image_metrics.csv",
    index=False
)
cv_epoch_history.to_csv(
    cv_output_dir / "baseline_cv_epoch_history.csv",
    index=False
)
cv_fold_results.to_csv(
    cv_output_dir / "baseline_cv_fold_results.csv",
    index=False
)

print("\nFold-level results")
display(cv_fold_results[
    [
        "fold", "best_epoch", "best_val_loss",
        "best_val_image_mean_dice", "elapsed_minutes"
    ]
])

print("\nCross-validation summary")
display(
    cv_fold_results[
        ["best_val_image_mean_dice", "best_val_loss"]
    ].agg(["mean", "std", "min", "max"])
)

print("\nImage-level out-of-fold summary")
display(
    cv_image_results[
        ["dice", "iou", "precision", "recall"]
    ].agg(["mean", "std", "min", "max"])
)

print(f"\nArtifacts saved under: {cv_output_dir}")


# Cell 36 — Analyze Patient-Level Cross-Validation Results

In [ ]:

# Cell 36: Patient-Level and Fold-Level Baseline Analysis

assert "cv_image_results" in globals() and not cv_image_results.empty

# Each training image should appear in exactly one validation fold.
assert cv_image_results["sample_id"].nunique() == len(cv_manifest), (
    "Some images are missing from, or duplicated in, out-of-fold results."
)

assert cv_image_results.groupby("sample_id")["fold"].nunique().eq(1).all()

# Equal weight per image within each patient.
patient_cv_results = (
    cv_image_results
    .groupby("patient_id")
    .agg(
        image_count=("sample_id", "count"),
        mean_image_dice=("dice", "mean"),
        std_image_dice=("dice", "std"),
        mean_image_iou=("iou", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_target_foreground=("target_foreground_fraction", "mean"),
        mean_predicted_foreground=("predicted_foreground_fraction", "mean"),
    )
    .reset_index()
    .sort_values("mean_image_dice")
    .reset_index(drop=True)
)

print("Patient-level out-of-fold metrics (lowest mean Dice first)")
display(patient_cv_results.round(4))

print("\nPatient-level summary statistics")
display(
    patient_cv_results[
        ["mean_image_dice", "mean_image_iou", "mean_precision", "mean_recall"]
    ].agg(["mean", "std", "min", "max"]).round(4)
)

print("\nFold-level validation Dice")
display(
    cv_fold_results[
        ["fold", "validation_images", "validation_patients",
         "best_epoch", "best_val_image_mean_dice"]
    ].round(4)
)

# Plot 1: fold-level selected validation Dice.
plt.figure(figsize=(8, 4))
plt.bar(
    cv_fold_results["fold"].astype(str),
    cv_fold_results["best_val_image_mean_dice"]
)
plt.axhline(
    cv_fold_results["best_val_image_mean_dice"].mean(),
    linestyle="--",
    label="Mean fold Dice"
)
plt.xlabel("Fold")
plt.ylabel("Best validation image-mean Dice")
plt.title("Baseline Performance Across Patient-Grouped Folds")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

# Plot 2: mean image Dice per patient.
plot_patients = patient_cv_results.sort_values("mean_image_dice")

plt.figure(figsize=(11, 5))
plt.bar(
    plot_patients["patient_id"].astype(str),
    plot_patients["mean_image_dice"]
)
plt.axhline(
    plot_patients["mean_image_dice"].mean(),
    linestyle="--",
    label="Unweighted mean across patients"
)
plt.xlabel("Patient ID")
plt.ylabel("Mean image Dice")
plt.title("Out-of-Fold Dice by Patient")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

# Save the derived patient-level results.
patient_results_path = (
    PROJECT_ROOT
    / "outputs"
    / "cross_validation"
    / "baseline_cv_patient_metrics.csv"
)

patient_cv_results.to_csv(patient_results_path, index=False)

print(f"Saved patient-level metrics: {patient_results_path}")
print(f"Unique patients evaluated: {patient_cv_results['patient_id'].nunique()}")
print(f"Unique images evaluated: {cv_image_results['sample_id'].nunique()}")
print("TestA/TestB used: No")


# Cell 37 — Define the AS-HMM-BO Search Protocol

In [ ]:

# Cell 37: Freeze the Initial Optimizer Experiment Protocol

from pathlib import Path
import json
import pandas as pd

SEARCH_CONFIG = {
    "experiment_name": "AS-HMM-BO_GlaS_Prototype",
    "experiment_version": "0.1.0",

    # Keep the final benchmark separate from hyperparameter discovery.
    "search_fold_ids": [3, 4],
    "final_evaluation_fold_ids": [1, 2, 3, 4, 5],

    # Small initial budget for validating the optimization machinery.
    "search_epochs": 3,
    "trials_per_optimizer": 8,
    "batch_size": BATCH_SIZE,
    "seed": SEED,

    # Initial search space.
    "learning_rate": {
        "type": "log_float",
        "low": 0.00005,
        "high": 0.0005,
    },
    "weight_decay": {
        "type": "log_float",
        "low": 0.000001,
        "high": 0.001,
    },
    "base_channels": [8, 16],
    "loss_bce_weight": {
        "type": "float",
        "low": 0.3,
        "high": 0.7,
    },

    # Objective and comparison rules.
    "primary_objective": "mean_validation_image_dice",
    "secondary_metrics": [
        "validation_loss",
        "mean_image_iou",
        "mean_precision",
        "mean_recall",
    ],
    "test_sets_allowed_during_search": False,
    "patient_grouping_required": True,

    # Controller diagnostics to log for AS-HMM-BO.
    "hmm_diagnostics": [
        "observation_vector",
        "inferred_state_probabilities",
        "selected_state",
        "candidate_sampling_policy",
        "resource_budget",
        "trial_score",
        "trial_duration_seconds",
    ],
}

assert len(SEARCH_CONFIG["search_fold_ids"]) >= 2
assert set(SEARCH_CONFIG["search_fold_ids"]).issubset(
    {fold["fold"] for fold in cv_folds}
)
assert SEARCH_CONFIG["trials_per_optimizer"] >= 2
assert SEARCH_CONFIG["search_epochs"] >= 1

search_config_path = (
    PROJECT_ROOT / "outputs" / "cross_validation"
    / "as_hmm_bo_search_config.json"
)
search_config_path.parent.mkdir(parents=True, exist_ok=True)

with open(search_config_path, "w", encoding="utf-8") as file:
    json.dump(SEARCH_CONFIG, file, indent=2)

print("Initial search protocol")
display(pd.DataFrame([
    {"setting": key, "value": str(value)}
    for key, value in SEARCH_CONFIG.items()
]))

print(f"\nSaved protocol: {search_config_path}")
print("Search folds:", SEARCH_CONFIG["search_fold_ids"])
print("Final evaluation folds:", SEARCH_CONFIG["final_evaluation_fold_ids"])
print("TestA/TestB available to optimizer: No")


# Cell 38 — Implement the Reusable Trial-Training Engine

In [ ]:

# Cell 38: Reusable Trial Training and Evaluation

import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

def mixed_segmentation_loss(logits, targets, positive_weight, bce_weight):
    """
    Weighted BCE + soft Dice loss.
    bce_weight is a real hyperparameter in [0, 1].
    """
    bce = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        pos_weight=positive_weight
    )

    probabilities = torch.sigmoid(logits)
    intersection = (probabilities * targets).sum(dim=(1, 2, 3))
    denominator = (
        probabilities.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    soft_dice = (
        2.0 * intersection + 1e-7
    ) / (
        denominator + 1e-7
    )

    dice_loss = 1.0 - soft_dice.mean()

    return bce_weight * bce + (1.0 - bce_weight) * dice_loss


def calculate_positive_weight(dataset, batch_size):
    """
    Estimate foreground/background pixel ratio from training data only.
    """
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    foreground = 0
    background = 0

    for batch in loader:
        mask = batch["mask"]
        foreground += int((mask > 0.5).sum().item())
        background += int((mask <= 0.5).sum().item())

    ratio = background / max(foreground, 1)

    return torch.tensor(
        [ratio],
        dtype=torch.float32,
        device=DEVICE
    ).view(1, 1, 1, 1)


def train_search_trial(
    params,
    fold_id,
    epochs=None,
    seed=None,
    return_history=True
):
    """
    Train one fresh model on one development fold.

    params:
        learning_rate, weight_decay, base_channels, loss_bce_weight

    Returns a dictionary containing the validation objective,
    secondary metrics, runtime, and optional epoch history.
    """
    epochs = (
        SEARCH_CONFIG["search_epochs"]
        if epochs is None else int(epochs)
    )
    seed = SEARCH_CONFIG["seed"] if seed is None else int(seed)

    if fold_id not in SEARCH_CONFIG["search_fold_ids"]:
        raise ValueError(
            f"Fold {fold_id} is not an authorized development fold."
        )

    required_params = {
        "learning_rate",
        "weight_decay",
        "base_channels",
        "loss_bce_weight"
    }

    missing = required_params - set(params)
    if missing:
        raise ValueError(f"Missing hyperparameters: {sorted(missing)}")

    if params["learning_rate"] <= 0 or params["weight_decay"] < 0:
        raise ValueError("Invalid learning rate or weight decay.")

    if params["base_channels"] not in SEARCH_CONFIG["base_channels"]:
        raise ValueError("Unsupported base_channels value.")

    if not 0.0 <= params["loss_bce_weight"] <= 1.0:
        raise ValueError("loss_bce_weight must be between 0 and 1.")

    fold_info = next(
        fold for fold in cv_folds if fold["fold"] == fold_id
    )

    # Independent, reproducible initialization per trial.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    train_ds = GlaSSegmentationDataset(fold_info["train_manifest"])
    val_ds = GlaSSegmentationDataset(fold_info["val_manifest"])

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_dl = DataLoader(
        train_ds,
        batch_size=SEARCH_CONFIG["batch_size"],
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        generator=generator
    )

    val_dl = DataLoader(
        val_ds,
        batch_size=SEARCH_CONFIG["batch_size"],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY
    )

    positive_weight = calculate_positive_weight(
        train_ds,
        SEARCH_CONFIG["batch_size"]
    )

    model = UNet(
        base_channels=int(params["base_channels"])
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(params["learning_rate"]),
        weight_decay=float(params["weight_decay"])
    )

    history = []
    best_epoch_dice = -1.0
    best_epoch = None
    best_val_loss = None
    best_state = None
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        for batch in train_dl:
            images = batch["image"].to(DEVICE)
            targets = batch["mask"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)

            loss = mixed_segmentation_loss(
                logits,
                targets,
                positive_weight,
                float(params["loss_bce_weight"])
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss at fold={fold_id}, epoch={epoch}"
                )

            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * images.size(0)
            train_count += images.size(0)

        train_loss = train_loss_sum / max(train_count, 1)

        model.eval()
        val_loss_sum = 0.0
        val_count = 0
        image_dice_values = []
        image_iou_values = []
        image_precision_values = []
        image_recall_values = []

        with torch.no_grad():
            for batch in val_dl:
                images = batch["image"].to(DEVICE)
                targets = batch["mask"].to(DEVICE)

                logits = model(images)
                loss = mixed_segmentation_loss(
                    logits,
                    targets,
                    positive_weight,
                    float(params["loss_bce_weight"])
                )

                val_loss_sum += loss.item() * images.size(0)
                val_count += images.size(0)

                predictions = torch.sigmoid(logits) >= 0.5
                truths = targets >= 0.5

                for i in range(images.size(0)):
                    pred = predictions[i]
                    truth = truths[i]

                    intersection = (pred & truth).sum().item()
                    pred_count = pred.sum().item()
                    truth_count = truth.sum().item()

                    dice = (
                        2.0 * intersection + 1e-7
                    ) / (
                        pred_count + truth_count + 1e-7
                    )

                    union = pred_count + truth_count - intersection

                    iou = (
                        intersection + 1e-7
                    ) / (union + 1e-7)

                    precision = (
                        intersection + 1e-7
                    ) / (pred_count + 1e-7)

                    recall = (
                        intersection + 1e-7
                    ) / (truth_count + 1e-7)

                    image_dice_values.append(float(dice))
                    image_iou_values.append(float(iou))
                    image_precision_values.append(float(precision))
                    image_recall_values.append(float(recall))

        val_loss = val_loss_sum / max(val_count, 1)
        mean_dice = float(np.mean(image_dice_values))

        epoch_result = {
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": val_loss,
            "mean_image_dice": mean_dice,
            "mean_image_iou": float(np.mean(image_iou_values)),
            "mean_precision": float(np.mean(image_precision_values)),
            "mean_recall": float(np.mean(image_recall_values))
        }
        history.append(epoch_result)

        if mean_dice > best_epoch_dice:
            best_epoch_dice = mean_dice
            best_epoch = epoch
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

    duration = time.time() - start_time

    # Report the best epoch within the fixed trial budget.
    result = {
        "fold": int(fold_id),
        "seed": int(seed),
        "epochs": int(epochs),
        "learning_rate": float(params["learning_rate"]),
        "weight_decay": float(params["weight_decay"]),
        "base_channels": int(params["base_channels"]),
        "loss_bce_weight": float(params["loss_bce_weight"]),
        "best_epoch": int(best_epoch),
        "mean_validation_image_dice": float(best_epoch_dice),
        "validation_loss": float(best_val_loss),
        "duration_seconds": float(duration),
        "model_state_dict": best_state
    }

    if return_history:
        result["history"] = history

    return result


print("Reusable trial engine defined.")
print("Authorized development folds:", SEARCH_CONFIG["search_fold_ids"])
print("Epochs per trial:", SEARCH_CONFIG["search_epochs"])
print("Sampled parameters actively used:")
print("  learning_rate -> Adam optimizer")
print("  weight_decay -> Adam optimizer")
print("  base_channels -> U-Net architecture")
print("  loss_bce_weight -> BCE/Dice loss mixture")


# Cell 39 — Smoke-Test the Search Trial Engine

In [ ]:

# Cell 39: Smoke-Test the Search Trial Engine

import pandas as pd
import numpy as np
import time

smoke_params = {
    "learning_rate": 2e-4,
    "weight_decay": 1e-4,
    "base_channels": 8,
    "loss_bce_weight": 0.5,
}

smoke_fold = SEARCH_CONFIG["search_fold_ids"][0]
smoke_epochs = SEARCH_CONFIG["search_epochs"]

print("Starting trial-engine smoke test...")
print(f"Fold: {smoke_fold}")
print(f"Epochs: {smoke_epochs}")
print(f"Parameters: {smoke_params}")

smoke_start = time.perf_counter()

smoke_result = train_search_trial(
    params=smoke_params,
    fold_id=smoke_fold,
    epochs=smoke_epochs,
    seed=SEED + 100,
    return_history=True,
)

smoke_elapsed = time.perf_counter() - smoke_start

print("\nSmoke-test result")
print("-" * 50)
print(f"Best epoch: {smoke_result['best_epoch']}")
print(
    "Validation Dice:",
    round(smoke_result["mean_validation_image_dice"], 4),
)
print("Validation loss:", round(smoke_result["validation_loss"], 4))
print("Trial duration:", round(smoke_elapsed, 2), "seconds")

history_df = pd.DataFrame(smoke_result["history"])
display(history_df)

assert smoke_result["fold"] == smoke_fold
assert 1 <= smoke_result["best_epoch"] <= smoke_epochs
assert np.isfinite(smoke_result["mean_validation_image_dice"])
assert np.isfinite(smoke_result["validation_loss"])
assert 0.0 <= smoke_result["mean_validation_image_dice"] <= 1.0
assert len(history_df) == smoke_epochs

print("\nPASS: trial engine completed and returned valid metrics.")


# Cell 40 — Run the TPE Baseline Search

In [ ]:

# Cell 40: TPE Baseline Hyperparameter Search

import time
import numpy as np
import pandas as pd
import optuna
from pathlib import Path

# Keep Optuna's output readable.
optuna.logging.set_verbosity(optuna.logging.WARNING)

tpe_trial_records = []
tpe_fold_records = []

def tpe_objective(trial):
    params = {
        "learning_rate": trial.suggest_float(
            "learning_rate",
            SEARCH_CONFIG["learning_rate"]["low"],
            SEARCH_CONFIG["learning_rate"]["high"],
            log=True,
        ),
        "weight_decay": trial.suggest_float(
            "weight_decay",
            SEARCH_CONFIG["weight_decay"]["low"],
            SEARCH_CONFIG["weight_decay"]["high"],
            log=True,
        ),
        "base_channels": trial.suggest_categorical(
            "base_channels",
            SEARCH_CONFIG["base_channels"],
        ),
        "loss_bce_weight": trial.suggest_float(
            "loss_bce_weight",
            SEARCH_CONFIG["loss_bce_weight"]["low"],
            SEARCH_CONFIG["loss_bce_weight"]["high"],
        ),
    }

    fold_scores = []
    fold_losses = []
    trial_start = time.perf_counter()

    for fold_id in SEARCH_CONFIG["search_fold_ids"]:
        # Same seed schedule can be reused when comparing optimizers.
        trial_seed = SEED + trial.number * 100 + fold_id

        result = train_search_trial(
            params=params,
            fold_id=fold_id,
            epochs=SEARCH_CONFIG["search_epochs"],
            seed=trial_seed,
            return_history=False,
        )

        dice = float(result["mean_validation_image_dice"])
        loss = float(result["validation_loss"])

        fold_scores.append(dice)
        fold_losses.append(loss)

        tpe_fold_records.append({
            "trial": trial.number,
            "fold": fold_id,
            "seed": trial_seed,
            "validation_dice": dice,
            "validation_loss": loss,
            "best_epoch": result["best_epoch"],
            "duration_seconds": result["duration_seconds"],
        })

        # Release the trained model state before the next fold.
        del result

    mean_dice = float(np.mean(fold_scores))
    mean_loss = float(np.mean(fold_losses))
    duration = time.perf_counter() - trial_start

    tpe_trial_records.append({
        "trial": trial.number,
        **params,
        "mean_validation_image_dice": mean_dice,
        "mean_validation_loss": mean_loss,
        "fold_3_dice": fold_scores[
            SEARCH_CONFIG["search_fold_ids"].index(3)
        ] if 3 in SEARCH_CONFIG["search_fold_ids"] else np.nan,
        "fold_4_dice": fold_scores[
            SEARCH_CONFIG["search_fold_ids"].index(4)
        ] if 4 in SEARCH_CONFIG["search_fold_ids"] else np.nan,
        "duration_seconds": duration,
    })

    return mean_dice


tpe_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    study_name="glas_tpe_baseline",
)

tpe_start = time.perf_counter()

tpe_study.optimize(
    tpe_objective,
    n_trials=SEARCH_CONFIG["trials_per_optimizer"],
    show_progress_bar=True,
)

tpe_elapsed = time.perf_counter() - tpe_start

tpe_trials_df = (
    pd.DataFrame(tpe_trial_records)
    .sort_values("trial")
    .reset_index(drop=True)
)

tpe_folds_df = pd.DataFrame(tpe_fold_records)

output_dir = PROJECT_ROOT / "outputs" / "cross_validation"
output_dir.mkdir(parents=True, exist_ok=True)

tpe_trials_df.to_csv(
    output_dir / "tpe_baseline_trial_results.csv",
    index=False,
)
tpe_folds_df.to_csv(
    output_dir / "tpe_baseline_fold_results.csv",
    index=False,
)

print("\nTPE baseline search completed")
print("-" * 55)
print(f"Trials: {len(tpe_trials_df)}")
print(f"Fold evaluations: {len(tpe_folds_df)}")
print(f"Elapsed time: {tpe_elapsed / 60:.2f} minutes")
print(f"Best trial: {tpe_study.best_trial.number}")
print(
    "Best mean development Dice:",
    round(tpe_study.best_value, 4),
)
print("Best parameters:")
for name, value in tpe_study.best_params.items():
    print(f"  {name}: {value}")

display(
    tpe_trials_df.sort_values(
        "mean_validation_image_dice",
        ascending=False,
    )
)

assert len(tpe_trials_df) == SEARCH_CONFIG["trials_per_optimizer"]
assert len(tpe_folds_df) == (
    SEARCH_CONFIG["trials_per_optimizer"]
    * len(SEARCH_CONFIG["search_fold_ids"])
)
assert np.isfinite(tpe_trials_df["mean_validation_image_dice"]).all()

print("\nPASS: TPE baseline results saved and validated.")


# Cell 41 — Define the HMM-Guided Search Controller

In [ ]:

# Cell 41: HMM State Controller and State-Dependent Proposals

import numpy as np
import pandas as pd

# Latent states: 0 = exploration, 1 = exploitation, 2 = cooling.
HMM_STATES = ["exploration", "exploitation", "cooling"]

# Transition probabilities: rows are current states; columns are next states.
HMM_TRANSITION = np.array([
    [0.70, 0.25, 0.05],  # exploration
    [0.15, 0.70, 0.15],  # exploitation
    [0.25, 0.20, 0.55],  # cooling
], dtype=float)

HMM_INITIAL = np.array([0.60, 0.30, 0.10], dtype=float)

# Observation vector:
# [recent reward improvement, recent reward variability, normalized cost]
#
# These are prototype emission assumptions, not learned parameters.
HMM_EMISSION_MEANS = np.array([
    [ 0.015, 0.080, 0.65],  # exploration
    [ 0.025, 0.025, 0.60],  # exploitation
    [ 0.000, 0.015, 0.90],  # cooling
], dtype=float)

HMM_EMISSION_STDS = np.array([
    [0.025, 0.060, 0.25],
    [0.020, 0.025, 0.25],
    [0.015, 0.020, 0.20],
], dtype=float)


def _normal_log_likelihood(observation, means, stds):
    """Independent Gaussian emission log-likelihood for each state."""
    z = (observation[None, :] - means) / stds
    return -0.5 * np.sum(
        z * z + 2.0 * np.log(stds) + np.log(2.0 * np.pi),
        axis=1,
    )


def infer_hmm_state(observations, previous_posterior=None):
    """
    Infer the current latent state using a filtering update.

    observations: sequence of shape (n_observations, 3)
    previous_posterior: optional posterior from the preceding update
    """
    observations = np.asarray(observations, dtype=float)

    if observations.ndim != 2 or observations.shape[1] != 3:
        raise ValueError("observations must have shape (n, 3)")
    if len(observations) == 0:
        raise ValueError("At least one observation is required")
    if not np.isfinite(observations).all():
        raise ValueError("Observations must all be finite")

    posterior = (
        HMM_INITIAL.copy()
        if previous_posterior is None
        else np.asarray(previous_posterior, dtype=float).copy()
    )

    if posterior.shape != (len(HMM_STATES),):
        raise ValueError("previous_posterior has an invalid shape")
    if (posterior < 0).any() or posterior.sum() <= 0:
        raise ValueError("previous_posterior must be nonnegative and nonzero")

    posterior /= posterior.sum()

    for observation in observations:
        predicted = posterior @ HMM_TRANSITION
        log_likelihood = _normal_log_likelihood(
            observation,
            HMM_EMISSION_MEANS,
            HMM_EMISSION_STDS,
        )

        # Normalize in log space to reduce numerical underflow.
        log_posterior = np.log(np.clip(predicted, 1e-300, None))
        log_posterior += log_likelihood
        log_posterior -= np.max(log_posterior)

        posterior = np.exp(log_posterior)
        posterior /= posterior.sum()

    state_index = int(np.argmax(posterior))
    return {
        "state_index": state_index,
        "state": HMM_STATES[state_index],
        "posterior": posterior,
    }


def build_hmm_observations(trial_results):
    """
    Convert completed trial results into chronological observations.

    Improvement and variability are computed from prior trial rewards.
    Cost is scaled against the largest observed trial duration.
    """
    if len(trial_results) < 2:
        raise ValueError("At least two completed trials are required")

    ordered = sorted(trial_results, key=lambda row: row["trial"])
    rewards = np.array(
        [row["mean_validation_image_dice"] for row in ordered],
        dtype=float,
    )
    durations = np.array(
        [row["duration_seconds"] for row in ordered],
        dtype=float,
    )

    if not np.isfinite(rewards).all() or not np.isfinite(durations).all():
        raise ValueError("Trial rewards and durations must be finite")

    duration_scale = max(float(durations.max()), 1e-8)
    observations = []

    for i in range(len(ordered)):
        recent = rewards[max(0, i - 2):i + 1]

        improvement = (
            float(rewards[i] - rewards[i - 1])
            if i > 0 else 0.0
        )
        variability = (
            float(np.std(recent, ddof=0))
            if len(recent) > 1 else 0.0
        )
        normalized_cost = float(durations[i] / duration_scale)

        observations.append([
            improvement,
            variability,
            normalized_cost,
        ])

    return np.asarray(observations, dtype=float)


def propose_hmm_guided_params(
    state,
    completed_results,
    rng,
):
    """
    Generate a new candidate whose sampling policy depends on HMM state.

    Exploitation centers learning rate and weight decay around the
    best completed configuration. Exploration samples broadly.
    Cooling narrows the local search but retains a random exploration
    component.
    """
    if state not in HMM_STATES:
        raise ValueError(f"Unknown HMM state: {state}")

    best_row = None
    if completed_results:
        best_row = max(
            completed_results,
            key=lambda row: row["mean_validation_image_dice"],
        )

    if state == "exploration" or best_row is None:
        learning_rate = float(np.exp(rng.uniform(
            np.log(SEARCH_CONFIG["learning_rate"]["low"]),
            np.log(SEARCH_CONFIG["learning_rate"]["high"]),
        )))
        weight_decay = float(np.exp(rng.uniform(
            np.log(SEARCH_CONFIG["weight_decay"]["low"]),
            np.log(SEARCH_CONFIG["weight_decay"]["high"]),
        )))
        base_channels = int(rng.choice(SEARCH_CONFIG["base_channels"]))
        bce_weight = float(rng.uniform(
            SEARCH_CONFIG["loss_bce_weight"]["low"],
            SEARCH_CONFIG["loss_bce_weight"]["high"],
        ))

    else:
        best = best_row

        # Work in log space for positive parameters.
        lr_center = np.log(best["learning_rate"])
        wd_center = np.log(best["weight_decay"])

        if state == "exploitation":
            lr_sigma, wd_sigma, bce_sigma = 0.30, 0.70, 0.10
            random_probability = 0.10
        else:  # cooling
            lr_sigma, wd_sigma, bce_sigma = 0.15, 0.40, 0.06
            random_probability = 0.20

        if rng.random() < random_probability:
            learning_rate = float(np.exp(rng.uniform(
                np.log(SEARCH_CONFIG["learning_rate"]["low"]),
                np.log(SEARCH_CONFIG["learning_rate"]["high"]),
            )))
            weight_decay = float(np.exp(rng.uniform(
                np.log(SEARCH_CONFIG["weight_decay"]["low"]),
                np.log(SEARCH_CONFIG["weight_decay"]["high"]),
            )))
            base_channels = int(rng.choice(SEARCH_CONFIG["base_channels"]))
            bce_weight = float(rng.uniform(
                SEARCH_CONFIG["loss_bce_weight"]["low"],
                SEARCH_CONFIG["loss_bce_weight"]["high"],
            ))
        else:
            learning_rate = float(np.exp(rng.normal(lr_center, lr_sigma)))
            weight_decay = float(np.exp(rng.normal(wd_center, wd_sigma)))
            learning_rate = float(np.clip(
                learning_rate,
                SEARCH_CONFIG["learning_rate"]["low"],
                SEARCH_CONFIG["learning_rate"]["high"],
            ))
            weight_decay = float(np.clip(
                weight_decay,
                SEARCH_CONFIG["weight_decay"]["low"],
                SEARCH_CONFIG["weight_decay"]["high"],
            ))

            base_channels = (
                int(best["base_channels"])
                if rng.random() < 0.75
                else int(rng.choice(SEARCH_CONFIG["base_channels"]))
            )
            bce_weight = float(np.clip(
                rng.normal(best["loss_bce_weight"], bce_sigma),
                SEARCH_CONFIG["loss_bce_weight"]["low"],
                SEARCH_CONFIG["loss_bce_weight"]["high"],
            ))

    return {
        "learning_rate": learning_rate,
        "weight_decay": weight_decay,
        "base_channels": base_channels,
        "loss_bce_weight": bce_weight,
    }


print("HMM controller defined.")
print("States:", HMM_STATES)
print("Transition matrix:")
display(pd.DataFrame(
    HMM_TRANSITION,
    index=HMM_STATES,
    columns=HMM_STATES,
))


# Cell 42 — Test State Inference and Proposal Changes

In [ ]:

# Cell 42: Validate HMM Inference and State-Dependent Sampling

observations = build_hmm_observations(tpe_trial_records)
hmm_result = infer_hmm_state(observations)

print("Observation matrix shape:", observations.shape)
print("Inferred state:", hmm_result["state"])
print("Posterior probabilities:")

display(pd.DataFrame({
    "state": HMM_STATES,
    "posterior_probability": hmm_result["posterior"],
}))

# Demonstrate that each state has a valid, state-dependent proposal policy.
proposal_rng = np.random.default_rng(SEED + 500)
proposal_examples = []

for state in HMM_STATES:
    proposal = propose_hmm_guided_params(
        state=state,
        completed_results=tpe_trial_records,
        rng=proposal_rng,
    )
    proposal_examples.append({
        "controller_state": state,
        **proposal,
    })

proposal_df = pd.DataFrame(proposal_examples)
display(proposal_df)

assert observations.shape == (len(tpe_trial_records), 3)
assert np.isfinite(observations).all()
assert np.isclose(hmm_result["posterior"].sum(), 1.0)
assert (hmm_result["posterior"] >= 0).all()
assert set(proposal_df["base_channels"]).issubset(
    set(SEARCH_CONFIG["base_channels"])
)
assert proposal_df["learning_rate"].between(
    SEARCH_CONFIG["learning_rate"]["low"],
    SEARCH_CONFIG["learning_rate"]["high"],
).all()
assert proposal_df["weight_decay"].between(
    SEARCH_CONFIG["weight_decay"]["low"],
    SEARCH_CONFIG["weight_decay"]["high"],
).all()
assert proposal_df["loss_bce_weight"].between(
    SEARCH_CONFIG["loss_bce_weight"]["low"],
    SEARCH_CONFIG["loss_bce_weight"]["high"],
).all()

print("PASS: state inference and proposal policies returned valid outputs.")


# Cell 43 — Run the AS-HMM-BO Development Search

In [ ]:

# Cell 43: AS-HMM-BO Adaptive Development Search

import time
import numpy as np
import pandas as pd
from pathlib import Path

as_hmm_trial_records = []
as_hmm_fold_records = []
as_hmm_state_history = []

as_rng = np.random.default_rng(SEED + 1000)
as_search_start = time.perf_counter()

for trial_number in range(SEARCH_CONFIG["trials_per_optimizer"]):
    completed = list(as_hmm_trial_records)

    # Bootstrap with broad exploration until two observations exist.
    # Thereafter, infer the current state from the completed trajectory.
    if len(completed) < 2:
        controller_state = "exploration"
        posterior = np.array([1.0, 0.0, 0.0])
        latest_observation = [np.nan, np.nan, np.nan]
        inference_source = "bootstrap"
    else:
        observation_matrix = build_hmm_observations(completed)
        hmm_result = infer_hmm_state(observation_matrix)

        controller_state = hmm_result["state"]
        posterior = hmm_result["posterior"]
        latest_observation = observation_matrix[-1].tolist()
        inference_source = "filtered_completed_trials"

    # The state selects the candidate proposal policy.
    params = propose_hmm_guided_params(
        state=controller_state,
        completed_results=completed,
        rng=as_rng,
    )

    if controller_state == "exploration":
        proposal_policy = "broad_random"
    elif controller_state == "exploitation":
        proposal_policy = "best_candidate_local_search"
    else:
        proposal_policy = "narrow_local_search_with_random_restart"

    print(
        f"\nTrial {trial_number + 1}/"
        f"{SEARCH_CONFIG['trials_per_optimizer']} | "
        f"state={controller_state} | "
        f"policy={proposal_policy}"
    )
    print("Parameters:", params)

    fold_scores = []
    fold_losses = []
    trial_start = time.perf_counter()

    for fold_id in SEARCH_CONFIG["search_fold_ids"]:
        trial_seed = SEED + trial_number * 100 + fold_id

        result = train_search_trial(
            params=params,
            fold_id=fold_id,
            epochs=SEARCH_CONFIG["search_epochs"],
            seed=trial_seed,
            return_history=False,
        )

        dice = float(result["mean_validation_image_dice"])
        loss = float(result["validation_loss"])

        fold_scores.append(dice)
        fold_losses.append(loss)

        as_hmm_fold_records.append({
            "trial": trial_number,
            "fold": fold_id,
            "seed": trial_seed,
            "controller_state": controller_state,
            "proposal_policy": proposal_policy,
            "validation_dice": dice,
            "validation_loss": loss,
            "best_epoch": result["best_epoch"],
            "duration_seconds": result["duration_seconds"],
        })

        del result

    duration = time.perf_counter() - trial_start
    mean_dice = float(np.mean(fold_scores))
    mean_loss = float(np.mean(fold_losses))

    trial_record = {
        "trial": trial_number,
        **params,
        "controller_state": controller_state,
        "proposal_policy": proposal_policy,
        "inference_source": inference_source,
        "posterior_exploration": float(posterior[0]),
        "posterior_exploitation": float(posterior[1]),
        "posterior_cooling": float(posterior[2]),
        "observation_improvement": float(latest_observation[0]),
        "observation_variability": float(latest_observation[1]),
        "observation_normalized_cost": float(latest_observation[2]),
        "mean_validation_image_dice": mean_dice,
        "mean_validation_loss": mean_loss,
        "fold_3_dice": (
            fold_scores[SEARCH_CONFIG["search_fold_ids"].index(3)]
            if 3 in SEARCH_CONFIG["search_fold_ids"] else np.nan
        ),
        "fold_4_dice": (
            fold_scores[SEARCH_CONFIG["search_fold_ids"].index(4)]
            if 4 in SEARCH_CONFIG["search_fold_ids"] else np.nan
        ),
        "duration_seconds": duration,
        "epochs_per_fold": SEARCH_CONFIG["search_epochs"],
    }

    as_hmm_trial_records.append(trial_record)

    as_hmm_state_history.append({
        "trial": trial_number,
        "controller_state": controller_state,
        "proposal_policy": proposal_policy,
        "posterior": posterior.tolist(),
        "observation": latest_observation,
        "parameters": params.copy(),
        "mean_validation_image_dice": mean_dice,
    })

    print(
        f"Mean development Dice={mean_dice:.4f} | "
        f"mean validation loss={mean_loss:.4f} | "
        f"elapsed={duration:.1f}s"
    )

as_hmm_elapsed = time.perf_counter() - as_search_start

as_hmm_trials_df = pd.DataFrame(as_hmm_trial_records)
as_hmm_folds_df = pd.DataFrame(as_hmm_fold_records)
as_hmm_states_df = pd.DataFrame([
    {
        "trial": item["trial"],
        "controller_state": item["controller_state"],
        "proposal_policy": item["proposal_policy"],
        "posterior_exploration": item["posterior"][0],
        "posterior_exploitation": item["posterior"][1],
        "posterior_cooling": item["posterior"][2],
        "observation": str(item["observation"]),
        "mean_validation_image_dice": item["mean_validation_image_dice"],
    }
    for item in as_hmm_state_history
])

output_dir = PROJECT_ROOT / "outputs" / "cross_validation"
output_dir.mkdir(parents=True, exist_ok=True)

as_hmm_trials_df.to_csv(
    output_dir / "as_hmm_bo_trial_results.csv",
    index=False,
)
as_hmm_folds_df.to_csv(
    output_dir / "as_hmm_bo_fold_results.csv",
    index=False,
)
as_hmm_states_df.to_csv(
    output_dir / "as_hmm_bo_state_history.csv",
    index=False,
)

best_as_row = as_hmm_trials_df.loc[
    as_hmm_trials_df["mean_validation_image_dice"].idxmax()
]

print("\nAS-HMM-BO development search completed")
print("-" * 55)
print(f"Trials: {len(as_hmm_trials_df)}")
print(f"Fold evaluations: {len(as_hmm_folds_df)}")
print(f"Elapsed time: {as_hmm_elapsed / 60:.2f} minutes")
print(
    "Best mean development Dice:",
    round(float(best_as_row["mean_validation_image_dice"]), 4),
)
print("Best trial:", int(best_as_row["trial"]))
print("\nController-state counts:")
display(as_hmm_trials_df["controller_state"].value_counts().rename("count"))
print("\nTrial results:")
display(
    as_hmm_trials_df.sort_values(
        "mean_validation_image_dice",
        ascending=False,
    )
)

assert len(as_hmm_trials_df) == SEARCH_CONFIG["trials_per_optimizer"]
assert len(as_hmm_folds_df) == (
    SEARCH_CONFIG["trials_per_optimizer"]
    * len(SEARCH_CONFIG["search_fold_ids"])
)
assert np.isfinite(
    as_hmm_trials_df["mean_validation_image_dice"]
).all()
assert as_hmm_trials_df["proposal_policy"].notna().all()
assert np.allclose(
    as_hmm_trials_df[
        [
            "posterior_exploration",
            "posterior_exploitation",
            "posterior_cooling",
        ]
    ].sum(axis=1).to_numpy(),
    1.0,
)

print("\nPASS: adaptive search completed and its state decisions were logged.")


# Cell 44 — Compare the Two Search Trajectories

In [ ]:

# Cell 44: TPE vs AS-HMM-BO Development Comparison

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

comparison_rows = []

for optimizer_name, frame in [
    ("TPE", tpe_trials_df),
    ("AS-HMM-BO", as_hmm_trials_df),
]:
    scores = frame["mean_validation_image_dice"].astype(float)

    comparison_rows.append({
        "optimizer": optimizer_name,
        "trials": len(frame),
        "best_dice": scores.max(),
        "mean_trial_dice": scores.mean(),
        "median_trial_dice": scores.median(),
        "std_trial_dice": scores.std(ddof=1),
        "mean_validation_loss": frame["mean_validation_loss"].mean(),
        "total_search_minutes": frame["duration_seconds"].sum() / 60.0,
        "best_trial": int(scores.idxmax()),
    })

comparison_df = pd.DataFrame(comparison_rows).set_index("optimizer")

print("Search-level comparison")
display(comparison_df.round(4))

# Compare per-fold performance for each optimizer's best configuration.
best_tpe_row = tpe_trials_df.loc[
    tpe_trials_df["mean_validation_image_dice"].idxmax()
]
best_as_row = as_hmm_trials_df.loc[
    as_hmm_trials_df["mean_validation_image_dice"].idxmax()
]

best_configuration_comparison = pd.DataFrame([
    {
        "optimizer": "TPE",
        "trial": int(best_tpe_row["trial"]),
        "fold_3_dice": best_tpe_row["fold_3_dice"],
        "fold_4_dice": best_tpe_row["fold_4_dice"],
        "mean_dice": best_tpe_row["mean_validation_image_dice"],
        "fold_gap": abs(
            best_tpe_row["fold_3_dice"] - best_tpe_row["fold_4_dice"]
        ),
    },
    {
        "optimizer": "AS-HMM-BO",
        "trial": int(best_as_row["trial"]),
        "fold_3_dice": best_as_row["fold_3_dice"],
        "fold_4_dice": best_as_row["fold_4_dice"],
        "mean_dice": best_as_row["mean_validation_image_dice"],
        "fold_gap": abs(
            best_as_row["fold_3_dice"] - best_as_row["fold_4_dice"]
        ),
    },
]).set_index("optimizer")

print("Best-configuration fold comparison")
display(best_configuration_comparison.round(4))

# Show the trial trajectories in evaluation order.
plt.figure(figsize=(9, 4.5))
plt.plot(
    tpe_trials_df["trial"] + 1,
    tpe_trials_df["mean_validation_image_dice"],
    marker="o",
    label="TPE",
)
plt.plot(
    as_hmm_trials_df["trial"] + 1,
    as_hmm_trials_df["mean_validation_image_dice"],
    marker="s",
    label="AS-HMM-BO",
)
plt.xlabel("Trial number")
plt.ylabel("Mean development Dice")
plt.title("Development Search Trajectories")
plt.xticks(range(1, SEARCH_CONFIG["trials_per_optimizer"] + 1))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Save a concise comparison artifact.
comparison_path = (
    PROJECT_ROOT / "outputs" / "cross_validation"
    / "optimizer_development_comparison.csv"
)
comparison_df.to_csv(comparison_path)

print(f"Comparison saved to: {comparison_path}")


# Cell 45 — HMM Controller Diagnostic Audit

In [ ]:

# Cell 45: Audit HMM Observations, Posterior Confidence, and State Transitions

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Reconstruct observations from the completed AS-HMM-BO trajectory.
diagnostic_observations = build_hmm_observations(as_hmm_trial_records)

diagnostic_rows = []
previous_state = None
previous_posterior = None

for i, observation in enumerate(diagnostic_observations):
    # Replay observations sequentially to avoid using future trials.
    if i == 0:
        inference = infer_hmm_state(
            diagnostic_observations[:1],
            previous_posterior=None,
        )
    else:
        inference = infer_hmm_state(
            diagnostic_observations[i:i+1],
            previous_posterior=previous_posterior,
        )

    posterior = inference["posterior"]
    state = inference["state"]

    diagnostic_rows.append({
        "trial": i,
        "improvement": observation[0],
        "reward_variability": observation[1],
        "normalized_duration": observation[2],
        "posterior_exploration": posterior[0],
        "posterior_exploitation": posterior[1],
        "posterior_cooling": posterior[2],
        "selected_state": state,
        "max_posterior": float(posterior.max()),
        "posterior_entropy": float(
            -np.sum(posterior * np.log(np.clip(posterior, 1e-12, 1.0)))
        ),
        "observed_dice": as_hmm_trial_records[i][
            "mean_validation_image_dice"
        ],
        "base_channels": as_hmm_trial_records[i]["base_channels"],
        "duration_seconds": as_hmm_trial_records[i]["duration_seconds"],
    })

    previous_state = state
    previous_posterior = posterior.copy()

hmm_diagnostics_df = pd.DataFrame(diagnostic_rows)

print("Sequential HMM diagnostic table")
display(hmm_diagnostics_df.round(4))

# Summarize posterior confidence and state transitions.
state_counts = (
    hmm_diagnostics_df["selected_state"]
    .value_counts()
    .reindex(HMM_STATES, fill_value=0)
    .rename("trial_count")
    .to_frame()
)

transition_counts = pd.crosstab(
    hmm_diagnostics_df["selected_state"].shift(1),
    hmm_diagnostics_df["selected_state"],
    dropna=True,
).reindex(
    index=HMM_STATES,
    columns=HMM_STATES,
    fill_value=0,
)

print("Sequentially inferred state counts")
display(state_counts)

print("Observed state-transition counts")
display(transition_counts)

print("Posterior confidence summary")
display(
    hmm_diagnostics_df[
        ["max_posterior", "posterior_entropy"]
    ].describe().round(4)
)

# Inspect whether runtime is associated with architecture size.
print("Runtime by architecture size")
display(
    hmm_diagnostics_df.groupby("base_channels")[
        ["duration_seconds", "normalized_duration"]
    ].agg(["count", "mean", "std"]).round(3)
)

# Visualize observations and posterior probabilities.
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(
    hmm_diagnostics_df["trial"] + 1,
    hmm_diagnostics_df["improvement"],
    marker="o",
    label="Reward improvement",
)
axes[0].plot(
    hmm_diagnostics_df["trial"] + 1,
    hmm_diagnostics_df["reward_variability"],
    marker="s",
    label="Reward variability",
)
axes[0].set_ylabel("Observation value")
axes[0].set_title("HMM Observation Signals")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for state, column in [
    ("Exploration", "posterior_exploration"),
    ("Exploitation", "posterior_exploitation"),
    ("Cooling", "posterior_cooling"),
]:
    axes[1].plot(
        hmm_diagnostics_df["trial"] + 1,
        hmm_diagnostics_df[column],
        marker="o",
        label=state,
    )

axes[1].set_xlabel("Completed trial")
axes[1].set_ylabel("Posterior probability")
axes[1].set_ylim(0, 1)
axes[1].set_title("Sequential HMM State Probabilities")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

diagnostic_path = (
    PROJECT_ROOT / "outputs" / "cross_validation"
    / "as_hmm_bo_controller_diagnostics.csv"
)
diagnostic_path.parent.mkdir(parents=True, exist_ok=True)
hmm_diagnostics_df.to_csv(diagnostic_path, index=False)

print(f"Diagnostics saved to: {diagnostic_path}")

assert np.isfinite(
    hmm_diagnostics_df[
        [
            "improvement",
            "reward_variability",
            "normalized_duration",
            "posterior_exploration",
            "posterior_exploitation",
            "posterior_cooling",
        ]
    ].to_numpy()
).all()

assert np.allclose(
    hmm_diagnostics_df[
        [
            "posterior_exploration",
            "posterior_exploitation",
            "posterior_cooling",
        ]
    ].sum(axis=1),
    1.0,
)

print("PASS: sequential diagnostic audit completed.")


# Cell 46 — Correct the HMM Observation Design

In [ ]:

# Cell 46: Corrected Search Observations and Online HMM Filtering

import numpy as np
import pandas as pd

# These are provisional, deliberately broad emission assumptions.
# Feature 1: improvement in best-so-far Dice.
# Feature 2: stagnation ratio, clipped to [0, 1].
CORRECTED_HMM_MEANS = np.array([
    [0.015, 0.10],  # exploration: searching broadly, limited stagnation
    [0.020, 0.05],  # exploitation: recent progress
    [0.000, 0.75],  # cooling: extended stagnation
], dtype=float)

CORRECTED_HMM_STDS = np.array([
    [0.035, 0.35],
    [0.030, 0.30],
    [0.020, 0.30],
], dtype=float)


def build_search_state_observations(trial_results):
    """Build fixed-scale observations without using model runtime."""
    ordered = sorted(trial_results, key=lambda row: row["trial"])

    if not ordered:
        raise ValueError("At least one completed trial is required")

    scores = np.asarray([
        float(row["mean_validation_image_dice"])
        for row in ordered
    ])

    if not np.isfinite(scores).all():
        raise ValueError("Trial scores must be finite")

    observations = []
    best_so_far = -np.inf
    trials_since_improvement = 0

    for score in scores:
        previous_best = best_so_far
        best_so_far = max(best_so_far, score)

        improvement = (
            max(0.0, best_so_far - previous_best)
            if np.isfinite(previous_best)
            else 0.0
        )

        if improvement > 1e-12:
            trials_since_improvement = 0
        else:
            trials_since_improvement += 1

        # A fixed denominator keeps earlier observations unchanged
        # when additional trials are appended.
        stagnation_ratio = min(trials_since_improvement / 4.0, 1.0)

        observations.append([improvement, stagnation_ratio])

    return np.asarray(observations, dtype=float)


def infer_corrected_hmm_state(observations):
    """Run a causal forward filter over observations in trial order."""
    observations = np.asarray(observations, dtype=float)

    if observations.ndim != 2 or observations.shape[1] != 2:
        raise ValueError("Expected observation shape (n, 2)")
    if len(observations) == 0 or not np.isfinite(observations).all():
        raise ValueError("Observations must be non-empty and finite")

    posterior = HMM_INITIAL.copy()
    records = []

    for index, observation in enumerate(observations):
        predicted = posterior @ HMM_TRANSITION

        log_likelihood = _normal_log_likelihood(
            observation,
            CORRECTED_HMM_MEANS,
            CORRECTED_HMM_STDS,
        )

        # Temper likelihoods to reduce excessive certainty from
        # manually specified emissions.
        log_posterior = (
            np.log(np.clip(predicted, 1e-300, None))
            + 0.5 * log_likelihood
        )
        log_posterior -= np.max(log_posterior)

        posterior = np.exp(log_posterior)
        posterior /= posterior.sum()

        state_index = int(np.argmax(posterior))

        records.append({
            "trial": index,
            "best_so_far_improvement": observation[0],
            "stagnation_ratio": observation[1],
            "selected_state": HMM_STATES[state_index],
            "posterior_exploration": posterior[0],
            "posterior_exploitation": posterior[1],
            "posterior_cooling": posterior[2],
            "max_posterior": posterior.max(),
            "posterior_entropy": -np.sum(
                posterior * np.log(np.clip(posterior, 1e-12, 1.0))
            ),
        })

    return pd.DataFrame(records)


corrected_observations = build_search_state_observations(
    as_hmm_trial_records
)
corrected_diagnostics_df = infer_corrected_hmm_state(
    corrected_observations
)

print("Corrected observations and sequential state estimates")
display(corrected_diagnostics_df.round(4))

print("State counts")
display(
    corrected_diagnostics_df["selected_state"]
    .value_counts()
    .reindex(HMM_STATES, fill_value=0)
    .rename("trial_count")
    .to_frame()
)

print("Posterior confidence summary")
display(
    corrected_diagnostics_df[
        ["max_posterior", "posterior_entropy"]
    ].describe().round(4)
)

# Check the key invariants.
posterior_columns = [
    "posterior_exploration",
    "posterior_exploitation",
    "posterior_cooling",
]

assert len(corrected_diagnostics_df) == len(as_hmm_trial_records)
assert np.isfinite(
    corrected_diagnostics_df[posterior_columns].to_numpy()
).all()
assert np.allclose(
    corrected_diagnostics_df[posterior_columns].sum(axis=1),
    1.0,
)
assert corrected_diagnostics_df["stagnation_ratio"].between(0, 1).all()

diagnostic_path = (
    PROJECT_ROOT / "outputs" / "cross_validation"
    / "as_hmm_bo_corrected_diagnostics.csv"
)
corrected_diagnostics_df.to_csv(diagnostic_path, index=False)

print(f"Saved corrected diagnostics to: {diagnostic_path}")
print("PASS: corrected observations and online filtering validated.")


# Cell 47 — Fix First-Trial Initialization

In [ ]:

# Cell 47: Correct First-Trial Initialization in Search Observations

def build_search_state_observations(trial_results):
    """Build causal, runtime-independent search-state observations."""
    ordered = sorted(trial_results, key=lambda row: row["trial"])

    if not ordered:
        raise ValueError("At least one completed trial is required")

    scores = np.asarray([
        float(row["mean_validation_image_dice"])
        for row in ordered
    ])

    if not np.isfinite(scores).all():
        raise ValueError("Trial scores must be finite")

    observations = []
    best_so_far = -np.inf
    trials_since_improvement = 0

    for index, score in enumerate(scores):
        if index == 0:
            # No previous trial exists, so improvement and stagnation
            # are both undefined operationally and initialized to zero.
            best_so_far = score
            observations.append([0.0, 0.0])
            continue

        previous_best = best_so_far
        best_so_far = max(best_so_far, score)

        improvement = max(0.0, best_so_far - previous_best)

        if improvement > 1e-12:
            trials_since_improvement = 0
        else:
            trials_since_improvement += 1

        stagnation_ratio = min(trials_since_improvement / 4.0, 1.0)
        observations.append([improvement, stagnation_ratio])

    return np.asarray(observations, dtype=float)


# Recompute the diagnostic trajectory with the corrected initialization.
corrected_observations = build_search_state_observations(
    as_hmm_trial_records
)
corrected_diagnostics_df = infer_corrected_hmm_state(
    corrected_observations
)

display(corrected_diagnostics_df.round(4))

assert np.allclose(corrected_observations[0], [0.0, 0.0])
assert np.isfinite(corrected_observations).all()
assert corrected_observations[:, 1].min() >= 0.0
assert corrected_observations[:, 1].max() <= 1.0

diagnostic_path = (
    PROJECT_ROOT / "outputs" / "cross_validation"
    / "as_hmm_bo_corrected_diagnostics.csv"
)
corrected_diagnostics_df.to_csv(diagnostic_path, index=False)

print("PASS: first-trial initialization corrected and diagnostics saved.")


# Cell 48 — Validate Online Controller Integration

In [ ]:

# Cell 48: Validate Online HMM Controller and Candidate Policies

import numpy as np
import pandas as pd

# 1. Verify append-invariance:
# Adding a future trial must not change earlier observations.
all_observations = build_search_state_observations(as_hmm_trial_records)

for end in range(1, len(as_hmm_trial_records) + 1):
    prefix_observations = build_search_state_observations(
        as_hmm_trial_records[:end]
    )
    assert np.allclose(
        prefix_observations,
        all_observations[:end],
        atol=1e-12,
    ), f"Append-invariance failed at prefix length {end}"

# 2. Verify the online HMM filter returns valid probabilities.
online_diagnostics = infer_corrected_hmm_state(all_observations)

posterior_columns = [
    "posterior_exploration",
    "posterior_exploitation",
    "posterior_cooling",
]

posterior_matrix = online_diagnostics[posterior_columns].to_numpy()

assert np.isfinite(posterior_matrix).all()
assert np.all(posterior_matrix >= 0.0)
assert np.allclose(posterior_matrix.sum(axis=1), 1.0, atol=1e-6)

# 3. Verify candidate-generation policies for every state.
# This is a policy smoke test, not a model-training experiment.
policy_rng = np.random.default_rng(SEED + 480)
policy_rows = []

for state in ["exploration", "exploitation", "cooling"]:
    candidate = propose_hmm_guided_params(
        state=state,
        completed_results=as_hmm_trial_records,
        rng=policy_rng,
    )

    required_keys = {
        "learning_rate",
        "weight_decay",
        "base_channels",
        "loss_bce_weight",
    }
    assert required_keys.issubset(candidate.keys())

    assert SEARCH_CONFIG["learning_rate"]["low"] <= (
        candidate["learning_rate"]
    ) <= SEARCH_CONFIG["learning_rate"]["high"]

    assert SEARCH_CONFIG["weight_decay"]["low"] <= (
        candidate["weight_decay"]
    ) <= SEARCH_CONFIG["weight_decay"]["high"]

    assert candidate["base_channels"] in SEARCH_CONFIG["base_channels"]

    assert SEARCH_CONFIG["loss_bce_weight"]["low"] <= (
        candidate["loss_bce_weight"]
    ) <= SEARCH_CONFIG["loss_bce_weight"]["high"]

    policy_rows.append({
        "state": state,
        **candidate,
    })

policy_validation_df = pd.DataFrame(policy_rows)

print("Append-invariance: PASS")
print("Posterior normalization: PASS")
print("Candidate parameter bounds: PASS")
display(policy_validation_df.round(7))


# Next step: Cell 49 — Audit the three sampling policies

In [ ]:

# Cell 49: Inspect HMM-Guided Candidate Sampling Policy

import inspect

print(inspect.getsource(propose_hmm_guided_params))


# Cell 50 — Improve HMM-Guided Sampling

In [ ]:

# Cell 50: Improved HMM-Guided Candidate Sampling

def _reflect_to_bounds(value, low, high):
    """Reflect a value into [low, high] without clipping pile-ups."""
    width = high - low
    if width <= 0:
        raise ValueError("Upper bound must exceed lower bound")

    position = (value - low) % (2.0 * width)
    if position > width:
        position = 2.0 * width - position

    return float(low + position)


def propose_hmm_guided_params(state, completed_results, rng):
    """
    Generate a candidate using a state-dependent sampling policy.

    Exploration:
        Broad sampling over the full configured search space.

    Exploitation:
        Local perturbations around the best completed configuration,
        with a 10% global restart probability.

    Cooling:
        Narrower local perturbations around the best configuration,
        with a 20% global restart probability.
    """
    if state not in HMM_STATES:
        raise ValueError(f"Unknown HMM state: {state}")

    lr_cfg = SEARCH_CONFIG["learning_rate"]
    wd_cfg = SEARCH_CONFIG["weight_decay"]
    bce_cfg = SEARCH_CONFIG["loss_bce_weight"]

    def global_sample():
        return {
            "learning_rate": float(np.exp(rng.uniform(
                np.log(lr_cfg["low"]), np.log(lr_cfg["high"])
            ))),
            "weight_decay": float(np.exp(rng.uniform(
                np.log(wd_cfg["low"]), np.log(wd_cfg["high"])
            ))),
            "base_channels": int(rng.choice(
                SEARCH_CONFIG["base_channels"]
            )),
            "loss_bce_weight": float(rng.uniform(
                bce_cfg["low"], bce_cfg["high"]
            )),
        }

    if state == "exploration" or not completed_results:
        return global_sample()

    best_row = max(
        completed_results,
        key=lambda row: row["mean_validation_image_dice"],
    )

    if state == "exploitation":
        lr_sigma, wd_sigma, bce_sigma = 0.30, 0.70, 0.10
        restart_probability = 0.10
    else:
        lr_sigma, wd_sigma, bce_sigma = 0.15, 0.40, 0.06
        restart_probability = 0.20

    if rng.random() < restart_probability:
        return global_sample()

    lr_log = rng.normal(np.log(best_row["learning_rate"]), lr_sigma)
    wd_log = rng.normal(np.log(best_row["weight_decay"]), wd_sigma)

    learning_rate = np.exp(_reflect_to_bounds(
        lr_log,
        np.log(lr_cfg["low"]),
        np.log(lr_cfg["high"]),
    ))
    weight_decay = np.exp(_reflect_to_bounds(
        wd_log,
        np.log(wd_cfg["low"]),
        np.log(wd_cfg["high"]),
    ))

    base_channels = (
        int(best_row["base_channels"])
        if rng.random() < 0.75
        else int(rng.choice(SEARCH_CONFIG["base_channels"]))
    )

    bce_weight = _reflect_to_bounds(
        rng.normal(best_row["loss_bce_weight"], bce_sigma),
        bce_cfg["low"],
        bce_cfg["high"],
    )

    return {
        "learning_rate": float(learning_rate),
        "weight_decay": float(weight_decay),
        "base_channels": base_channels,
        "loss_bce_weight": float(bce_weight),
    }


# Validate the improved sampler using the existing completed trials.
sampling_test_rows = []

for state_index, state in enumerate(
    ["exploration", "exploitation", "cooling"]
):
    test_rng = np.random.default_rng(SEED + 500 + state_index)
    candidates = [
        propose_hmm_guided_params(
            state=state,
            completed_results=as_hmm_trial_records,
            rng=test_rng,
        )
        for _ in range(500)
    ]

    candidate_df = pd.DataFrame(candidates)

    assert candidate_df["learning_rate"].between(
        lr_cfg["low"], lr_cfg["high"]
    ).all() if "lr_cfg" in globals() else candidate_df["learning_rate"].between(
        SEARCH_CONFIG["learning_rate"]["low"],
        SEARCH_CONFIG["learning_rate"]["high"],
    ).all()

    assert candidate_df["weight_decay"].between(
        SEARCH_CONFIG["weight_decay"]["low"],
        SEARCH_CONFIG["weight_decay"]["high"],
    ).all()

    assert candidate_df["loss_bce_weight"].between(
        SEARCH_CONFIG["loss_bce_weight"]["low"],
        SEARCH_CONFIG["loss_bce_weight"]["high"],
    ).all()

    sampling_test_rows.append({
        "state": state,
        "samples": len(candidate_df),
        "unique_learning_rates": candidate_df["learning_rate"].nunique(),
        "unique_weight_decays": candidate_df["weight_decay"].nunique(),
        "unique_channel_values": candidate_df["base_channels"].nunique(),
        "learning_rate_at_bounds": int(
            np.isclose(
                candidate_df["learning_rate"],
                SEARCH_CONFIG["learning_rate"]["low"],
            ).sum()
            + np.isclose(
                candidate_df["learning_rate"],
                SEARCH_CONFIG["learning_rate"]["high"],
            ).sum()
        ),
        "weight_decay_at_bounds": int(
            np.isclose(
                candidate_df["weight_decay"],
                SEARCH_CONFIG["weight_decay"]["low"],
            ).sum()
            + np.isclose(
                candidate_df["weight_decay"],
                SEARCH_CONFIG["weight_decay"]["high"],
            ).sum()
        ),
    })

sampling_audit_df = pd.DataFrame(sampling_test_rows)
display(sampling_audit_df)

print("PASS: 1,500 candidate proposals checked across all three states.")


In [ ]:

# Cell 52: Verify Search-Trial Result Schema Without Training

import inspect

print(inspect.getsource(train_search_trial))


# Cell 51 — Define the Corrected Online AS-HMM-BO Search

In [ ]:

# Cell 51: Corrected Online AS-HMM-BO Search

import time
import numpy as np
import pandas as pd


def run_corrected_as_hmm_bo(
    folds=None,
    trials_per_search=8,
    epochs_per_trial=None,
    seed=None,
):
    """
    Run HMM-guided hyperparameter optimization on authorized folds only.

    Each candidate is sampled using the state inferred from completed
    trials. The first candidate uses broad exploration.

    Returns:
        trials: one row per candidate
        folds: one row per candidate-fold evaluation
        states: posterior and controller decision per candidate
    """
    if seed is None:
        seed = int(SEARCH_CONFIG["seed"])

    if epochs_per_trial is None:
        epochs_per_trial = int(SEARCH_CONFIG["search_epochs"])

    if folds is None:
        folds = list(SEARCH_CONFIG["search_fold_ids"])
    else:
        folds = [int(fold_id) for fold_id in folds]

    authorized_folds = set(SEARCH_CONFIG["search_fold_ids"])

    if not folds:
        raise ValueError("At least one search fold is required.")

    if len(set(folds)) != len(folds):
        raise ValueError("Duplicate fold IDs are not allowed.")

    unauthorized = set(folds) - authorized_folds
    if unauthorized:
        raise ValueError(
            f"Unauthorized search folds: {sorted(unauthorized)}"
        )

    if trials_per_search < 1 or epochs_per_trial < 1:
        raise ValueError("Trial and epoch budgets must be positive.")

    rng = np.random.default_rng(seed)

    completed_results = []
    trial_rows = []
    fold_rows = []
    state_rows = []

    for trial_number in range(trials_per_search):

        # Infer the state using completed trials only.
        if not completed_results:
            posterior = np.array([1.0, 0.0, 0.0])
            state = "exploration"
        else:
            observations = build_search_state_observations(
                completed_results
            )
            diagnostics = infer_corrected_hmm_state(observations)
            latest = diagnostics.iloc[-1]

            posterior = np.array([
                latest["posterior_exploration"],
                latest["posterior_exploitation"],
                latest["posterior_cooling"],
            ], dtype=float)

            state = str(latest["selected_state"])

        if not np.isfinite(posterior).all():
            raise RuntimeError("HMM posterior contains non-finite values.")

        if not np.isclose(posterior.sum(), 1.0, atol=1e-6):
            raise RuntimeError("HMM posterior does not sum to one.")

        # The inferred state controls actual candidate generation.
        candidate = propose_hmm_guided_params(
            state=state,
            completed_results=completed_results,
            rng=rng,
        )

        candidate_fold_results = []
        trial_start = time.perf_counter()

        for fold_id in folds:
            result = train_search_trial(
                params=candidate,
                fold_id=fold_id,
                epochs=epochs_per_trial,
                seed=seed + trial_number * 100 + int(fold_id),
                return_history=True,
            )

            # Use the actual keys returned by train_search_trial.
            fold_result = {
                "trial": trial_number,
                "fold": int(fold_id),
                "selected_state": state,
                **candidate,
                "best_epoch": result["best_epoch"],
                "validation_dice": result[
                    "mean_validation_image_dice"
                ],
                "validation_loss": result["validation_loss"],
                "duration_seconds": result["duration_seconds"],
            }

            fold_rows.append(fold_result)
            candidate_fold_results.append(fold_result)

        total_duration = time.perf_counter() - trial_start

        mean_dice = float(np.mean([
            row["validation_dice"]
            for row in candidate_fold_results
        ]))
        mean_loss = float(np.mean([
            row["validation_loss"]
            for row in candidate_fold_results
        ]))

        # Store only compact metadata, not model weights or histories.
        trial_record = {
            "trial": trial_number,
            **candidate,
            "mean_validation_image_dice": mean_dice,
            "mean_validation_loss": mean_loss,
            "duration_seconds": float(total_duration),
            "selected_state": state,
        }

        completed_results.append(trial_record.copy())
        trial_rows.append(trial_record.copy())

        state_rows.append({
            "trial": trial_number,
            "selected_state": state,
            "posterior_exploration": float(posterior[0]),
            "posterior_exploitation": float(posterior[1]),
            "posterior_cooling": float(posterior[2]),
            "posterior_sum": float(posterior.sum()),
            "candidate_sampling_policy": state,
            "resource_budget_epochs_per_fold": int(epochs_per_trial),
            "trial_score": mean_dice,
            "trial_duration_seconds": float(total_duration),
            **candidate,
        })

        print(
            f"Trial {trial_number + 1}/{trials_per_search} | "
            f"state={state:<12} | "
            f"Dice={mean_dice:.4f} | "
            f"loss={mean_loss:.4f} | "
            f"time={total_duration:.1f}s"
        )

    trials_df = pd.DataFrame(trial_rows)
    folds_df = pd.DataFrame(fold_rows)
    states_df = pd.DataFrame(state_rows)

    assert len(trials_df) == trials_per_search
    assert len(folds_df) == trials_per_search * len(folds)
    assert np.allclose(states_df["posterior_sum"], 1.0, atol=1e-6)
    assert set(folds_df["fold"]).issubset(authorized_folds)

    return {
        "trials": trials_df,
        "folds": folds_df,
        "states": states_df,
    }


print("Corrected online search function defined.")
print("No model training has been run.")
print("Authorized folds:", SEARCH_CONFIG["search_fold_ids"])
print("Epochs per trial:", SEARCH_CONFIG["search_epochs"])


# Cell 52 — Validate Before Training

In [ ]:

# Cell 52: Pre-Run Search Configuration Validation

import inspect

assert "mean_validation_image_dice" in inspect.getsource(
    run_corrected_as_hmm_bo
)
assert 'result["validation_loss"]' in inspect.getsource(
    run_corrected_as_hmm_bo
)

assert set(SEARCH_CONFIG["search_fold_ids"]) == {3, 4}
assert SEARCH_CONFIG["test_sets_allowed_during_search"] is False

# Verify that unauthorized folds are rejected before training.
try:
    run_corrected_as_hmm_bo(
        folds=[1],
        trials_per_search=1,
        epochs_per_trial=1,
        seed=SEED,
    )
except ValueError as exc:
    assert "Unauthorized search folds" in str(exc)
else:
    raise AssertionError("Unauthorized fold was not rejected.")

print("PASS: result keys match the training engine.")
print("PASS: unauthorized folds are rejected.")
print("PASS: test sets remain excluded.")


In [ ]:
corrected_search_results = run_corrected_as_hmm_bo(
    folds=[3, 4],
    trials_per_search=8,
    epochs_per_trial=3,
    seed=SEED,
)

# Cell 53 — Summarize and Save the Corrected Search

In [ ]:

# Cell 53: Summarize and Persist Corrected AS-HMM-BO Results

from pathlib import Path
import numpy as np
import pandas as pd

corrected_trials_df = corrected_search_results["trials"].copy()
corrected_folds_df = corrected_search_results["folds"].copy()
corrected_states_df = corrected_search_results["states"].copy()

# Rank candidates by their mean validation objective.
ranked_trials_df = corrected_trials_df.sort_values(
    "mean_validation_image_dice",
    ascending=False,
).reset_index(drop=True)

score_column = "mean_validation_image_dice"

corrected_summary = pd.DataFrame([{
    "optimizer": "Corrected AS-HMM-BO",
    "trials": len(corrected_trials_df),
    "best_dice": corrected_trials_df[score_column].max(),
    "mean_dice": corrected_trials_df[score_column].mean(),
    "median_dice": corrected_trials_df[score_column].median(),
    "std_dice": corrected_trials_df[score_column].std(ddof=1),
    "mean_validation_loss": corrected_trials_df[
        "mean_validation_loss"
    ].mean(),
    "total_runtime_minutes": corrected_trials_df[
        "duration_seconds"
    ].sum() / 60.0,
    "best_trial": int(
        corrected_trials_df.loc[
            corrected_trials_df[score_column].idxmax(), "trial"
        ]
    ),
}])

print("Corrected search summary")
display(corrected_summary.round(4))

print("Trials ranked by mean validation Dice")
display(ranked_trials_df[[
    "trial",
    "selected_state",
    "learning_rate",
    "weight_decay",
    "base_channels",
    "loss_bce_weight",
    "mean_validation_image_dice",
    "mean_validation_loss",
    "duration_seconds",
]].round(5))

print("HMM state usage and posterior probabilities")
display(corrected_states_df[[
    "trial",
    "selected_state",
    "posterior_exploration",
    "posterior_exploitation",
    "posterior_cooling",
    "trial_score",
    "candidate_sampling_policy",
]].round(4))

print("Per-fold results")
display(corrected_folds_df[[
    "trial",
    "fold",
    "selected_state",
    "validation_dice",
    "validation_loss",
    "best_epoch",
]].sort_values(["trial", "fold"]).round(4))

# Save as a separate experiment; do not overwrite historical runs.
output_dir = PROJECT_ROOT / "outputs" / "cross_validation"
output_dir.mkdir(parents=True, exist_ok=True)

corrected_trials_df.to_csv(
    output_dir / "as_hmm_bo_corrected_trial_results.csv",
    index=False,
)
corrected_folds_df.to_csv(
    output_dir / "as_hmm_bo_corrected_fold_results.csv",
    index=False,
)
corrected_states_df.to_csv(
    output_dir / "as_hmm_bo_corrected_state_history.csv",
    index=False,
)
corrected_summary.to_csv(
    output_dir / "as_hmm_bo_corrected_summary.csv",
    index=False,
)

print(f"Saved corrected experiment outputs to: {output_dir}")
print("PASS: corrected results saved without overwriting prior runs.")


# Cell 54 — Diagnose the Live HMM Controller

In [ ]:

# Cell 54: Diagnose Corrected HMM State Behavior

import numpy as np
import pandas as pd

states_df = corrected_search_results["states"].copy()
trials_df = corrected_search_results["trials"].copy()

# Rebuild the observations from the completed search.
live_observations = build_search_state_observations(
    trials_df.to_dict("records")
)

# Independently replay the filter over the complete trajectory.
replayed_diagnostics = infer_corrected_hmm_state(live_observations)

print("1. Search observations")
observation_df = pd.DataFrame(
    live_observations,
    columns=["best_so_far_improvement", "stagnation_ratio"],
)
observation_df.insert(
    0, "trial", trials_df["trial"].to_numpy()
)
display(observation_df.round(4))

print("2. Live controller decisions")
display(states_df[[
    "trial",
    "selected_state",
    "posterior_exploration",
    "posterior_exploitation",
    "posterior_cooling",
    "trial_score",
]].round(4))

print("3. Independently replayed posterior trajectory")
display(replayed_diagnostics[[
    "trial",
    "selected_state",
    "posterior_exploration",
    "posterior_exploitation",
    "posterior_cooling",
    "max_posterior",
    "posterior_entropy",
]].round(4))

# Quantify posterior mass assigned to each state.
posterior_columns = [
    "posterior_exploration",
    "posterior_exploitation",
    "posterior_cooling",
]

posterior_summary = states_df[posterior_columns].agg(
    ["mean", "min", "max"]
).T
posterior_summary.columns = [
    "mean_posterior", "minimum_posterior", "maximum_posterior"
]

print("4. Posterior summary during live candidate selection")
display(posterior_summary.round(4))

print("5. Selected-state transition counts")
transition_df = pd.DataFrame({
    "previous_state": states_df["selected_state"].shift(1),
    "current_state": states_df["selected_state"],
}).dropna()

if not transition_df.empty:
    display(pd.crosstab(
        transition_df["previous_state"],
        transition_df["current_state"],
    ))
else:
    print("Not enough trials to calculate transitions.")

print("PASS: controller diagnostics generated without retraining.")


In [ ]:

# Cell 55: Synthetic HMM Cooling-State Stress Test

import numpy as np
import pandas as pd

# Simulate a search that improves initially and then stagnates.
synthetic_observations = np.array([
    [0.080, 0.00],
    [0.040, 0.00],
    [0.010, 0.00],
    [0.000, 0.25],
    [0.000, 0.50],
    [0.000, 0.75],
    [0.000, 1.00],
    [0.000, 1.00],
    [0.000, 1.00],
    [0.000, 1.00],
], dtype=float)

synthetic_diagnostics = infer_corrected_hmm_state(
    synthetic_observations
)

print("Synthetic search trajectory")
display(
    synthetic_diagnostics[
        [
            "trial",
            "best_so_far_improvement",
            "stagnation_ratio",
            "selected_state",
            "posterior_exploration",
            "posterior_exploitation",
            "posterior_cooling",
            "max_posterior",
        ]
    ].round(4)
)

cooling_summary = {
    "cooling_selected_trials": int(
        (synthetic_diagnostics["selected_state"] == "cooling").sum()
    ),
    "maximum_cooling_posterior": float(
        synthetic_diagnostics["posterior_cooling"].max()
    ),
    "final_cooling_posterior": float(
        synthetic_diagnostics["posterior_cooling"].iloc[-1]
    ),
}

print("\nCooling activation summary")
display(pd.DataFrame([cooling_summary]).round(4))

assert np.allclose(
    synthetic_diagnostics[
        [
            "posterior_exploration",
            "posterior_exploitation",
            "posterior_cooling",
        ]
    ].sum(axis=1).to_numpy(),
    1.0,
    atol=1e-6,
), "Posterior probabilities do not sum to one."

print("PASS: synthetic cooling-state stress test completed.")


# Cell 56: Extended Corrected AS-HMM-BO Search

In [ ]:

# Cell 56: Extended Corrected AS-HMM-BO Search

corrected_search_16 = run_corrected_as_hmm_bo(
    folds=[3, 4],
    trials_per_search=16,
    epochs_per_trial=3,
    seed=SEED,
)

extended_trials_df = corrected_search_16["trials"].copy()
extended_states_df = corrected_search_16["states"].copy()
extended_folds_df = corrected_search_16["folds"].copy()

print("Extended search summary")
print(f"Trials completed: {len(extended_trials_df)}")

display(
    extended_trials_df[
        [
            "trial",
            "selected_state",
            "mean_validation_image_dice",
            "validation_loss",
            "duration_seconds",
        ]
    ].round(4)
)

print("\nState selection counts")
display(
    extended_states_df["selected_state"]
    .value_counts()
    .reindex(
        ["exploration", "exploitation", "cooling"],
        fill_value=0,
    )
    .rename("trial_count")
    .to_frame()
)

print("\nPosterior probability summary")
posterior_cols = [
    "posterior_exploration",
    "posterior_exploitation",
    "posterior_cooling",
]

display(
    extended_states_df[posterior_cols]
    .agg(["mean", "min", "max"])
    .round(4)
)

print("\nBest observed trial")
best_trial_idx = extended_trials_df[
    "mean_validation_image_dice"
].idxmax()

display(
    extended_trials_df.loc[[best_trial_idx]].round(4)
)


# Cell 57: Inspect Results and Complete the Summary

In [ ]:
# Cell 57: Inspect Extended Search Results and Summarize Controller Behavior

import pandas as pd
import numpy as np

# 1. Inspect the actual output schemas.
print("Trial DataFrame columns:")
print(extended_trials_df.columns.tolist())

print("\nState DataFrame columns:")
print(extended_states_df.columns.tolist())

print("\nFold DataFrame columns:")
print(extended_folds_df.columns.tolist())

# 2. Combine trial results with state decisions if necessary.
if "selected_state" not in extended_trials_df.columns:
    state_columns = [
        col for col in [
            "trial",
            "selected_state",
            "posterior_exploration",
            "posterior_exploitation",
            "posterior_cooling",
        ]
        if col in extended_states_df.columns
    ]

    summary_df = extended_trials_df.merge(
        extended_states_df[state_columns],
        on="trial",
        how="left",
        validate="one_to_one",
    )
else:
    summary_df = extended_trials_df.copy()

# 3. Select columns that actually exist.
preferred_columns = [
    "trial",
    "selected_state",
    "mean_validation_image_dice",
    "validation_loss",
    "mean_validation_loss",
    "duration_seconds",
    "posterior_exploration",
    "posterior_exploitation",
    "posterior_cooling",
]

display_columns = [
    col for col in preferred_columns
    if col in summary_df.columns
]

print("\nExtended search results:")
display(summary_df[display_columns].round(4))

# 4. State-selection counts.
print("\nState-selection counts:")
state_counts = (
    summary_df["selected_state"]
    .value_counts()
    .reindex(["exploration", "exploitation", "cooling"], fill_value=0)
    .rename("trial_count")
    .to_frame()
)
display(state_counts)

# 5. Score summary.
score_column = (
    "mean_validation_image_dice"
    if "mean_validation_image_dice" in summary_df.columns
    else None
)

if score_column is not None:
    scores = pd.to_numeric(summary_df[score_column], errors="coerce")

    print("\nValidation Dice summary:")
    display(
        pd.DataFrame([{
            "trials": int(scores.notna().sum()),
            "best_dice": scores.max(),
            "mean_dice": scores.mean(),
            "median_dice": scores.median(),
            "std_dice": scores.std(),
            "best_trial": summary_df.loc[scores.idxmax(), "trial"],
        }]).round(4)
    )

    print("\nBest observed trial:")
    display(summary_df.loc[[scores.idxmax()], display_columns].round(4))
else:
    print("\nCould not identify the Dice column automatically.")

# 6. State-wise performance, if the score column is available.
if score_column is not None:
    print("\nValidation Dice by selected state:")
    display(
        summary_df.groupby("selected_state")[score_column]
        .agg(["count", "mean", "median", "max", "std"])
        .reindex(["exploration", "exploitation", "cooling"])
        .round(4)
    )

print("\nPASS: results summarized without retraining.")

# Cell 58: Save Extended Search Results and Controller Diagnostics

In [ ]:
# Cell 58: Persist Extended Search Results

from pathlib import Path
import json
import numpy as np
import pandas as pd

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "cross_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Copy the existing results.
extended_trials_df = corrected_search_16["trials"].copy()
extended_states_df = corrected_search_16["states"].copy()
extended_folds_df = corrected_search_16["folds"].copy()

# Merge state information into the trial-level table if needed.
if "selected_state" not in extended_trials_df.columns:
    state_columns = [
        col for col in [
            "trial",
            "selected_state",
            "posterior_exploration",
            "posterior_exploitation",
            "posterior_cooling",
        ]
        if col in extended_states_df.columns
    ]

    extended_summary_df = extended_trials_df.merge(
        extended_states_df[state_columns],
        on="trial",
        how="left",
        validate="one_to_one",
    )
else:
    extended_summary_df = extended_trials_df.copy()

# Persist detailed results.
trial_path = OUTPUT_DIR / "as_hmm_bo_extended16_trial_results.csv"
state_path = OUTPUT_DIR / "as_hmm_bo_extended16_state_history.csv"
fold_path = OUTPUT_DIR / "as_hmm_bo_extended16_fold_results.csv"

extended_summary_df.to_csv(trial_path, index=False)
extended_states_df.to_csv(state_path, index=False)
extended_folds_df.to_csv(fold_path, index=False)

# Identify the score column from the actual schema.
score_column = "mean_validation_image_dice"

if score_column not in extended_summary_df.columns:
    raise KeyError(
        f"Expected score column '{score_column}' was not found. "
        f"Available columns: {extended_summary_df.columns.tolist()}"
    )

scores = pd.to_numeric(
    extended_summary_df[score_column],
    errors="coerce",
)

if scores.isna().any():
    raise ValueError("Some trial Dice scores are missing or non-numeric.")

best_idx = scores.idxmax()

# Summarize search outcomes.
state_counts = (
    extended_summary_df["selected_state"]
    .value_counts()
    .reindex(["exploration", "exploitation", "cooling"], fill_value=0)
    .to_dict()
)

summary = {
    "experiment": "Corrected AS-HMM-BO extended search",
    "trials_completed": int(len(extended_summary_df)),
    "search_folds": [3, 4],
    "epochs_per_trial": 3,
    "best_dice": float(scores.max()),
    "mean_dice": float(scores.mean()),
    "median_dice": float(scores.median()),
    "std_dice": float(scores.std(ddof=1)),
    "best_trial": int(extended_summary_df.loc[best_idx, "trial"]),
    "state_counts": {
        key: int(value) for key, value in state_counts.items()
    },
    "testA_used_for_search": False,
    "testB_used_for_search": False,
}

summary_path = OUTPUT_DIR / "as_hmm_bo_extended16_summary.json"

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

print("Saved experiment artifacts:")
print(f"  Trial results: {trial_path}")
print(f"  State history: {state_path}")
print(f"  Fold results:  {fold_path}")
print(f"  Summary:       {summary_path}")

print("\nExperiment summary:")
display(pd.DataFrame([{
    key: value for key, value in summary.items()
    if key != "state_counts"
}]).round(4))

print("\nState counts:")
display(pd.Series(state_counts, name="trial_count").to_frame())

print("\nPASS: extended search results saved without retraining.")

# Cell 59: Define a controlled optimizer comparison

In [ ]:

# Cell 59: Freeze the Controlled Optimizer Comparison Protocol

import json
from pathlib import Path

COMPARISON_CONFIG = {
    "experiment_name": "AS-HMM-BO Controlled Optimizer Comparison",
    "version": "0.1.0",

    # Development folds already used for hyperparameter search.
    "search_folds": [3, 4],

    # Do not treat search folds as independent final evaluation data.
    "independent_test_sets": ["testA", "testB"],
    "test_sets_allowed_during_search": False,

    # Equal nominal search budgets.
    "optimizers": ["random_search", "optuna_tpe", "as_hmm_bo"],
    "trials_per_optimizer": 16,
    "epochs_per_trial": 3,
    "batch_size": 2,

    # Repeated runs to measure sensitivity to randomness.
    "seeds": [42, 123, 2026],

    # Fixed candidate search space.
    "search_space": {
        "learning_rate": {
            "type": "log_float",
            "low": 0.00005,
            "high": 0.0005
        },
        "weight_decay": {
            "type": "log_float",
            "low": 0.000001,
            "high": 0.001
        },
        "base_channels": [8, 16],
        "loss_bce_weight": {
            "type": "float",
            "low": 0.3,
            "high": 0.7
        }
    },

    "primary_metric": "best_so_far_mean_validation_image_dice",
    "secondary_metrics": [
        "validation_loss",
        "mean_image_iou",
        "mean_precision",
        "mean_recall",
        "wall_clock_seconds"
    ],

    "reporting": {
        "compare_at_matched_trial_counts": True,
        "compare_at_matched_compute_budgets": True,
        "report_mean_and_standard_deviation_across_seeds": True,
        "report_per_fold_results": True
    }
}

comparison_path = (
    PROJECT_ROOT
    / "outputs"
    / "cross_validation"
    / "optimizer_comparison_config.json"
)

comparison_path.parent.mkdir(parents=True, exist_ok=True)

with open(comparison_path, "w", encoding="utf-8") as file:
    json.dump(COMPARISON_CONFIG, file, indent=2)

print("Comparison protocol saved:")
print(comparison_path)

print("\nOptimizers:", COMPARISON_CONFIG["optimizers"])
print("Trials per optimizer:", COMPARISON_CONFIG["trials_per_optimizer"])
print("Epochs per trial:", COMPARISON_CONFIG["epochs_per_trial"])
print("Seeds:", COMPARISON_CONFIG["seeds"])
print("Search folds:", COMPARISON_CONFIG["search_folds"])
print("Test sets available to search:", 
      COMPARISON_CONFIG["test_sets_allowed_during_search"])

print("\nPASS: comparison protocol saved. No training performed.")


# Cell 60: Implement and Validate Random-Search Candidate Generation

In [ ]:

# Cell 60: Implement and Validate Random-Search Candidate Generation

import numpy as np
import pandas as pd

def sample_random_search_params(rng):
    """Sample one configuration uniformly over the defined search ranges."""
    space = COMPARISON_CONFIG["search_space"]

    return {
        "learning_rate": float(
            np.exp(
                rng.uniform(
                    np.log(space["learning_rate"]["low"]),
                    np.log(space["learning_rate"]["high"]),
                )
            )
        ),
        "weight_decay": float(
            np.exp(
                rng.uniform(
                    np.log(space["weight_decay"]["low"]),
                    np.log(space["weight_decay"]["high"]),
                )
            )
        ),
        "base_channels": int(rng.choice(space["base_channels"])),
        "loss_bce_weight": float(
            rng.uniform(
                space["loss_bce_weight"]["low"],
                space["loss_bce_weight"]["high"],
            )
        ),
    }


# Generate a reproducible preview without training.
rng = np.random.default_rng(COMPARISON_CONFIG["seeds"][0])

random_candidates = [
    sample_random_search_params(rng)
    for _ in range(16)
]

random_candidates_df = pd.DataFrame(random_candidates)
random_candidates_df.insert(
    0, "trial", np.arange(len(random_candidates_df))
)

# Validate parameter bounds.
space = COMPARISON_CONFIG["search_space"]

assert random_candidates_df["learning_rate"].between(
    space["learning_rate"]["low"],
    space["learning_rate"]["high"],
).all()

assert random_candidates_df["weight_decay"].between(
    space["weight_decay"]["low"],
    space["weight_decay"]["high"],
).all()

assert random_candidates_df["base_channels"].isin(
    space["base_channels"]
).all()

assert random_candidates_df["loss_bce_weight"].between(
    space["loss_bce_weight"]["low"],
    space["loss_bce_weight"]["high"],
).all()

# Confirm reproducibility.
rng_repeat = np.random.default_rng(COMPARISON_CONFIG["seeds"][0])
repeat_candidates = [
    sample_random_search_params(rng_repeat)
    for _ in range(16)
]

assert random_candidates == repeat_candidates, (
    "Random candidate generation is not reproducible."
)

print("Preview of 16 random-search configurations:")
display(random_candidates_df.round(7))

print("\nUnique configurations:", len(
    random_candidates_df.drop_duplicates(
        subset=[
            "learning_rate",
            "weight_decay",
            "base_channels",
            "loss_bce_weight",
        ]
    )
))

print("PASS: parameter bounds and reproducibility verified.")
print("No model training performed.")


# Cell 61: Implement and Validate the TPE Sampler

In [ ]:

# Cell 61: Implement and Validate TPE Candidate Generation

import optuna
import numpy as np
import pandas as pd

def suggest_tpe_params(trial):
    """Suggest one configuration from the shared comparison search space."""
    space = COMPARISON_CONFIG["search_space"]

    return {
        "learning_rate": trial.suggest_float(
            "learning_rate",
            space["learning_rate"]["low"],
            space["learning_rate"]["high"],
            log=True,
        ),
        "weight_decay": trial.suggest_float(
            "weight_decay",
            space["weight_decay"]["low"],
            space["weight_decay"]["high"],
            log=True,
        ),
        "base_channels": trial.suggest_categorical(
            "base_channels",
            space["base_channels"],
        ),
        "loss_bce_weight": trial.suggest_float(
            "loss_bce_weight",
            space["loss_bce_weight"]["low"],
            space["loss_bce_weight"]["high"],
        ),
    }


# Use a temporary study to validate suggestions only.
# No objective function is run and no model training occurs.
tpe_sampler = optuna.samplers.TPESampler(
    seed=COMPARISON_CONFIG["seeds"][0]
)

tpe_study = optuna.create_study(
    direction="maximize",
    sampler=tpe_sampler,
)

tpe_preview = []

for trial_number in range(16):
    trial = tpe_study.ask()
    params = suggest_tpe_params(trial)

    tpe_preview.append({
        "trial": trial_number,
        **params,
    })

    # Complete with a neutral placeholder so subsequent suggestions
    # can be generated and the sampler interface can be tested.
    # These placeholder values are NOT experimental results.
    tpe_study.tell(trial, 0.0)

tpe_preview_df = pd.DataFrame(tpe_preview)

print("Preview of TPE-generated configurations:")
display(tpe_preview_df.round(7))

# Validate parameter bounds.
space = COMPARISON_CONFIG["search_space"]

assert tpe_preview_df["learning_rate"].between(
    space["learning_rate"]["low"],
    space["learning_rate"]["high"],
).all()

assert tpe_preview_df["weight_decay"].between(
    space["weight_decay"]["low"],
    space["weight_decay"]["high"],
).all()

assert tpe_preview_df["base_channels"].isin(
    space["base_channels"]
).all()

assert tpe_preview_df["loss_bce_weight"].between(
    space["loss_bce_weight"]["low"],
    space["loss_bce_weight"]["high"],
).all()

print("\nUnique configurations:", len(
    tpe_preview_df.drop_duplicates(
        subset=[
            "learning_rate",
            "weight_decay",
            "base_channels",
            "loss_bce_weight",
        ]
    )
))

print("PASS: TPE suggestions respect all parameter bounds.")
print("No model training performed.")
print("Important: placeholder objective values are for interface testing only.")


# Cell 62: Add Fixed-Budget Trial Evaluation

In [ ]:

# Cell 62: Correct Fixed-Budget Evaluator and Smoke Test

import numpy as np
import pandas as pd

def evaluate_fixed_budget_candidate(
    params,
    fold_id,
    seed,
    epochs=3,
):
    """
    Evaluate a candidate using its final-epoch validation metrics.
    All candidates receive the same number of training epochs.
    """
    result = train_search_trial(
        params=params,
        fold_id=fold_id,
        epochs=epochs,
        seed=seed,
        return_history=True,
    )

    history = result.get("history")

    if not history or len(history) != epochs:
        raise ValueError(
            f"Expected {epochs} history entries; "
            f"received {0 if not history else len(history)}."
        )

    final_epoch = history[-1]

    required_metrics = [
        "validation_loss",
        "mean_image_dice",
        "mean_image_iou",
        "mean_precision",
        "mean_recall",
    ]

    missing = [
        key for key in required_metrics
        if key not in final_epoch
    ]

    if missing:
        raise KeyError(
            f"Final history entry is missing metrics: {missing}. "
            f"Available keys: {list(final_epoch.keys())}"
        )

    metrics = {
        key: float(final_epoch[key])
        for key in required_metrics
    }

    if not all(np.isfinite(value) for value in metrics.values()):
        raise ValueError("Final-epoch metrics contain non-finite values.")

    return {
        "fold": int(fold_id),
        "seed": int(seed),
        "epochs_completed": len(history),
        "final_epoch": int(final_epoch.get("epoch", epochs)),
        **metrics,
        "duration_seconds": float(result["duration_seconds"]),
    }


# One smoke test only; no full search is launched.
smoke_params = {
    "learning_rate": 0.0002,
    "weight_decay": 0.0001,
    "base_channels": 8,
    "loss_bce_weight": 0.5,
}

smoke_result = evaluate_fixed_budget_candidate(
    params=smoke_params,
    fold_id=3,
    seed=SEED + 9000,
    epochs=3,
)

print("Fixed-budget smoke-test result:")
display(pd.DataFrame([smoke_result]).round(4))

assert smoke_result["epochs_completed"] == 3
assert smoke_result["final_epoch"] == 3

print("PASS: final-epoch metrics extracted correctly.")


# Cell 64: Shared Candidate Evaluation Across Search Folds

In [ ]:

# Cell 64: Shared Multi-Fold Candidate Evaluation

import numpy as np
import pandas as pd
import time

def evaluate_candidate_across_folds(
    params,
    trial_number,
    seed,
    folds=None,
    epochs=None,
):
    """
    Evaluate one hyperparameter configuration on the same
    development folds and with the same epoch budget.
    """
    if folds is None:
        folds = COMPARISON_CONFIG["search_folds"]

    if epochs is None:
        epochs = COMPARISON_CONFIG["epochs_per_trial"]

    allowed_folds = set(COMPARISON_CONFIG["search_folds"])
    folds = list(folds)

    if not folds:
        raise ValueError("At least one search fold is required.")

    if not set(folds).issubset(allowed_folds):
        raise ValueError(
            f"Only configured development folds are allowed: "
            f"{sorted(allowed_folds)}"
        )

    fold_results = []
    start_time = time.perf_counter()

    for fold_id in folds:
        # Same seed schedule across optimizers for matched trial/fold.
        fold_seed = int(seed + trial_number * 100 + fold_id)

        metrics = evaluate_fixed_budget_candidate(
            params=params,
            fold_id=fold_id,
            seed=fold_seed,
            epochs=epochs,
        )

        fold_results.append({
            "trial": int(trial_number),
            "fold": int(fold_id),
            "optimizer_seed": int(seed),
            **params,
            **metrics,
        })

    wall_clock_seconds = time.perf_counter() - start_time

    fold_df = pd.DataFrame(fold_results)

    return {
        "trial": int(trial_number),
        **params,
        "mean_validation_dice": float(
            fold_df["mean_image_dice"].mean()
        ),
        "std_validation_dice_across_folds": float(
            fold_df["mean_image_dice"].std(ddof=0)
        ),
        "mean_validation_loss": float(
            fold_df["validation_loss"].mean()
        ),
        "mean_image_iou": float(
            fold_df["mean_image_iou"].mean()
        ),
        "mean_precision": float(
            fold_df["mean_precision"].mean()
        ),
        "mean_recall": float(
            fold_df["mean_recall"].mean()
        ),
        "training_seconds": float(
            fold_df["duration_seconds"].sum()
        ),
        "wall_clock_seconds": float(wall_clock_seconds),
    }, fold_results


print("Shared candidate evaluator defined.")
print("Search folds:", COMPARISON_CONFIG["search_folds"])
print("Epochs per fold:", COMPARISON_CONFIG["epochs_per_trial"])
print("No training was launched by this cell.")


In [ ]:

# Cell 64: Shared Multi-Fold Candidate Evaluation

import numpy as np
import pandas as pd
import time

def evaluate_candidate_across_folds(
    params,
    trial_number,
    seed,
    folds=None,
    epochs=None,
):
    """
    Evaluate one hyperparameter configuration on the same
    development folds and with the same epoch budget.
    """
    if folds is None:
        folds = COMPARISON_CONFIG["search_folds"]

    if epochs is None:
        epochs = COMPARISON_CONFIG["epochs_per_trial"]

    allowed_folds = set(COMPARISON_CONFIG["search_folds"])
    folds = list(folds)

    if not folds:
        raise ValueError("At least one search fold is required.")

    if not set(folds).issubset(allowed_folds):
        raise ValueError(
            f"Only configured development folds are allowed: "
            f"{sorted(allowed_folds)}"
        )

    fold_results = []
    start_time = time.perf_counter()

    for fold_id in folds:
        # Same seed schedule across optimizers for matched trial/fold.
        fold_seed = int(seed + trial_number * 100 + fold_id)

        metrics = evaluate_fixed_budget_candidate(
            params=params,
            fold_id=fold_id,
            seed=fold_seed,
            epochs=epochs,
        )

        fold_results.append({
            "trial": int(trial_number),
            "fold": int(fold_id),
            "optimizer_seed": int(seed),
            **params,
            **metrics,
        })

    wall_clock_seconds = time.perf_counter() - start_time

    fold_df = pd.DataFrame(fold_results)

    return {
        "trial": int(trial_number),
        **params,
        "mean_validation_dice": float(
            fold_df["mean_image_dice"].mean()
        ),
        "std_validation_dice_across_folds": float(
            fold_df["mean_image_dice"].std(ddof=0)
        ),
        "mean_validation_loss": float(
            fold_df["validation_loss"].mean()
        ),
        "mean_image_iou": float(
            fold_df["mean_image_iou"].mean()
        ),
        "mean_precision": float(
            fold_df["mean_precision"].mean()
        ),
        "mean_recall": float(
            fold_df["mean_recall"].mean()
        ),
        "training_seconds": float(
            fold_df["duration_seconds"].sum()
        ),
        "wall_clock_seconds": float(wall_clock_seconds),
    }, fold_results


print("Shared candidate evaluator defined.")
print("Search folds:", COMPARISON_CONFIG["search_folds"])
print("Epochs per fold:", COMPARISON_CONFIG["epochs_per_trial"])
print("No training was launched by this cell.")


# Cell 65: Random Search Runner

In [ ]:

# Cell 65: Random Search Runner

import numpy as np
import pandas as pd
import time

def run_random_search_comparison(
    trials_per_search=None,
    seed=None,
    folds=None,
    epochs=None,
):
    if trials_per_search is None:
        trials_per_search = COMPARISON_CONFIG["trials_per_optimizer"]

    if seed is None:
        seed = COMPARISON_CONFIG["seeds"][0]

    if folds is None:
        folds = COMPARISON_CONFIG["search_folds"]

    if epochs is None:
        epochs = COMPARISON_CONFIG["epochs_per_trial"]

    rng = np.random.default_rng(seed)

    trial_records = []
    fold_records = []
    best_so_far = float("-inf")
    cumulative_seconds = 0.0

    for trial_number in range(1, trials_per_search + 1):
        params = sample_random_search_params(rng)

        trial_result, current_fold_records = (
            evaluate_candidate_across_folds(
                params=params,
                trial_number=trial_number,
                seed=seed,
                folds=folds,
                epochs=epochs,
            )
        )

        cumulative_seconds += trial_result["wall_clock_seconds"]
        best_so_far = max(
            best_so_far,
            trial_result["mean_validation_dice"],
        )

        trial_records.append({
            "optimizer": "random_search",
            "seed": int(seed),
            **trial_result,
            "best_so_far_mean_validation_dice": float(best_so_far),
            "cumulative_wall_clock_seconds": float(cumulative_seconds),
        })

        for record in current_fold_records:
            fold_records.append({
                "optimizer": "random_search",
                **record,
            })

        print(
            f"Random Search {trial_number}/{trials_per_search} | "
            f"Dice={trial_result['mean_validation_dice']:.4f} | "
            f"Best={best_so_far:.4f} | "
            f"Elapsed={cumulative_seconds / 60:.2f} min"
        )

    trial_df = pd.DataFrame(trial_records)
    fold_df = pd.DataFrame(fold_records)

    summary = {
        "optimizer": "random_search",
        "seed": int(seed),
        "trials_completed": int(len(trial_df)),
        "search_folds": list(folds),
        "epochs_per_trial_per_fold": int(epochs),
        "best_mean_validation_dice": float(
            trial_df["mean_validation_dice"].max()
        ),
        "mean_trial_dice": float(
            trial_df["mean_validation_dice"].mean()
        ),
        "total_wall_clock_seconds": float(
            trial_df["wall_clock_seconds"].sum()
        ),
        "test_sets_used": False,
    }

    return {
        "trials": trial_df,
        "folds": fold_df,
        "summary": summary,
    }


print("Random-search runner defined; no trials have been trained.")


# Cell 66: Random Search Smoke Test

In [ ]:

# Cell 66: Random Search Smoke Test — One Trial

random_smoke = run_random_search_comparison(
    trials_per_search=1,
    seed=42,
    folds=[3, 4],
    epochs=3,
)

print("\nTrial result:")
display(
    random_smoke["trials"][
        [
            "optimizer",
            "trial",
            "learning_rate",
            "weight_decay",
            "base_channels",
            "loss_bce_weight",
            "mean_validation_dice",
            "best_so_far_mean_validation_dice",
            "cumulative_wall_clock_seconds",
        ]
    ].round(4)
)

print("\nPer-fold results:")
display(
    random_smoke["folds"][
        [
            "fold",
            "seed",
            "mean_image_dice",
            "validation_loss",
            "mean_image_iou",
            "duration_seconds",
        ]
    ].round(4)
)

print("\nSmoke-test summary:")
display(pd.DataFrame([random_smoke["summary"]]))

assert len(random_smoke["trials"]) == 1
assert len(random_smoke["folds"]) == 2
assert random_smoke["summary"]["test_sets_used"] is False

print("PASS: random-search runner completed one trial on both folds.")


# Cell 67: Optuna TPE Comparison Runner

In [ ]:

# Cell 67: Optuna TPE Comparison Runner

import optuna
import pandas as pd
import time

def run_tpe_comparison(
    trials_per_search=None,
    seed=None,
    folds=None,
    epochs=None,
):
    if trials_per_search is None:
        trials_per_search = COMPARISON_CONFIG["trials_per_optimizer"]

    if seed is None:
        seed = COMPARISON_CONFIG["seeds"][0]

    if folds is None:
        folds = COMPARISON_CONFIG["search_folds"]

    if epochs is None:
        epochs = COMPARISON_CONFIG["epochs_per_trial"]

    trial_records = []
    fold_records = []
    cumulative_seconds = 0.0
    best_so_far = float("-inf")

    sampler = optuna.samplers.TPESampler(seed=seed)
    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
    )

    def objective(trial):
        nonlocal cumulative_seconds, best_so_far

        params = suggest_tpe_params(trial)

        trial_result, current_fold_records = (
            evaluate_candidate_across_folds(
                params=params,
                trial_number=trial.number + 1,
                seed=seed,
                folds=folds,
                epochs=epochs,
            )
        )

        cumulative_seconds += trial_result["wall_clock_seconds"]
        best_so_far = max(
            best_so_far,
            trial_result["mean_validation_dice"],
        )

        trial_records.append({
            "optimizer": "optuna_tpe",
            "seed": int(seed),
            **trial_result,
            "best_so_far_mean_validation_dice": float(best_so_far),
            "cumulative_wall_clock_seconds": float(cumulative_seconds),
        })

        for record in current_fold_records:
            fold_records.append({
                "optimizer": "optuna_tpe",
                **record,
            })

        print(
            f"TPE {trial.number + 1}/{trials_per_search} | "
            f"Dice={trial_result['mean_validation_dice']:.4f} | "
            f"Best={best_so_far:.4f} | "
            f"Elapsed={cumulative_seconds / 60:.2f} min"
        )

        return trial_result["mean_validation_dice"]

    study.optimize(
        objective,
        n_trials=trials_per_search,
        show_progress_bar=False,
    )

    trial_df = pd.DataFrame(trial_records).sort_values("trial")
    fold_df = pd.DataFrame(fold_records).sort_values(["trial", "fold"])

    summary = {
        "optimizer": "optuna_tpe",
        "seed": int(seed),
        "trials_completed": int(len(trial_df)),
        "search_folds": list(folds),
        "epochs_per_trial_per_fold": int(epochs),
        "best_mean_validation_dice": float(
            trial_df["mean_validation_dice"].max()
        ),
        "mean_trial_dice": float(
            trial_df["mean_validation_dice"].mean()
        ),
        "total_wall_clock_seconds": float(
            trial_df["wall_clock_seconds"].sum()
        ),
        "test_sets_used": False,
    }

    return {
        "trials": trial_df,
        "folds": fold_df,
        "summary": summary,
        "study": study,
    }


print("TPE comparison runner defined; no trials have been trained.")


# Cell 68: TPE Smoke Test

In [ ]:

# Cell 68: TPE Smoke Test — One Trial

tpe_smoke = run_tpe_comparison(
    trials_per_search=1,
    seed=42,
    folds=[3, 4],
    epochs=3,
)

print("\nTPE trial result:")
display(
    tpe_smoke["trials"][
        [
            "trial",
            "learning_rate",
            "weight_decay",
            "base_channels",
            "loss_bce_weight",
            "mean_validation_dice",
            "best_so_far_mean_validation_dice",
            "cumulative_wall_clock_seconds",
        ]
    ].round(6)
)

print("\nTPE per-fold results:")
display(
    tpe_smoke["folds"][
        [
            "trial",
            "fold",
            "seed",
            "mean_image_dice",
            "validation_loss",
            "duration_seconds",
        ]
    ].round(4)
)

assert len(tpe_smoke["trials"]) == 1
assert len(tpe_smoke["folds"]) == 2
assert tpe_smoke["summary"]["test_sets_used"] is False

print("PASS: TPE runner completed one real objective evaluation.")


# Cell 69: Inspect the Existing HMM Controller

In [ ]:

# Cell 69: Inspect HMM Controller Interfaces

import inspect

functions_to_inspect = [
    build_search_state_observations,
    infer_corrected_hmm_state,
    propose_hmm_guided_params,
    run_corrected_as_hmm_bo,
]

for function in functions_to_inspect:
    print("\n" + "=" * 80)
    print("FUNCTION:", function.__name__)
    print("SIGNATURE:", inspect.signature(function))
    print("-" * 80)
    print(inspect.getsource(function))


# Cell 70: Controlled AS-HMM-BO Runner

In [ ]:

# Cell 70: Controlled AS-HMM-BO Comparison Runner

import numpy as np
import pandas as pd
import time

def run_as_hmm_bo_comparison(
    trials_per_search=None,
    seed=None,
    folds=None,
    epochs=None,
):
    if trials_per_search is None:
        trials_per_search = COMPARISON_CONFIG["trials_per_optimizer"]

    if seed is None:
        seed = COMPARISON_CONFIG["seeds"][0]

    if folds is None:
        folds = COMPARISON_CONFIG["search_folds"]

    if epochs is None:
        epochs = COMPARISON_CONFIG["epochs_per_trial"]

    folds = list(folds)
    allowed_folds = set(COMPARISON_CONFIG["search_folds"])

    if not folds or not set(folds).issubset(allowed_folds):
        raise ValueError(
            f"Use only configured development folds: "
            f"{sorted(allowed_folds)}"
        )

    if len(set(folds)) != len(folds):
        raise ValueError("Duplicate fold IDs are not allowed.")

    if trials_per_search < 1 or epochs < 1:
        raise ValueError("Trial and epoch budgets must be positive.")

    rng = np.random.default_rng(seed)

    completed_results = []
    trial_records = []
    fold_records = []
    state_records = []

    best_so_far = float("-inf")
    cumulative_seconds = 0.0

    for trial_number in range(1, trials_per_search + 1):

        # Infer the state using completed trials only.
        if not completed_results:
            posterior = np.array([1.0, 0.0, 0.0])
            state = "exploration"
        else:
            observations = build_search_state_observations(
                completed_results
            )
            diagnostics = infer_corrected_hmm_state(observations)
            latest = diagnostics.iloc[-1]

            posterior = np.array([
                latest["posterior_exploration"],
                latest["posterior_exploitation"],
                latest["posterior_cooling"],
            ], dtype=float)

            state = str(latest["selected_state"])

        if (
            not np.isfinite(posterior).all()
            or not np.isclose(posterior.sum(), 1.0, atol=1e-6)
        ):
            raise RuntimeError("Invalid HMM posterior.")

        # The state changes the actual candidate-generation policy.
        candidate = propose_hmm_guided_params(
            state=state,
            completed_results=completed_results,
            rng=rng,
        )

        trial_result, current_fold_records = (
            evaluate_candidate_across_folds(
                params=candidate,
                trial_number=trial_number,
                seed=seed,
                folds=folds,
                epochs=epochs,
            )
        )

        cumulative_seconds += trial_result["wall_clock_seconds"]

        score = float(trial_result["mean_validation_dice"])
        best_so_far = max(best_so_far, score)

        # This key is required by the existing HMM observation builder
        # and candidate proposer. The score is final-epoch Dice.
        completed_results.append({
            "trial": trial_number,
            **candidate,
            "mean_validation_image_dice": score,
            "mean_validation_loss": trial_result["mean_validation_loss"],
        })

        trial_records.append({
            "optimizer": "as_hmm_bo",
            "seed": int(seed),
            "selected_state": state,
            **trial_result,
            "best_so_far_mean_validation_dice": float(best_so_far),
            "cumulative_wall_clock_seconds": float(cumulative_seconds),
        })

        for record in current_fold_records:
            fold_records.append({
                "optimizer": "as_hmm_bo",
                "selected_state": state,
                **record,
            })

        state_records.append({
            "trial": trial_number,
            "selected_state": state,
            "posterior_exploration": float(posterior[0]),
            "posterior_exploitation": float(posterior[1]),
            "posterior_cooling": float(posterior[2]),
            "posterior_sum": float(posterior.sum()),
            "candidate_sampling_policy": state,
            "resource_budget_epochs_per_fold": int(epochs),
            "trial_score": score,
            "trial_duration_seconds": trial_result["wall_clock_seconds"],
            **candidate,
        })

        print(
            f"AS-HMM-BO {trial_number}/{trials_per_search} | "
            f"state={state:<12} | "
            f"Dice={score:.4f} | "
            f"Best={best_so_far:.4f} | "
            f"Elapsed={cumulative_seconds / 60:.2f} min"
        )

    trial_df = pd.DataFrame(trial_records)
    fold_df = pd.DataFrame(fold_records)
    state_df = pd.DataFrame(state_records)

    assert len(trial_df) == trials_per_search
    assert len(fold_df) == trials_per_search * len(folds)
    assert np.allclose(state_df["posterior_sum"], 1.0, atol=1e-6)
    assert set(fold_df["fold"]).issubset(allowed_folds)

    summary = {
        "optimizer": "as_hmm_bo",
        "seed": int(seed),
        "trials_completed": int(len(trial_df)),
        "search_folds": folds,
        "epochs_per_trial_per_fold": int(epochs),
        "best_mean_validation_dice": float(
            trial_df["mean_validation_dice"].max()
        ),
        "mean_trial_dice": float(
            trial_df["mean_validation_dice"].mean()
        ),
        "total_wall_clock_seconds": float(
            trial_df["wall_clock_seconds"].sum()
        ),
        "state_counts": {
            str(state_name): int(count)
            for state_name, count
            in state_df["selected_state"].value_counts().items()
        },
        "test_sets_used": False,
    }

    return {
        "trials": trial_df,
        "folds": fold_df,
        "states": state_df,
        "summary": summary,
    }


print("Controlled AS-HMM-BO runner defined.")
print("No training was launched.")


# Cell 71: AS-HMM-BO Smoke Test

In [ ]:

# Cell 71: AS-HMM-BO Smoke Test — One Trial

hmm_smoke = run_as_hmm_bo_comparison(
    trials_per_search=1,
    seed=42,
    folds=[3, 4],
    epochs=3,
)

print("\nTrial result:")
display(
    hmm_smoke["trials"][
        [
            "trial",
            "selected_state",
            "learning_rate",
            "weight_decay",
            "base_channels",
            "loss_bce_weight",
            "mean_validation_dice",
            "best_so_far_mean_validation_dice",
            "cumulative_wall_clock_seconds",
        ]
    ].round(6)
)

print("\nPer-fold results:")
display(
    hmm_smoke["folds"][
        [
            "trial",
            "fold",
            "seed",
            "mean_image_dice",
            "validation_loss",
            "duration_seconds",
        ]
    ].round(4)
)

print("\nHMM state diagnostics:")
display(hmm_smoke["states"].round(4))

print("\nSmoke-test summary:")
display(pd.DataFrame([hmm_smoke["summary"]]))

assert len(hmm_smoke["trials"]) == 1
assert len(hmm_smoke["folds"]) == 2
assert hmm_smoke["trials"].iloc[0]["selected_state"] == "exploration"
assert np.isclose(hmm_smoke["states"].iloc[0]["posterior_sum"], 1.0)
assert hmm_smoke["summary"]["test_sets_used"] is False

print("PASS: AS-HMM-BO uses the shared fixed-budget evaluator.")


# Cell 72: Validate Search-Space Consistency

In [ ]:

# Cell 72: Validate Shared Search Space and Sampler Bounds

import numpy as np

comparison_space = COMPARISON_CONFIG["search_space"]

hmm_space = {
    "learning_rate": SEARCH_CONFIG["learning_rate"],
    "weight_decay": SEARCH_CONFIG["weight_decay"],
    "base_channels": SEARCH_CONFIG["base_channels"],
    "loss_bce_weight": SEARCH_CONFIG["loss_bce_weight"],
}

assert np.isclose(
    comparison_space["learning_rate"]["low"],
    hmm_space["learning_rate"]["low"],
)
assert np.isclose(
    comparison_space["learning_rate"]["high"],
    hmm_space["learning_rate"]["high"],
)
assert np.isclose(
    comparison_space["weight_decay"]["low"],
    hmm_space["weight_decay"]["low"],
)
assert np.isclose(
    comparison_space["weight_decay"]["high"],
    hmm_space["weight_decay"]["high"],
)
assert comparison_space["base_channels"] == hmm_space["base_channels"]
assert np.isclose(
    comparison_space["loss_bce_weight"]["low"],
    hmm_space["loss_bce_weight"]["low"],
)
assert np.isclose(
    comparison_space["loss_bce_weight"]["high"],
    hmm_space["loss_bce_weight"]["high"],
)

print("PASS: comparison and HMM search-space bounds match.")

print("\nConfigured comparison budget:")
print("Optimizers:", COMPARISON_CONFIG["optimizers"])
print("Trials per optimizer:", COMPARISON_CONFIG["trials_per_optimizer"])
print("Seeds:", COMPARISON_CONFIG["seeds"])
print("Search folds:", COMPARISON_CONFIG["search_folds"])
print("Epochs per fold:", COMPARISON_CONFIG["epochs_per_trial"])

total_fold_evaluations = (
    len(COMPARISON_CONFIG["optimizers"])
    * len(COMPARISON_CONFIG["seeds"])
    * COMPARISON_CONFIG["trials_per_optimizer"]
    * len(COMPARISON_CONFIG["search_folds"])
)

total_training_epochs = (
    total_fold_evaluations
    * COMPARISON_CONFIG["epochs_per_trial"]
)

print("\nPlanned full experiment:")
print("Candidate-fold evaluations:", total_fold_evaluations)
print("Training epochs:", total_training_epochs)
print("Independent test sets used during search: False")


# Cell 73: Full Random Search — Seed 42

In [ ]:

# Cell 73: Full Random Search — Seed 42

import json
from pathlib import Path
import pandas as pd

random_seed42 = run_random_search_comparison(
    trials_per_search=16,
    seed=42,
    folds=[3, 4],
    epochs=3,
)

# Resolve the project root from the expected notebook directory.
current_dir = Path.cwd()
project_root = (
    current_dir.parent
    if current_dir.name.lower() == "notebooks"
    else current_dir
)

results_dir = project_root / "outputs" / "cross_validation"
results_dir.mkdir(parents=True, exist_ok=True)

random_seed42["trials"].to_csv(
    results_dir / "random_search_seed42_trials.csv",
    index=False,
)
random_seed42["folds"].to_csv(
    results_dir / "random_search_seed42_folds.csv",
    index=False,
)

with open(
    results_dir / "random_search_seed42_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(random_seed42["summary"], file, indent=2)

print("\nRandom Search — Seed 42 summary")
display(pd.DataFrame([random_seed42["summary"]]))

print("\nBest trials:")
display(
    random_seed42["trials"]
    .sort_values("mean_validation_dice", ascending=False)
    [
        [
            "trial",
            "learning_rate",
            "weight_decay",
            "base_channels",
            "loss_bce_weight",
            "mean_validation_dice",
            "best_so_far_mean_validation_dice",
            "cumulative_wall_clock_seconds",
        ]
    ]
    .head(5)
    .round(6)
)

print("\nArtifacts saved to:", results_dir)


# Cell 74: Full TPE Experiment — Seed 42

In [ ]:

# Cell 74: Full Optuna TPE — Seed 42

import json
from pathlib import Path
import pandas as pd

tpe_seed42 = run_tpe_comparison(
    trials_per_search=16,
    seed=42,
    folds=[3, 4],
    epochs=3,
)

# Reuse the results directory established in Cell 73.
results_dir = project_root / "outputs" / "cross_validation"
results_dir.mkdir(parents=True, exist_ok=True)

tpe_seed42["trials"].to_csv(
    results_dir / "tpe_seed42_trials.csv",
    index=False,
)
tpe_seed42["folds"].to_csv(
    results_dir / "tpe_seed42_folds.csv",
    index=False,
)

with open(
    results_dir / "tpe_seed42_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(tpe_seed42["summary"], file, indent=2)

print("\nOptuna TPE — Seed 42 summary")
display(pd.DataFrame([tpe_seed42["summary"]]))

print("\nBest trials:")
display(
    tpe_seed42["trials"]
    .sort_values("mean_validation_dice", ascending=False)
    [
        [
            "trial",
            "learning_rate",
            "weight_decay",
            "base_channels",
            "loss_bce_weight",
            "mean_validation_dice",
            "best_so_far_mean_validation_dice",
            "cumulative_wall_clock_seconds",
        ]
    ]
    .head(5)
    .round(6)
)

print("\nArtifacts saved to:", results_dir)


# Cell 75: Full AS-HMM-BO Experiment — Seed 42

In [ ]:

# Cell 75: Full AS-HMM-BO — Seed 42

import json
from pathlib import Path
import pandas as pd

hmm_seed42 = run_as_hmm_bo_comparison(
    trials_per_search=16,
    seed=42,
    folds=[3, 4],
    epochs=3,
)

results_dir = project_root / "outputs" / "cross_validation"
results_dir.mkdir(parents=True, exist_ok=True)

hmm_seed42["trials"].to_csv(
    results_dir / "as_hmm_bo_seed42_trials.csv",
    index=False,
)
hmm_seed42["folds"].to_csv(
    results_dir / "as_hmm_bo_seed42_folds.csv",
    index=False,
)
hmm_seed42["states"].to_csv(
    results_dir / "as_hmm_bo_seed42_states.csv",
    index=False,
)

with open(
    results_dir / "as_hmm_bo_seed42_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(hmm_seed42["summary"], file, indent=2)

print("\nAS-HMM-BO — Seed 42 summary")
display(pd.DataFrame([hmm_seed42["summary"]]))

print("\nBest trials:")
display(
    hmm_seed42["trials"]
    .sort_values("mean_validation_dice", ascending=False)
    [
        [
            "trial",
            "selected_state",
            "learning_rate",
            "weight_decay",
            "base_channels",
            "loss_bce_weight",
            "mean_validation_dice",
            "best_so_far_mean_validation_dice",
            "cumulative_wall_clock_seconds",
        ]
    ]
    .head(5)
    .round(6)
)

print("\nHMM state counts:")
display(
    hmm_seed42["states"]["selected_state"]
    .value_counts()
    .rename_axis("state")
    .reset_index(name="trial_count")
)

print("\nArtifacts saved to:", results_dir)


# Cell 76 — Aggregate the controlled optimizer comparison

In [ ]:

# Cell 76: Aggregate the controlled optimizer comparison

import json
import numpy as np
import pandas as pd
from pathlib import Path

results_dir = project_root / "outputs" / "cross_validation"
results_dir.mkdir(parents=True, exist_ok=True)

# Use only the new controlled runs, not historical best-epoch results.
optimizer_runs = {
    "random_search": random_seed42,
    "optuna_tpe": tpe_seed42,
    "as_hmm_bo": hmm_seed42,
}

comparison_rows = []
curve_frames = []

for optimizer_name, run in optimizer_runs.items():
    trials = run["trials"].copy()
    trials = trials.sort_values("trial").reset_index(drop=True)

    required_columns = {
        "trial",
        "mean_validation_dice",
        "best_so_far_mean_validation_dice",
        "cumulative_wall_clock_seconds",
    }
    missing = required_columns.difference(trials.columns)

    if missing:
        raise ValueError(
            f"{optimizer_name} is missing required columns: {sorted(missing)}"
        )

    trial_count = len(trials)
    if trial_count < 2:
        raise ValueError(
            f"{optimizer_name} needs at least two trials for AUC comparison."
        )

    trial_x = trials["trial"].to_numpy(dtype=float)
    best_dice = trials["best_so_far_mean_validation_dice"].to_numpy(dtype=float)
    elapsed_x = trials["cumulative_wall_clock_seconds"].to_numpy(dtype=float)

    # Normalize by the span of the budget axis, so the AUC is interpretable
    # as average best-so-far Dice across the observed search budget.
    trial_auc = np.trapezoid(best_dice, trial_x) / (
        trial_x[-1] - trial_x[0]
    )

    elapsed_span = elapsed_x[-1] - elapsed_x[0]
    time_auc = (
        np.trapezoid(best_dice, elapsed_x) / elapsed_span
        if elapsed_span > 0
        else np.nan
    )

    best_index = int(np.argmax(trials["mean_validation_dice"].to_numpy()))
    best_trial = trials.iloc[best_index]

    summary = run["summary"]

    comparison_rows.append({
        "optimizer": optimizer_name,
        "seed": 42,
        "trials_completed": trial_count,
        "best_mean_validation_dice": float(
            trials["mean_validation_dice"].max()
        ),
        "mean_trial_dice": float(
            trials["mean_validation_dice"].mean()
        ),
        "best_trial": int(best_trial["trial"]),
        "best_so_far_auc_per_trial": float(trial_auc),
        "best_so_far_auc_per_second": float(time_auc),
        "total_wall_clock_seconds": float(
            summary["total_wall_clock_seconds"]
        ),
        "test_sets_used": bool(summary.get("test_sets_used", False)),
    })

    curve_frames.append(
        trials.assign(optimizer=optimizer_name)[[
            "optimizer",
            "trial",
            "mean_validation_dice",
            "best_so_far_mean_validation_dice",
            "cumulative_wall_clock_seconds",
        ]]
    )

comparison_df = pd.DataFrame(comparison_rows).sort_values(
    "optimizer"
).reset_index(drop=True)

curves_df = pd.concat(curve_frames, ignore_index=True)

comparison_path = results_dir / "controlled_optimizer_comparison_seed42.csv"
curves_path = results_dir / "controlled_optimizer_curves_seed42.csv"

comparison_df.to_csv(comparison_path, index=False)
curves_df.to_csv(curves_path, index=False)

print("CONTROLLED OPTIMIZER COMPARISON — SEED 42")
display(comparison_df.round(6))

print("\nBEST-SO-FAR SCORES AT SELECTED TRIAL BUDGETS")
selected_trials = [1, 4, 8, 12, 16]

budget_table = (
    curves_df[curves_df["trial"].isin(selected_trials)]
    .pivot(
        index="trial",
        columns="optimizer",
        values="best_so_far_mean_validation_dice",
    )
    .sort_index()
)

display(budget_table.round(6))

print("\nSaved comparison artifacts:")
print(comparison_path)
print(curves_path)

assert not comparison_df["test_sets_used"].any(), (
    "Unexpected: a comparison run reports using independent test sets."
)
print("\nValidation passed: independent test sets were not used.")


# Cell 77 — Plot the search trajectories

In [ ]:
# Cell 77: Visualize controlled optimizer search trajectories

import matplotlib.pyplot as plt

optimizer_order = [
    "random_search",
    "optuna_tpe",
    "as_hmm_bo",
]

display_names = {
    "random_search": "Random Search",
    "optuna_tpe": "Optuna TPE",
    "as_hmm_bo": "AS-HMM-BO",
}

# Chart 1: Best-so-far Dice versus number of trials
plt.figure(figsize=(10, 6))

for optimizer in optimizer_order:
    data = (
        curves_df[curves_df["optimizer"] == optimizer]
        .sort_values("trial")
    )

    plt.plot(
        data["trial"],
        data["best_so_far_mean_validation_dice"],
        marker="o",
        markersize=4,
        label=display_names[optimizer],
    )

plt.xlabel("Completed trials")
plt.ylabel("Best-so-far mean validation Dice")
plt.title("Optimizer Comparison: Best Validation Dice vs. Trial Budget")
plt.xticks(range(1, 17))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# Chart 2: Best-so-far Dice versus elapsed time
plt.figure(figsize=(10, 6))

for optimizer in optimizer_order:
    data = (
        curves_df[curves_df["optimizer"] == optimizer]
        .sort_values("cumulative_wall_clock_seconds")
    )

    # Include the first recorded point for each optimizer.
    plt.plot(
        data["cumulative_wall_clock_seconds"] / 60.0,
        data["best_so_far_mean_validation_dice"],
        marker="o",
        markersize=4,
        label=display_names[optimizer],
    )

plt.xlabel("Elapsed time (minutes)")
plt.ylabel("Best-so-far mean validation Dice")
plt.title("Optimizer Comparison: Best Validation Dice vs. Elapsed Time")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Cell 78 — Repeat the comparison with seed 123

In [ ]:
# Cell 78: Matched-seed comparison — seed 123

import json
import pandas as pd

results_dir = project_root / "outputs" / "cross_validation"
results_dir.mkdir(parents=True, exist_ok=True)

REPEAT_SEED = 123
REPEAT_TRIALS = 16
REPEAT_FOLDS = [3, 4]
REPEAT_EPOCHS = 3


def save_optimizer_run(run, optimizer_name, seed):
    """Save all available results for a completed optimizer run."""

    prefix = f"{optimizer_name}_seed{seed}"

    run["trials"].to_csv(
        results_dir / f"{prefix}_trials.csv",
        index=False,
    )

    run["folds"].to_csv(
        results_dir / f"{prefix}_folds.csv",
        index=False,
    )

    if "states" in run:
        run["states"].to_csv(
            results_dir / f"{prefix}_states.csv",
            index=False,
        )

    with open(
        results_dir / f"{prefix}_summary.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(run["summary"], file, indent=2)

    print(f"Saved {optimizer_name} results to: {results_dir}")


# 1. Random Search
random_seed123 = run_random_search_comparison(
    trials_per_search=REPEAT_TRIALS,
    seed=REPEAT_SEED,
    folds=REPEAT_FOLDS,
    epochs=REPEAT_EPOCHS,
)

save_optimizer_run(random_seed123, "random_search", REPEAT_SEED)


# 2. Optuna TPE
tpe_seed123 = run_tpe_comparison(
    trials_per_search=REPEAT_TRIALS,
    seed=REPEAT_SEED,
    folds=REPEAT_FOLDS,
    epochs=REPEAT_EPOCHS,
)

save_optimizer_run(tpe_seed123, "tpe", REPEAT_SEED)


# 3. AS-HMM-BO
hmm_seed123 = run_as_hmm_bo_comparison(
    trials_per_search=REPEAT_TRIALS,
    seed=REPEAT_SEED,
    folds=REPEAT_FOLDS,
    epochs=REPEAT_EPOCHS,
)

save_optimizer_run(hmm_seed123, "as_hmm_bo", REPEAT_SEED)


# Display the results from seed 123.
seed123_summary = pd.DataFrame([
    random_seed123["summary"],
    tpe_seed123["summary"],
    hmm_seed123["summary"],
])

print("\nCONTROLLED OPTIMIZER COMPARISON — SEED 123")
display(seed123_summary)

print("\nIndependent test sets remain excluded from optimizer search.")
assert not seed123_summary["test_sets_used"].astype(bool).any()

# Cell 79: Matched-seed comparison — seed 2026

In [ ]:
# Cell 79: Matched-seed comparison — seed 2026

import json
import pandas as pd

results_dir = project_root / "outputs" / "cross_validation"
results_dir.mkdir(parents=True, exist_ok=True)

REPEAT_SEED = 2026
REPEAT_TRIALS = 16
REPEAT_FOLDS = [3, 4]
REPEAT_EPOCHS = 3


def save_seed2026_run(run, optimizer_name):
    """Persist each completed optimizer run."""

    prefix = f"{optimizer_name}_seed{REPEAT_SEED}"

    run["trials"].to_csv(
        results_dir / f"{prefix}_trials.csv",
        index=False,
    )

    run["folds"].to_csv(
        results_dir / f"{prefix}_folds.csv",
        index=False,
    )

    if "states" in run:
        run["states"].to_csv(
            results_dir / f"{prefix}_states.csv",
            index=False,
        )

    with open(
        results_dir / f"{prefix}_summary.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(run["summary"], file, indent=2)

    print(f"Saved: {prefix}")


# 1. Random Search
random_seed2026 = run_random_search_comparison(
    trials_per_search=REPEAT_TRIALS,
    seed=REPEAT_SEED,
    folds=REPEAT_FOLDS,
    epochs=REPEAT_EPOCHS,
)
save_seed2026_run(random_seed2026, "random_search")


# 2. Optuna TPE
tpe_seed2026 = run_tpe_comparison(
    trials_per_search=REPEAT_TRIALS,
    seed=REPEAT_SEED,
    folds=REPEAT_FOLDS,
    epochs=REPEAT_EPOCHS,
)
save_seed2026_run(tpe_seed2026, "tpe")


# 3. AS-HMM-BO
hmm_seed2026 = run_as_hmm_bo_comparison(
    trials_per_search=REPEAT_TRIALS,
    seed=REPEAT_SEED,
    folds=REPEAT_FOLDS,
    epochs=REPEAT_EPOCHS,
)
save_seed2026_run(hmm_seed2026, "as_hmm_bo")


# Display the third-seed results.
seed2026_summary = pd.DataFrame([
    random_seed2026["summary"],
    tpe_seed2026["summary"],
    hmm_seed2026["summary"],
])

print("\nCONTROLLED OPTIMIZER COMPARISON — SEED 2026")
display(seed2026_summary)

assert len(seed2026_summary) == 3
assert seed2026_summary["trials_completed"].eq(16).all()
assert not seed2026_summary["test_sets_used"].astype(bool).any()

print("\nValidation passed: all three runs completed 16 trials.")
print("Independent test sets were not used.")

# Cell 80 — Aggregate results across all three seeds

In [ ]:
# Cell 80: Aggregate the controlled comparison across three seeds

import numpy as np
import pandas as pd
import json
from pathlib import Path

results_dir = project_root / "outputs" / "cross_validation"
results_dir.mkdir(parents=True, exist_ok=True)

# Use the completed runs already present in memory.
all_runs = {
    ("random_search", 42): random_seed42,
    ("optuna_tpe", 42): tpe_seed42,
    ("as_hmm_bo", 42): hmm_seed42,

    ("random_search", 123): random_seed123,
    ("optuna_tpe", 123): tpe_seed123,
    ("as_hmm_bo", 123): hmm_seed123,

    ("random_search", 2026): random_seed2026,
    ("optuna_tpe", 2026): tpe_seed2026,
    ("as_hmm_bo", 2026): hmm_seed2026,
}

seed_rows = []
curve_frames = []

for (optimizer, seed), run in all_runs.items():
    trials = run["trials"].copy().sort_values("trial").reset_index(drop=True)

    required_columns = {
        "trial",
        "mean_validation_dice",
        "best_so_far_mean_validation_dice",
        "cumulative_wall_clock_seconds",
    }
    missing = required_columns - set(trials.columns)

    if missing:
        raise ValueError(
            f"{optimizer}, seed {seed}: missing columns {sorted(missing)}"
        )

    if len(trials) != 16:
        raise ValueError(
            f"{optimizer}, seed {seed}: expected 16 trials, got {len(trials)}"
        )

    if trials["mean_validation_dice"].isna().any():
        raise ValueError(f"Missing trial Dice values for {optimizer}, seed {seed}")

    trial_x = trials["trial"].to_numpy(dtype=float)
    best_curve = trials["best_so_far_mean_validation_dice"].to_numpy(dtype=float)
    time_x = trials["cumulative_wall_clock_seconds"].to_numpy(dtype=float)

    trial_auc = np.trapezoid(best_curve, trial_x) / (
        trial_x[-1] - trial_x[0]
    )

    time_span = time_x[-1] - time_x[0]
    time_auc = (
        np.trapezoid(best_curve, time_x) / time_span
        if time_span > 0
        else np.nan
    )

    best_idx = trials["mean_validation_dice"].idxmax()
    best_trial = trials.loc[best_idx]

    summary = run["summary"]

    seed_rows.append({
        "optimizer": optimizer,
        "seed": seed,
        "trials_completed": len(trials),
        "best_mean_validation_dice": float(
            trials["mean_validation_dice"].max()
        ),
        "mean_trial_dice": float(
            trials["mean_validation_dice"].mean()
        ),
        "best_trial": int(best_trial["trial"]),
        "best_so_far_auc_per_trial": float(trial_auc),
        "best_so_far_auc_per_second": float(time_auc),
        "total_wall_clock_seconds": float(
            summary["total_wall_clock_seconds"]
        ),
        "test_sets_used": bool(summary.get("test_sets_used", False)),
    })

    curve_frames.append(
        trials.assign(
            optimizer=optimizer,
            seed=seed,
        )[[
            "optimizer",
            "seed",
            "trial",
            "mean_validation_dice",
            "best_so_far_mean_validation_dice",
            "cumulative_wall_clock_seconds",
        ]]
    )

seed_results_df = pd.DataFrame(seed_rows)
all_curves_df = pd.concat(curve_frames, ignore_index=True)

# Aggregate means and sample standard deviations across seeds.
metric_columns = [
    "best_mean_validation_dice",
    "mean_trial_dice",
    "best_so_far_auc_per_trial",
    "best_so_far_auc_per_second",
    "total_wall_clock_seconds",
]

aggregate_df = (
    seed_results_df
    .groupby("optimizer")[metric_columns]
    .agg(["mean", "std"])
)

aggregate_df.columns = [
    f"{metric}_{stat}"
    for metric, stat in aggregate_df.columns
]
aggregate_df = aggregate_df.reset_index()

# Save the detailed results and aggregates.
seed_results_path = results_dir / "three_seed_optimizer_results.csv"
aggregate_path = results_dir / "three_seed_optimizer_aggregate.csv"
curves_path = results_dir / "three_seed_optimizer_curves.csv"

seed_results_df.to_csv(seed_results_path, index=False)
aggregate_df.to_csv(aggregate_path, index=False)
all_curves_df.to_csv(curves_path, index=False)

print("PER-SEED RESULTS")
display(
    seed_results_df.sort_values(["seed", "optimizer"]).round(6)
)

print("\nAGGREGATE RESULTS: MEAN AND STANDARD DEVIATION ACROSS SEEDS")
display(aggregate_df.round(6))

# Compare AS-HMM-BO against both baselines within each matched seed.
wide_best = seed_results_df.pivot(
    index="seed",
    columns="optimizer",
    values="best_mean_validation_dice",
)

wide_mean = seed_results_df.pivot(
    index="seed",
    columns="optimizer",
    values="mean_trial_dice",
)

paired_differences = pd.DataFrame({
    "seed": wide_best.index,
    "AS_HMM_BO_minus_Random_best_Dice":
        (wide_best["as_hmm_bo"] - wide_best["random_search"]).values,
    "AS_HMM_BO_minus_TPE_best_Dice":
        (wide_best["as_hmm_bo"] - wide_best["optuna_tpe"]).values,
    "AS_HMM_BO_minus_Random_mean_trial_Dice":
        (wide_mean["as_hmm_bo"] - wide_mean["random_search"]).values,
    "AS_HMM_BO_minus_TPE_mean_trial_Dice":
        (wide_mean["as_hmm_bo"] - wide_mean["optuna_tpe"]).values,
})

print("\nPAIRED DIFFERENCES: AS-HMM-BO MINUS EACH BASELINE")
display(paired_differences.round(6))

# Verify protocol consistency and test-set isolation.
assert len(seed_results_df) == 9
assert seed_results_df["trials_completed"].eq(16).all()
assert not seed_results_df["test_sets_used"].any()

print("\nSaved artifacts:")
print(seed_results_path)
print(aggregate_path)
print(curves_path)

print("\nValidation passed: nine runs, 16 trials each, no independent test-set use.")

# Cell 81 — Plot mean search progress across all three seeds

In [ ]:
# Cell 81: Cross-seed search trajectory analysis

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

optimizer_order = ["random_search", "optuna_tpe", "as_hmm_bo"]

display_names = {
    "random_search": "Random Search",
    "optuna_tpe": "Optuna TPE",
    "as_hmm_bo": "AS-HMM-BO",
}

# Verify the expected nine runs and common trial budget.
assert all_curves_df["seed"].nunique() == 3
assert all_curves_df["optimizer"].nunique() == 3

for optimizer in optimizer_order:
    per_seed_counts = (
        all_curves_df[all_curves_df["optimizer"] == optimizer]
        .groupby("seed")["trial"]
        .nunique()
    )
    assert len(per_seed_counts) == 3 and per_seed_counts.eq(16).all()


# Plot 1: Best-so-far validation Dice across seeds.
plt.figure(figsize=(10, 6))

for optimizer in optimizer_order:
    data = all_curves_df[all_curves_df["optimizer"] == optimizer]

    grouped = data.groupby("trial")[
        "best_so_far_mean_validation_dice"
    ].agg(["mean", "std"])

    x = grouped.index.to_numpy()
    mean = grouped["mean"].to_numpy()
    std = grouped["std"].fillna(0).to_numpy()

    plt.plot(x, mean, marker="o", markersize=4, label=display_names[optimizer])
    plt.fill_between(x, mean - std, mean + std, alpha=0.15)

plt.xlabel("Completed trials")
plt.ylabel("Best-so-far mean validation Dice")
plt.title("Mean Search Progress Across Three Seeds")
plt.xticks(range(1, 17))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# Plot 2: Per-trial validation Dice across seeds.
plt.figure(figsize=(10, 6))

for optimizer in optimizer_order:
    data = all_curves_df[all_curves_df["optimizer"] == optimizer]

    grouped = data.groupby("trial")["mean_validation_dice"].agg(["mean", "std"])

    x = grouped.index.to_numpy()
    mean = grouped["mean"].to_numpy()
    std = grouped["std"].fillna(0).to_numpy()

    plt.plot(x, mean, marker="o", markersize=4, label=display_names[optimizer])
    plt.fill_between(x, mean - std, mean + std, alpha=0.15)

plt.xlabel("Trial number")
plt.ylabel("Mean validation Dice for that trial")
plt.title("Candidate Quality Across Three Seeds")
plt.xticks(range(1, 17))
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()


# Save a numerical version of the plotted trajectories.
trajectory_summary = (
    all_curves_df
    .groupby(["optimizer", "trial"])
    .agg(
        mean_trial_dice=("mean_validation_dice", "mean"),
        std_trial_dice=("mean_validation_dice", "std"),
        mean_best_so_far_dice=(
            "best_so_far_mean_validation_dice", "mean"
        ),
        std_best_so_far_dice=(
            "best_so_far_mean_validation_dice", "std"
        ),
    )
    .reset_index()
)

trajectory_path = (
    project_root / "outputs" / "cross_validation"
    / "three_seed_trajectory_summary.csv"
)
trajectory_summary.to_csv(trajectory_path, index=False)

print("Saved cross-seed trajectory summary:")
print(trajectory_path)
display(trajectory_summary.head(12).round(6))

# Cell 82: Compare all optimizers at selected trial budgets

In [ ]:
# Cell 82: Compare all optimizers at selected trial budgets

trajectory_summary = pd.read_csv(
    project_root / "outputs" / "cross_validation"
    / "three_seed_trajectory_summary.csv"
)

selected_trials = [1, 4, 8, 12, 16]

for metric in [
    "mean_best_so_far_dice",
    "mean_trial_dice",
]:
    print(f"\n{metric} — mean across three seeds")

    table = trajectory_summary[
        trajectory_summary["trial"].isin(selected_trials)
    ].pivot(
        index="trial",
        columns="optimizer",
        values=metric,
    ).sort_index()

    display(table.round(4))

print("\nFinal-trial best-so-far variability:")
final_trial = trajectory_summary[
    trajectory_summary["trial"] == 16
][[
    "optimizer",
    "mean_best_so_far_dice",
    "std_best_so_far_dice",
    "mean_trial_dice",
    "std_trial_dice",
]]

display(final_trial.round(5))

# Cell 83: Inspect the existing optimizer interfaces

In [ ]:

# Cell 83: Inspect optimizer and evaluation interfaces
import inspect

keywords = (
    "optimizer",
    "search",
    "hmm",
    "evaluate_candidate",
    "evaluate_fixed_budget",
    "build_search_state",
    "propose_",
)

for name, obj in sorted(globals().items()):
    if not callable(obj) or not any(k in name.lower() for k in keywords):
        continue

    if inspect.isfunction(obj):
        try:
            signature = inspect.signature(obj)
        except (TypeError, ValueError):
            signature = "(signature unavailable)"

        print(f"\n{name}{signature}")

        try:
            source_lines = inspect.getsource(obj).splitlines()
            for line in source_lines[:5]:
                print(f"    {line}")
            if len(source_lines) > 5:
                print("    ...")
        except (OSError, TypeError):
            print("    [Source unavailable]")


# Cell 84: Inspect the full run structure before adding the baseline

In [ ]:

# Cell 84: Inspect the existing optimizer implementation details
import inspect

functions_to_inspect = [
    "run_as_hmm_bo_comparison",
    "propose_hmm_guided_params",
    "sample_random_search_params",
    "evaluate_candidate_across_folds",
    "save_optimizer_run",
]

for name in functions_to_inspect:
    fn = globals().get(name)

    print("\n" + "=" * 90)
    print(f"FUNCTION: {name}")

    if fn is None:
        print("Not found in the current notebook namespace.")
        continue

    try:
        print(inspect.getsource(fn))
    except (OSError, TypeError) as exc:
        print(f"Could not retrieve source: {exc}")


# Cell 85: Implement the rule-based adaptive baseline

In [ ]:

# Cell 85: Rule-based adaptive search baseline
import numpy as np
import pandas as pd
import time

def run_rule_based_adaptive_comparison(
    trials_per_search=None,
    seed=None,
    folds=None,
    epochs=None,
):
    """Adaptive search using explicit rules, without HMM state inference."""

    if trials_per_search is None:
        trials_per_search = COMPARISON_CONFIG["trials_per_optimizer"]
    if seed is None:
        seed = COMPARISON_CONFIG["seeds"][0]
    if folds is None:
        folds = COMPARISON_CONFIG["search_folds"]
    if epochs is None:
        epochs = COMPARISON_CONFIG["epochs_per_trial"]

    folds = list(folds)
    allowed_folds = set(COMPARISON_CONFIG["search_folds"])

    if not folds or not set(folds).issubset(allowed_folds):
        raise ValueError(
            f"Use only configured development folds: {sorted(allowed_folds)}"
        )
    if len(set(folds)) != len(folds):
        raise ValueError("Duplicate fold IDs are not allowed.")
    if trials_per_search < 1 or epochs < 1:
        raise ValueError("Trial and epoch budgets must be positive.")

    rng = np.random.default_rng(seed)
    completed_results = []
    trial_records = []
    fold_records = []
    policy_records = []

    best_so_far = float("-inf")
    cumulative_seconds = 0.0
    stagnation = 0

    space = COMPARISON_CONFIG["search_space"]

    def sample_local_candidate(best_row, narrow=False):
        """Perturb the best candidate, respecting configured bounds."""
        lr = space["learning_rate"]
        wd = space["weight_decay"]
        bce = space["loss_bce_weight"]

        lr_sigma = 0.18 if narrow else 0.30
        wd_sigma = 0.40 if narrow else 0.70
        bce_sigma = 0.06 if narrow else 0.10

        lr_log = rng.normal(np.log(best_row["learning_rate"]), lr_sigma)
        wd_log = rng.normal(np.log(best_row["weight_decay"]), wd_sigma)

        lr_log = _reflect_to_bounds(
            lr_log, np.log(lr["low"]), np.log(lr["high"])
        )
        wd_log = _reflect_to_bounds(
            wd_log, np.log(wd["low"]), np.log(wd["high"])
        )

        bce_value = _reflect_to_bounds(
            rng.normal(best_row["loss_bce_weight"], bce_sigma),
            bce["low"],
            bce["high"],
        )

        return {
            "learning_rate": float(np.exp(lr_log)),
            "weight_decay": float(np.exp(wd_log)),
            "base_channels": int(
                best_row["base_channels"]
                if rng.random() < 0.75
                else rng.choice(space["base_channels"])
            ),
            "loss_bce_weight": float(bce_value),
        }

    for trial_number in range(1, trials_per_search + 1):

        # Rule 1: globally explore for the first three trials.
        if trial_number <= 3 or not completed_results:
            policy = "initial_exploration"
            candidate = sample_random_search_params(rng)

        # Rule 2: after three consecutive non-improving trials,
        # restart global exploration.
        elif stagnation >= 3:
            policy = "stagnation_restart"
            candidate = sample_random_search_params(rng)

        # Rule 3: otherwise exploit locally, with a 15% global restart.
        elif rng.random() < 0.15:
            policy = "global_restart"
            candidate = sample_random_search_params(rng)

        else:
            policy = "local_exploitation"
            best_row = max(
                completed_results,
                key=lambda row: row["mean_validation_image_dice"],
            )
            candidate = sample_local_candidate(
                best_row,
                narrow=(stagnation == 0),
            )

        trial_result, current_fold_records = evaluate_candidate_across_folds(
            params=candidate,
            trial_number=trial_number,
            seed=seed,
            folds=folds,
            epochs=epochs,
        )

        score = float(trial_result["mean_validation_dice"])
        previous_best = best_so_far
        best_so_far = max(best_so_far, score)

        if score > previous_best + 1e-12:
            stagnation = 0
        else:
            stagnation += 1

        cumulative_seconds += trial_result["wall_clock_seconds"]

        completed_results.append({
            "trial": trial_number,
            **candidate,
            "mean_validation_image_dice": score,
            "mean_validation_loss": trial_result["mean_validation_loss"],
        })

        trial_records.append({
            "optimizer": "rule_based_adaptive",
            "seed": int(seed),
            "selected_state": policy,
            **trial_result,
            "best_so_far_mean_validation_dice": best_so_far,
            "cumulative_wall_clock_seconds": cumulative_seconds,
        })

        for record in current_fold_records:
            fold_records.append({
                "optimizer": "rule_based_adaptive",
                "selected_state": policy,
                **record,
            })

        policy_records.append({
            "trial": trial_number,
            "selected_state": policy,
            "stagnation_count": stagnation,
            "trial_score": score,
            "best_so_far_dice": best_so_far,
            "trial_duration_seconds": trial_result["wall_clock_seconds"],
            **candidate,
        })

        print(
            f"Rule-based {trial_number}/{trials_per_search} | "
            f"policy={policy:<20} | Dice={score:.4f} | "
            f"Best={best_so_far:.4f} | "
            f"Elapsed={cumulative_seconds / 60:.2f} min"
        )

    trial_df = pd.DataFrame(trial_records)
    fold_df = pd.DataFrame(fold_records)
    policy_df = pd.DataFrame(policy_records)

    assert len(trial_df) == trials_per_search
    assert len(fold_df) == trials_per_search * len(folds)
    assert set(fold_df["fold"]).issubset(allowed_folds)

    summary = {
        "optimizer": "rule_based_adaptive",
        "seed": int(seed),
        "trials_completed": int(len(trial_df)),
        "search_folds": folds,
        "epochs_per_trial_per_fold": int(epochs),
        "best_mean_validation_dice": float(
            trial_df["mean_validation_dice"].max()
        ),
        "mean_trial_dice": float(
            trial_df["mean_validation_dice"].mean()
        ),
        "total_wall_clock_seconds": float(
            trial_df["wall_clock_seconds"].sum()
        ),
        "policy_counts": {
            str(name): int(count)
            for name, count in policy_df["selected_state"].value_counts().items()
        },
        "test_sets_used": False,
    }

    run = {
        "trials": trial_df,
        "folds": fold_df,
        "states": policy_df,
        "summary": summary,
    }

    save_optimizer_run(run, "rule_based_adaptive", seed)
    return run


print("Rule-based adaptive optimizer defined.")
print("No training or evaluation has been started by this cell.")


# Cell 86: Run a single-seed smoke test

In [ ]:

# Cell 86: Smoke test — two trials only
rule_based_smoke_test = run_rule_based_adaptive_comparison(
    trials_per_search=2,
    seed=42,
    folds=[3, 4],
    epochs=3,
)

print("\nSmoke-test summary:")
display(pd.DataFrame([rule_based_smoke_test["summary"]]))

print("\nTrial records:")
display(rule_based_smoke_test["trials"][
    [
        "trial",
        "selected_state",
        "mean_validation_dice",
        "best_so_far_mean_validation_dice",
        "wall_clock_seconds",
    ]
])


# Cell 87: Run the full rule-based baseline across three seeds

In [ ]:

# Cell 87: Full rule-based adaptive baseline — three seeds
from pathlib import Path
import shutil
import pandas as pd

output_dir = Path(results_dir)
smoke_prefix = "rule_based_adaptive_seed42"
full_prefix = "rule_based_adaptive_smoke_seed42"

# Preserve smoke-test artifacts before the full seed-42 run overwrites them.
for suffix in ["trials.csv", "folds.csv", "states.csv", "summary.json"]:
    source = output_dir / f"{smoke_prefix}_{suffix}"
    destination = output_dir / f"{full_prefix}_{suffix}"

    if source.exists():
        shutil.copy2(source, destination)

seeds_for_ablation = [42, 123, 2026]
rule_based_runs = {}

for seed in seeds_for_ablation:
    print("\n" + "=" * 72)
    print(f"FULL RULE-BASED RUN | seed={seed}")
    print("=" * 72)

    run = run_rule_based_adaptive_comparison(
        trials_per_search=16,
        seed=seed,
        folds=[3, 4],
        epochs=3,
    )

    rule_based_runs[seed] = run

    assert len(run["trials"]) == 16
    assert len(run["folds"]) == 32
    assert run["summary"]["test_sets_used"] is False

    print("\nRun summary:")
    display(pd.DataFrame([run["summary"]]))

print("\nAll three rule-based runs completed.")
print("Smoke-test artifacts preserved with prefix:", full_prefix)


# Cell 88: Aggregate the four optimizers

In [ ]:

# Cell 88: Four-optimizer, three-seed comparison
from pathlib import Path
import numpy as np
import pandas as pd

cv_dir = Path(results_dir)

# Load the original trial-level comparison curves.
original_curves_path = cv_dir / "three_seed_optimizer_curves.csv"
if not original_curves_path.exists():
    raise FileNotFoundError(
        f"Original comparison curves not found: {original_curves_path}"
    )

original_curves = pd.read_csv(original_curves_path)

# Load the three rule-based runs from their saved trial records.
rule_frames = []
for seed in [42, 123, 2026]:
    path = cv_dir / f"rule_based_adaptive_seed{seed}_trials.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing rule-based results: {path}")

    frame = pd.read_csv(path)
    frame["optimizer"] = "rule_based_adaptive"
    frame["seed"] = seed
    rule_frames.append(frame)

rule_curves = pd.concat(rule_frames, ignore_index=True)

# Standardize optimizer labels and expected columns.
curves = pd.concat(
    [original_curves, rule_curves],
    ignore_index=True,
    sort=False,
)

required_columns = {
    "optimizer",
    "seed",
    "trial",
    "mean_validation_dice",
    "wall_clock_seconds",
}
missing = required_columns - set(curves.columns)
if missing:
    raise ValueError(
        f"Missing required columns: {sorted(missing)}. "
        f"Available columns: {list(curves.columns)}"
    )

curves["trial"] = pd.to_numeric(curves["trial"], errors="raise")
curves["seed"] = pd.to_numeric(curves["seed"], errors="raise")
curves["mean_validation_dice"] = pd.to_numeric(
    curves["mean_validation_dice"], errors="raise"
)
curves["wall_clock_seconds"] = pd.to_numeric(
    curves["wall_clock_seconds"], errors="raise"
)

curves = curves.sort_values(
    ["optimizer", "seed", "trial"]
).reset_index(drop=True)

run_rows = []

for (optimizer, seed), group in curves.groupby(["optimizer", "seed"]):
    group = group.sort_values("trial").copy()

    if len(group) != 16 or group["trial"].nunique() != 16:
        raise ValueError(
            f"{optimizer}, seed {seed}: expected 16 unique trials; "
            f"found {len(group)} rows and {group['trial'].nunique()} trials."
        )

    scores = group["mean_validation_dice"].to_numpy(dtype=float)
    durations = group["wall_clock_seconds"].to_numpy(dtype=float)
    best_curve = np.maximum.accumulate(scores)
    cumulative_seconds = np.cumsum(durations)

    # Normalize each AUC by its horizontal-axis range.
    trial_auc = float(
        np.trapezoid(best_curve, dx=1) / (len(best_curve) - 1)
    )
    time_auc = float(
        np.trapezoid(best_curve, x=cumulative_seconds)
        / cumulative_seconds[-1]
    )

    run_rows.append({
        "optimizer": optimizer,
        "seed": int(seed),
        "best_validation_dice": float(scores.max()),
        "mean_trial_dice": float(scores.mean()),
        "best_so_far_auc_per_trial": trial_auc,
        "best_so_far_auc_per_second": time_auc,
        "total_wall_clock_seconds": float(durations.sum()),
        "final_best_so_far_dice": float(best_curve[-1]),
        "test_sets_used": False,
    })

run_metrics = pd.DataFrame(run_rows)

aggregate_metrics = (
    run_metrics.groupby("optimizer")
    .agg(
        seeds=("seed", "nunique"),
        best_validation_dice_mean=("best_validation_dice", "mean"),
        best_validation_dice_std=("best_validation_dice", "std"),
        mean_trial_dice_mean=("mean_trial_dice", "mean"),
        mean_trial_dice_std=("mean_trial_dice", "std"),
        auc_per_trial_mean=("best_so_far_auc_per_trial", "mean"),
        auc_per_trial_std=("best_so_far_auc_per_trial", "std"),
        auc_per_second_mean=("best_so_far_auc_per_second", "mean"),
        auc_per_second_std=("best_so_far_auc_per_second", "std"),
        runtime_seconds_mean=("total_wall_clock_seconds", "mean"),
        runtime_seconds_std=("total_wall_clock_seconds", "std"),
    )
    .reset_index()
)

# Persist as new files; original three-optimizer results remain untouched.
run_metrics.to_csv(
    cv_dir / "four_optimizer_run_metrics.csv", index=False
)
aggregate_metrics.to_csv(
    cv_dir / "four_optimizer_aggregate.csv", index=False
)

print("Per-seed results:")
display(run_metrics.round(5))

print("\nAggregate results: mean ± sample standard deviation across seeds")
display(aggregate_metrics.round(5))

print("\nSaved:")
print(cv_dir / "four_optimizer_run_metrics.csv")
print(cv_dir / "four_optimizer_aggregate.csv")


# Cell 89: Recalculate all four optimizers from raw trial records

In [ ]:
from pathlib import Path

cv_dir = PROJECT_ROOT / "outputs" / "cross_validation"

print("Directory exists:", cv_dir.exists())
print("\nCSV files in cross_validation:")

if cv_dir.exists():
    for path in sorted(cv_dir.glob("*.csv")):
        print(path.name)
else:
    print("The expected directory does not exist.")

print("\nSearching project outputs for optimizer trial files:")
project_dir = PROJECT_ROOT
for path in sorted(project_dir.glob("outputs/**/*trials.csv")):
    print(path)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

cv_dir = PROJECT_ROOT / "outputs" / "cross_validation"

seeds = [42, 123, 2026]

# Display name -> saved filename prefix
optimizer_files = {
    "AS-HMM-BO": "as_hmm_bo",
    "Optuna TPE": "tpe",
    "Random Search": "random_search",
    "Rule-Based Adaptive": "rule_based_adaptive",
}

run_rows = []
curve_rows = []

for optimizer_name, file_prefix in optimizer_files.items():
    for seed in seeds:
        path = cv_dir / f"{file_prefix}_seed{seed}_trials.csv"

        if not path.exists():
            raise FileNotFoundError(f"Missing raw trial file: {path}")

        df = pd.read_csv(path).sort_values("trial").copy()

        required = {"trial", "mean_validation_dice", "wall_clock_seconds"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(
                f"{path.name} is missing columns: {sorted(missing)}. "
                f"Available columns: {df.columns.tolist()}"
            )

        if len(df) != 16:
            raise ValueError(
                f"{path.name} contains {len(df)} trials; expected 16."
            )

        scores = df["mean_validation_dice"].to_numpy(dtype=float)
        durations = df["wall_clock_seconds"].to_numpy(dtype=float)

        if not np.isfinite(scores).all():
            raise ValueError(f"Non-finite Dice value in {path.name}")
        if not np.isfinite(durations).all() or (durations <= 0).any():
            raise ValueError(f"Invalid duration in {path.name}")

        best_curve = np.maximum.accumulate(scores)
        cumulative_seconds = np.cumsum(durations)

        auc_trial = np.trapezoid(best_curve) / (len(best_curve) - 1)
        auc_time = (
            np.trapezoid(best_curve, x=cumulative_seconds)
            / cumulative_seconds[-1]
        )

        best_idx = int(np.argmax(scores))

        run_rows.append({
            "optimizer": optimizer_name,
            "seed": seed,
            "best_validation_dice": float(scores[best_idx]),
            "best_trial": int(df.iloc[best_idx]["trial"]),
            "mean_trial_dice": float(scores.mean()),
            "best_so_far_auc_per_trial": float(auc_trial),
            "best_so_far_auc_per_second": float(auc_time),
            "runtime_seconds": float(durations.sum()),
        })

        for i, (_, row) in enumerate(df.iterrows()):
            curve_rows.append({
                "optimizer": optimizer_name,
                "seed": seed,
                "trial": int(row["trial"]),
                "validation_dice": float(scores[i]),
                "best_so_far_dice": float(best_curve[i]),
                "trial_seconds": float(durations[i]),
                "cumulative_seconds": float(cumulative_seconds[i]),
            })

run_metrics = pd.DataFrame(run_rows)
curve_metrics = pd.DataFrame(curve_rows)

metric_columns = [
    "best_validation_dice",
    "mean_trial_dice",
    "best_so_far_auc_per_trial",
    "best_so_far_auc_per_second",
    "runtime_seconds",
]

aggregate = (
    run_metrics.groupby("optimizer")[metric_columns]
    .agg(["mean", "std"])
    .round(6)
)

run_metrics.to_csv(cv_dir / "four_optimizer_run_metrics.csv", index=False)
curve_metrics.to_csv(cv_dir / "four_optimizer_trial_metrics.csv", index=False)
aggregate.to_csv(cv_dir / "four_optimizer_aggregate.csv")

print("PER-SEED RESULTS")
display(run_metrics.round(6))

print("\nAGGREGATE RESULTS: mean and standard deviation across 3 seeds")
display(aggregate)

print("\nValidation checks")
print("Runs:", len(run_metrics), "| Expected:", len(optimizer_files) * len(seeds))
print("Curve rows:", len(curve_metrics), "| Expected:", len(optimizer_files) * len(seeds) * 16)
print("Missing values in run metrics:", int(run_metrics.isna().sum().sum()))
print("TestA/TestB used: No")

# Cell 97 — Paired optimizer comparison

In [ ]:
import numpy as np
import pandas as pd

# Reuse run_metrics created in Cell 96.
primary_metrics = [
    "best_validation_dice",
    "mean_trial_dice",
    "best_so_far_auc_per_trial",
    "best_so_far_auc_per_second",
    "runtime_seconds",
]

reference = (
    run_metrics[run_metrics["optimizer"] == "AS-HMM-BO"]
    .set_index("seed")
)

comparison_rows = []

for baseline in [
    "Optuna TPE",
    "Random Search",
    "Rule-Based Adaptive",
]:
    other = run_metrics[run_metrics["optimizer"] == baseline].set_index("seed")

    for metric in primary_metrics:
        # Positive means AS-HMM-BO is higher.
        # For runtime, negative means AS-HMM-BO is faster.
        differences = reference[metric] - other[metric]

        comparison_rows.append({
            "baseline": baseline,
            "metric": metric,
            "AS_minus_baseline_mean": differences.mean(),
            "AS_minus_baseline_sd": differences.std(ddof=1),
            "seeds_AS_better": int((differences > 0).sum()),
            "seeds_tied": int((differences == 0).sum()),
            "per_seed_differences": differences.round(6).to_dict(),
        })

paired_comparison = pd.DataFrame(comparison_rows)

paired_comparison.to_csv(
    cv_dir / "as_hmm_bo_paired_comparison.csv",
    index=False,
)

display(paired_comparison.round(6))

# Cell 98 — Inspect the existing HMM controller

In [ ]:
# Cell 98 — Inspect HMM controller and proposal functions

import inspect

functions_to_inspect = [
    "infer_corrected_hmm_state",
    "propose_hmm_guided_params",
    "build_search_state_observations",
]

for function_name in functions_to_inspect:
    function_obj = globals().get(function_name)

    print("\n" + "=" * 90)
    print(f"FUNCTION: {function_name}")
    print("=" * 90)

    if function_obj is None:
        print("Function not found in the current notebook namespace.")
    else:
        print(inspect.getsource(function_obj))

# Cell 99 — Configure the HMM ablation

In [ ]:
# Cell 99 — State-independent adaptive ablation configuration

ABLATION_CONFIG = {
    "name": "state_independent_adaptive",
    "initial_global_trials": 3,
    "global_restart_probability": 0.15,
    "learning_rate_log_sigma": 0.225,
    "weight_decay_log_sigma": 0.55,
    "loss_bce_sigma": 0.08,
    "keep_best_base_channels_probability": 0.75,
    "seeds": [42, 123, 2026],
    "folds": [3, 4],
    "trials_per_seed": 16,
    "epochs_per_fold": 3,
}

# Confirm the ablation uses the same search bounds as the main study.
print("Search space:")
for key, value in SEARCH_CONFIG.items():
    print(f"  {key}: {value}")

print("\nAblation budget:")
for key, value in ABLATION_CONFIG.items():
    print(f"  {key}: {value}")

assert ABLATION_CONFIG["trials_per_seed"] == 16
assert ABLATION_CONFIG["folds"] == [3, 4]
assert ABLATION_CONFIG["seeds"] == [42, 123, 2026]

print("\nConfiguration validated. No training has been started.")

# Cell 100 — Implement the state-independent adaptive sampler

In [ ]:
# Cell 100 — State-independent adaptive candidate sampler

def propose_state_independent_adaptive_params(completed_results, rng):
    """
    Adaptive sampling without HMM state inference.

    Uses:
      - Global sampling for the first configured trials.
      - A fixed global-restart probability afterward.
      - Fixed-width perturbations around the best completed candidate.
      - The same search bounds and parameter types as the HMM sampler.
    """
    cfg = ABLATION_CONFIG

    lr_cfg = SEARCH_CONFIG["learning_rate"]
    wd_cfg = SEARCH_CONFIG["weight_decay"]
    bce_cfg = SEARCH_CONFIG["loss_bce_weight"]

    def global_sample():
        return {
            "learning_rate": float(np.exp(rng.uniform(
                np.log(lr_cfg["low"]), np.log(lr_cfg["high"])
            ))),
            "weight_decay": float(np.exp(rng.uniform(
                np.log(wd_cfg["low"]), np.log(wd_cfg["high"])
            ))),
            "base_channels": int(rng.choice(
                SEARCH_CONFIG["base_channels"]
            )),
            "loss_bce_weight": float(rng.uniform(
                bce_cfg["low"], bce_cfg["high"]
            )),
        }

    # Initial global exploration.
    if len(completed_results) < cfg["initial_global_trials"]:
        return global_sample()

    best_row = max(
        completed_results,
        key=lambda row: row["mean_validation_image_dice"],
    )

    # Fixed restart probability, independent of HMM state.
    if rng.random() < cfg["global_restart_probability"]:
        return global_sample()

    lr_log = rng.normal(
        np.log(best_row["learning_rate"]),
        cfg["learning_rate_log_sigma"],
    )
    wd_log = rng.normal(
        np.log(best_row["weight_decay"]),
        cfg["weight_decay_log_sigma"],
    )

    learning_rate = np.exp(_reflect_to_bounds(
        lr_log,
        np.log(lr_cfg["low"]),
        np.log(lr_cfg["high"]),
    ))
    weight_decay = np.exp(_reflect_to_bounds(
        wd_log,
        np.log(wd_cfg["low"]),
        np.log(wd_cfg["high"]),
    ))

    base_channels = (
        int(best_row["base_channels"])
        if rng.random() < cfg["keep_best_base_channels_probability"]
        else int(rng.choice(SEARCH_CONFIG["base_channels"]))
    )

    bce_weight = _reflect_to_bounds(
        rng.normal(
            best_row["loss_bce_weight"],
            cfg["loss_bce_sigma"],
        ),
        bce_cfg["low"],
        bce_cfg["high"],
    )

    return {
        "learning_rate": float(learning_rate),
        "weight_decay": float(weight_decay),
        "base_channels": base_channels,
        "loss_bce_weight": float(bce_weight),
    }


print("State-independent adaptive sampler defined.")

# Cell 101 — Candidate-generation smoke test

In [ ]:
# Cell 101 — Validate state-independent candidate generation

import numpy as np

rng = np.random.default_rng(42)
completed_results = []

def validate_candidate(params):
    assert SEARCH_CONFIG["learning_rate"]["low"] <= params["learning_rate"] <= SEARCH_CONFIG["learning_rate"]["high"]
    assert SEARCH_CONFIG["weight_decay"]["low"] <= params["weight_decay"] <= SEARCH_CONFIG["weight_decay"]["high"]
    assert params["base_channels"] in SEARCH_CONFIG["base_channels"]
    assert SEARCH_CONFIG["loss_bce_weight"]["low"] <= params["loss_bce_weight"] <= SEARCH_CONFIG["loss_bce_weight"]["high"]

# Three initial global candidates.
for trial in range(3):
    candidate = propose_state_independent_adaptive_params(
        completed_results, rng
    )
    validate_candidate(candidate)

    completed_results.append({
        **candidate,
        "trial": trial + 1,
        "mean_validation_image_dice": [0.61, 0.66, 0.64][trial],
    })

    print(f"Initial trial {trial + 1}: {candidate}")

# Candidate after initial exploration should use the adaptive branch
# unless the random restart is selected.
adaptive_candidate = propose_state_independent_adaptive_params(
    completed_results, rng
)
validate_candidate(adaptive_candidate)

print("\nAdaptive-stage candidate:", adaptive_candidate)
print("\nAll candidates satisfy the configured search bounds.")

# Cell 102 — Implement the ablation runner

In [ ]:
# Cell 102 — State-independent adaptive comparison runner

def run_state_independent_adaptive_comparison(
    trials_per_search=None,
    seed=None,
    folds=None,
    epochs=None,
):
    cfg = ABLATION_CONFIG

    trials_per_search = (
        cfg["trials_per_seed"] if trials_per_search is None
        else int(trials_per_search)
    )
    seed = 42 if seed is None else int(seed)
    folds = cfg["folds"] if folds is None else list(folds)
    epochs = cfg["epochs_per_fold"] if epochs is None else int(epochs)

    rng = np.random.default_rng(seed)
    completed_results = []
    trial_records = []
    fold_records = []

    for trial_number in range(1, trials_per_search + 1):
        params = propose_state_independent_adaptive_params(
            completed_results, rng
        )

        # Reuse the existing evaluation pipeline.
        trial_result, current_fold_records = (
            evaluate_candidate_across_folds(
                params=params,
                trial_number=trial_number,
                seed=seed,
                folds=folds,
                epochs=epochs,
            )
        )

        score = float(trial_result["mean_validation_dice"])

        completed_results.append({
            **params,
            "trial": trial_number,
            "mean_validation_image_dice": score,
        })

        trial_records.append({
            **trial_result,
            **params,
            "optimizer": "state_independent_adaptive",
            "seed": seed,
            "trial": trial_number,
            "mean_validation_dice": score,
        })

        for fold_record in current_fold_records:
            fold_records.append({
                **fold_record,
                "optimizer": "state_independent_adaptive",
                "seed": seed,
                "trial": trial_number,
            })

        best_so_far = max(
            row["mean_validation_image_dice"]
            for row in completed_results
        )

        print(
            f"Seed {seed} | Trial {trial_number:02d}/{trials_per_search} "
            f"| Dice {score:.6f} | Best {best_so_far:.6f}"
        )

    trials_df = pd.DataFrame(trial_records).sort_values("trial")
    folds_df = pd.DataFrame(fold_records)

    summary = {
        "optimizer": "state_independent_adaptive",
        "seed": seed,
        "trials": trials_per_search,
        "folds": folds,
        "epochs_per_fold": epochs,
        "best_validation_dice": float(
            trials_df["mean_validation_dice"].max()
        ),
        "mean_trial_dice": float(
            trials_df["mean_validation_dice"].mean()
        ),
        "runtime_seconds": float(
            trials_df["wall_clock_seconds"].sum()
        ),
    }

    run = {
        "trials": trials_df,
        "folds": folds_df,
        "states": pd.DataFrame(),
        "summary": summary,
    }

    save_optimizer_run(
        run,
        optimizer_name="state_independent_adaptive",
        seed=seed,
    )

    print("\nRun summary:")
    display(pd.DataFrame([summary]))

    return run


print("Ablation runner defined. No training has been started.")

# Cell 103 — Two-trial smoke test

In [ ]:
# Cell 103 — Smoke test the ablation evaluation pipeline

ablation_smoke = run_state_independent_adaptive_comparison(
    trials_per_search=2,
    seed=9876,          # Separate smoke-test seed
    folds=[3, 4],
    epochs=3,
)

print("\nSmoke-test validation:")

assert len(ablation_smoke["trials"]) == 2
assert len(ablation_smoke["folds"]) == 4
assert ablation_smoke["trials"]["mean_validation_dice"].notna().all()
assert (
    ablation_smoke["trials"]["wall_clock_seconds"] > 0
).all()

print("PASS: two trials completed with finite Dice scores and valid runtimes.")
print("PASS: four fold evaluations recorded.")
print("PASS: smoke-test results saved separately from the main seeds.")

# Cell 104 — Run the full ablation experiment

In [ ]:
# Cell 104 — Full state-independent adaptive ablation

ABLATION_RUNS = {}

for seed in ABLATION_CONFIG["seeds"]:
    print("\n" + "=" * 75)
    print(f"STATE-INDEPENDENT ADAPTIVE ABLATION — SEED {seed}")
    print("=" * 75)

    run = run_state_independent_adaptive_comparison(
        trials_per_search=ABLATION_CONFIG["trials_per_seed"],
        seed=seed,
        folds=ABLATION_CONFIG["folds"],
        epochs=ABLATION_CONFIG["epochs_per_fold"],
    )

    ABLATION_RUNS[seed] = run

    print(f"\nSeed {seed} completed and saved.")
    print(run["summary"])

print("\nAll three ablation runs completed.")

# 
# 

# Cell 105 — Compare all five optimizers

In [ ]:
# Cell 105 — Five-optimizer comparison including HMM ablation

import numpy as np
import pandas as pd
from pathlib import Path

cv_dir = PROJECT_ROOT / "outputs" / "cross_validation"

seeds = [42, 123, 2026]

optimizer_files = {
    "AS-HMM-BO": "as_hmm_bo",
    "Optuna TPE": "tpe",
    "Random Search": "random_search",
    "Rule-Based Adaptive": "rule_based_adaptive",
    "State-Independent Adaptive": "state_independent_adaptive",
}

run_rows = []
curve_rows = []

for optimizer, prefix in optimizer_files.items():
    for seed in seeds:
        path = cv_dir / f"{prefix}_seed{seed}_trials.csv"

        if not path.exists():
            raise FileNotFoundError(f"Missing result file: {path}")

        df = pd.read_csv(path).sort_values("trial")

        required = {
            "trial",
            "mean_validation_dice",
            "wall_clock_seconds",
        }
        missing = required - set(df.columns)
        if missing:
            raise ValueError(
                f"{path.name} missing columns: {sorted(missing)}"
            )

        if len(df) != 16:
            raise ValueError(
                f"{path.name} has {len(df)} trials; expected 16."
            )

        scores = df["mean_validation_dice"].to_numpy(float)
        durations = df["wall_clock_seconds"].to_numpy(float)

        if not np.isfinite(scores).all():
            raise ValueError(f"Invalid Dice values in {path.name}")
        if not np.isfinite(durations).all() or (durations <= 0).any():
            raise ValueError(f"Invalid runtimes in {path.name}")

        best_curve = np.maximum.accumulate(scores)
        cumulative_seconds = np.cumsum(durations)
        best_idx = int(np.argmax(scores))

        run_rows.append({
            "optimizer": optimizer,
            "seed": seed,
            "best_validation_dice": scores[best_idx],
            "best_trial": int(df.iloc[best_idx]["trial"]),
            "mean_trial_dice": scores.mean(),
            "best_so_far_auc_per_trial": (
                np.trapezoid(best_curve) / (len(scores) - 1)
            ),
            "best_so_far_auc_per_second": (
                np.trapezoid(best_curve, x=cumulative_seconds)
                / cumulative_seconds[-1]
            ),
            "runtime_seconds": durations.sum(),
        })

        for i, (_, row) in enumerate(df.iterrows()):
            curve_rows.append({
                "optimizer": optimizer,
                "seed": seed,
                "trial": int(row["trial"]),
                "validation_dice": scores[i],
                "best_so_far_dice": best_curve[i],
                "trial_seconds": durations[i],
                "cumulative_seconds": cumulative_seconds[i],
            })

run_metrics_5 = pd.DataFrame(run_rows)
curves_5 = pd.DataFrame(curve_rows)

metric_columns = [
    "best_validation_dice",
    "mean_trial_dice",
    "best_so_far_auc_per_trial",
    "best_so_far_auc_per_second",
    "runtime_seconds",
]

aggregate_5 = (
    run_metrics_5.groupby("optimizer")[metric_columns]
    .agg(["mean", "std"])
    .round(6)
)

run_metrics_5.to_csv(
    cv_dir / "five_optimizer_run_metrics.csv", index=False
)
curves_5.to_csv(
    cv_dir / "five_optimizer_trial_metrics.csv", index=False
)
aggregate_5.to_csv(
    cv_dir / "five_optimizer_aggregate.csv"
)

print("FIVE-OPTIMIZER AGGREGATE RESULTS")
display(aggregate_5)

print("\nPER-SEED RESULTS")
display(run_metrics_5.round(6))

print("\nValidation checks")
print("Runs:", len(run_metrics_5), "| Expected: 15")
print("Curve rows:", len(curves_5), "| Expected: 240")
print("Missing values:", int(run_metrics_5.isna().sum().sum()))
print("TestA/TestB used: No")

# Cell 106 — Paired AS-HMM-BO vs state-independent ablation

In [ ]:
# Cell 106 — Paired AS-HMM-BO vs state-independent ablation

paired_metrics = [
    "best_validation_dice",
    "mean_trial_dice",
    "best_so_far_auc_per_trial",
    "best_so_far_auc_per_second",
    "runtime_seconds",
]

paired_rows = []

for metric in paired_metrics:
    pivot = run_metrics_5[
        run_metrics_5["optimizer"].isin([
            "AS-HMM-BO",
            "State-Independent Adaptive",
        ])
    ].pivot(index="seed", columns="optimizer", values=metric)

    diff = (
        pivot["AS-HMM-BO"]
        - pivot["State-Independent Adaptive"]
    )

    paired_rows.append({
        "metric": metric,
        "AS_HMM_BO_mean": pivot["AS-HMM-BO"].mean(),
        "state_independent_mean": (
            pivot["State-Independent Adaptive"].mean()
        ),
        "mean_paired_difference": diff.mean(),
        "paired_difference_sd": diff.std(ddof=1),
        "seeds_AS_HMM_BO_higher": int((diff > 0).sum()),
        "seed_differences": diff.round(6).to_dict(),
    })

paired_ablation = pd.DataFrame(paired_rows)
paired_ablation.to_csv(
    cv_dir / "paired_hmm_ablation_analysis.csv",
    index=False,
)

display(paired_ablation.round(6))

# Cell 107 — Diagnose state-independent adaptive runtime variation

In [ ]:
# Cell 107 — Diagnose state-independent adaptive runtime variation

import pandas as pd
import numpy as np

runtime_rows = []

for seed in [42, 123, 2026]:
    path = results_dir / (
        f"state_independent_adaptive_seed{seed}_trials.csv"
    )

    if not path.exists():
        print(f"Missing file: {path}")
        continue

    df = pd.read_csv(path)

    if "wall_clock_seconds" not in df.columns:
        print(f"Missing wall_clock_seconds in {path.name}")
        print("Available columns:", df.columns.tolist())
        continue

    durations = pd.to_numeric(
        df["wall_clock_seconds"], errors="coerce"
    ).dropna()

    runtime_rows.append({
        "seed": seed,
        "trial_rows": len(df),
        "valid_duration_rows": len(durations),
        "runtime_sum_seconds": durations.sum(),
        "runtime_median_seconds": durations.median(),
        "runtime_min_seconds": durations.min(),
        "runtime_max_seconds": durations.max(),
        "runtime_std_seconds": durations.std(),
        "invalid_or_missing_durations": int(
            df["wall_clock_seconds"].isna().sum()
        ),
    })

    print(f"\nSeed {seed}: per-trial runtimes")
    display(
        df[[
            c for c in [
                "trial",
                "mean_validation_dice",
                "wall_clock_seconds",
            ]
            if c in df.columns
        ]].sort_values("wall_clock_seconds")
    )

print("\nRuntime summary")
display(pd.DataFrame(runtime_rows).round(3))

print(
    "\nNote: trial durations may vary with model configuration, "
    "system load, and other execution effects."
)

# Cell 108 — Verify fold-level evaluation consistency

In [ ]:
# Cell 108 — Verify fold-level evaluation consistency

for seed in [42, 123, 2026]:
    path = results_dir / (
        f"state_independent_adaptive_seed{seed}_folds.csv"
    )

    print(f"\n{'=' * 20} Seed {seed} {'=' * 20}")

    if not path.exists():
        print(f"Missing fold file: {path}")
        continue

    folds_df = pd.read_csv(path)
    print("Rows:", len(folds_df))
    print("Columns:", folds_df.columns.tolist())

    # Summarize how many fold records were saved per trial.
    if "trial" in folds_df.columns:
        print("\nFold records per trial:")
        display(
            folds_df.groupby("trial")
            .size()
            .rename("fold_records")
            .to_frame()
        )

    # Display fields that may reveal incomplete or different work.
    relevant_columns = [
        c for c in folds_df.columns
        if any(term in c.lower() for term in [
            "trial", "fold", "epoch", "duration", "second",
            "time", "status", "error", "dice", "loss"
        ])
    ]

    print("\nRelevant fold-level records:")
    display(folds_df[relevant_columns])

    # Check for missing values in the displayed fields.
    if relevant_columns:
        print("\nMissing values:")
        display(folds_df[relevant_columns].isna().sum().to_frame("missing"))

# Cell 109 — Check model configurations and evaluation implementation

In [ ]:
# Cell 109 — Diagnose runtime differences by configuration and code path

import inspect
import pandas as pd

for seed in [42, 123, 2026]:
    path = results_dir / (
        f"state_independent_adaptive_seed{seed}_folds.csv"
    )
    df = pd.read_csv(path)

    print(f"\n{'=' * 18} Seed {seed} {'=' * 18}")

    if "base_channels" in df.columns:
        print("Base-channel counts:")
        display(df["base_channels"].value_counts(dropna=False).to_frame("fold_rows"))

        print("Duration by base_channels:")
        display(
            df.groupby("base_channels")["duration_seconds"]
            .agg(["count", "mean", "median", "min", "max"])
            .round(3)
        )

    print("Duration by fold:")
    display(
        df.groupby("fold")["duration_seconds"]
        .agg(["count", "mean", "median", "min", "max"])
        .round(3)
    )

# Inspect the exact evaluation functions used by the runner.
for function_name in [
    "evaluate_candidate_across_folds",
    "evaluate_fixed_budget_candidate",
]:
    function = globals().get(function_name)

    print(f"\n{'=' * 12} Source: {function_name} {'=' * 12}")

    if function is None:
        print("Function not found in the current notebook namespace.")
        continue

    try:
        print(inspect.getsource(function))
    except (OSError, TypeError) as exc:
        print(f"Could not retrieve source: {exc}")

# Cell 110 — Compare model-size selection and runtime across all optimizers

In [ ]:
# Cell 110 — Compare model-size selection and runtime across all optimizers

import pandas as pd

optimizer_prefixes = {
    "AS-HMM-BO": "as_hmm_bo",
    "Optuna TPE": "tpe",
    "Random Search": "random_search",
    "Rule-Based Adaptive": "rule_based_adaptive",
    "State-Independent Adaptive": "state_independent_adaptive",
}

audit_rows = []

for optimizer_name, prefix in optimizer_prefixes.items():
    for seed in [42, 123, 2026]:
        path = results_dir / f"{prefix}_seed{seed}_folds.csv"

        if not path.exists():
            print(f"File not found: {path.name}")
            continue

        df = pd.read_csv(path)

        if not {"base_channels", "duration_seconds"}.issubset(df.columns):
            print(f"Required columns missing in {path.name}")
            continue

        channel_counts = df["base_channels"].value_counts()

        for channels in sorted(df["base_channels"].dropna().unique()):
            subset = df[df["base_channels"] == channels]

            audit_rows.append({
                "optimizer": optimizer_name,
                "seed": seed,
                "base_channels": int(channels),
                "fold_evaluations": len(subset),
                "selection_percentage": (
                    100 * len(subset) / len(df)
                ),
                "mean_fold_seconds": subset["duration_seconds"].mean(),
                "median_fold_seconds": subset["duration_seconds"].median(),
            })

audit_df = pd.DataFrame(audit_rows)

print("Model-size selection and runtime by optimizer, seed, and configuration:")
display(
    audit_df.sort_values(["optimizer", "seed", "base_channels"])
    .round(3)
)

print("\nOverall configuration selection percentages:")
if not audit_df.empty:
    totals = (
        audit_df.groupby(["optimizer", "base_channels"])["fold_evaluations"]
        .sum()
        .rename("fold_evaluations")
        .reset_index()
    )
    totals["selection_percentage"] = (
        totals["fold_evaluations"]
        / totals.groupby("optimizer")["fold_evaluations"].transform("sum")
        * 100
    )
    display(totals.round(2))

# Cell 111 — Validation performance by model size

In [ ]:
# Cell 111 — Compare validation performance by selected model size

import pandas as pd

optimizer_prefixes = {
    "AS-HMM-BO": "as_hmm_bo",
    "Optuna TPE": "tpe",
    "Random Search": "random_search",
    "Rule-Based Adaptive": "rule_based_adaptive",
    "State-Independent Adaptive": "state_independent_adaptive",
}

trial_rows = []

for optimizer_name, prefix in optimizer_prefixes.items():
    for seed in [42, 123, 2026]:
        path = results_dir / f"{prefix}_seed{seed}_folds.csv"

        if not path.exists():
            print(f"Missing file: {path.name}")
            continue

        df = pd.read_csv(path)

        required = {
            "trial", "base_channels", "mean_image_dice",
            "duration_seconds", "fold"
        }
        missing = required - set(df.columns)

        if missing:
            print(f"Skipping {path.name}; missing columns: {missing}")
            continue

        # Each trial should have one configuration evaluated on both folds.
        grouped = df.groupby("trial").agg(
            base_channels=("base_channels", "first"),
            fold_count=("fold", "nunique"),
            trial_dice=("mean_image_dice", "mean"),
            fold_runtime_seconds=("duration_seconds", "sum"),
        ).reset_index()

        grouped["optimizer"] = optimizer_name
        grouped["seed"] = seed
        trial_rows.append(grouped)

trial_df = pd.concat(trial_rows, ignore_index=True)

print("Check that each trial used both development folds:")
display(trial_df.groupby(
    ["optimizer", "seed", "fold_count"]
).size().rename("trial_count").to_frame())

print("\nPerformance grouped by optimizer, seed, and model size:")
conditional_summary = (
    trial_df.groupby(["optimizer", "seed", "base_channels"])
    .agg(
        trials=("trial", "count"),
        mean_trial_dice=("trial_dice", "mean"),
        best_trial_dice=("trial_dice", "max"),
        mean_trial_runtime_seconds=("fold_runtime_seconds", "mean"),
    )
    .reset_index()
)

display(conditional_summary.round(4))

print("\nOverall descriptive summary by optimizer and model size:")
overall_summary = (
    trial_df.groupby(["optimizer", "base_channels"])
    .agg(
        trials=("trial", "count"),
        mean_trial_dice=("trial_dice", "mean"),
        best_trial_dice=("trial_dice", "max"),
        mean_trial_runtime_seconds=("fold_runtime_seconds", "mean"),
    )
    .reset_index()
)

display(overall_summary.round(4))

# Cell 112 — Audit optimizer functions before the fixed-size experiment

In [ ]:

# Cell 112 — Audit optimizer functions before fixed-size ablation

import inspect

functions_to_audit = [
    "sample_random_search_params",
    "propose_hmm_guided_params",
    "run_as_hmm_bo_comparison",
    "run_random_search_comparison",
    "run_rule_based_adaptive_comparison",
    "propose_state_independent_adaptive_params",
    "run_state_independent_adaptive_comparison",
]

for function_name in functions_to_audit:
    function_obj = globals().get(function_name)

    print("\n" + "=" * 80)
    print(f"FUNCTION: {function_name}")

    if function_obj is None:
        print("NOT FOUND")
        continue

    print("SIGNATURE:", inspect.signature(function_obj))

    try:
        source = inspect.getsource(function_obj)
        relevant_lines = [
            (line_number, line)
            for line_number, line in enumerate(source.splitlines(), start=1)
            if any(
                term in line.lower()
                for term in [
                    "base_channels",
                    "sample_random_search_params",
                    "propose_hmm_guided_params",
                    "propose_state_independent_adaptive_params",
                    "evaluate_candidate_across_folds",
                    "save_optimizer_run",
                    "search_space",
                ]
            )
        ]

        if relevant_lines:
            print("Relevant source lines:")
            for line_number, line in relevant_lines:
                print(f"{line_number:>3}: {line}")
        else:
            print("No matching lines found; full source may need inspection.")

    except (OSError, TypeError) as exc:
        print("Source unavailable:", exc)

print("\n" + "=" * 80)
print("Current base_channels search-space definition:")
print(COMPARISON_CONFIG.get("search_space", {}).get("base_channels", "NOT FOUND"))

print("\nCurrent SEARCH_CONFIG base_channels definition:")
print(SEARCH_CONFIG.get("base_channels", "NOT FOUND"))


# Cell 113 — Add a safe fixed-model-size experiment wrapper

In [ ]:

# Cell 113 — Safe fixed-model-size experiment wrapper

from copy import deepcopy

FIXED_SIZE_CONFIG = {
    "base_channels": 16,
    "seeds": [42, 123, 2026],
    "folds": [3, 4],
    "trials_per_seed": 16,
    "epochs_per_fold": 3,
    "result_prefix": "fixed16",
}

# Preserve the original search-space definitions.
ORIGINAL_COMPARISON_SEARCH_SPACE = deepcopy(
    COMPARISON_CONFIG["search_space"]
)
ORIGINAL_SEARCH_CONFIG = deepcopy(SEARCH_CONFIG)

def run_fixed16_experiment(run_function, seed):
    """
    Run one existing optimizer with base_channels fixed to 16.

    Original search-space configurations are restored even if the run fails.
    The caller must ensure the optimizer saves to a distinct result prefix.
    """
    comparison_space_before = deepcopy(COMPARISON_CONFIG["search_space"])
    search_config_before = deepcopy(SEARCH_CONFIG)

    try:
        COMPARISON_CONFIG["search_space"]["base_channels"] = [16]
        SEARCH_CONFIG["base_channels"] = [16]

        run = run_function(
            trials_per_search=FIXED_SIZE_CONFIG["trials_per_seed"],
            seed=seed,
            folds=FIXED_SIZE_CONFIG["folds"],
            epochs=FIXED_SIZE_CONFIG["epochs_per_fold"],
        )

        # Validate recorded configurations when the runner returns trial rows.
        trials = run.get("trials")
        if trials is not None and "base_channels" in trials.columns:
            observed_sizes = sorted(
                trials["base_channels"].dropna().astype(int).unique().tolist()
            )
            if observed_sizes != [16]:
                raise AssertionError(
                    f"Expected only base_channels=16; got {observed_sizes}"
                )

        return run

    finally:
        COMPARISON_CONFIG["search_space"] = comparison_space_before
        SEARCH_CONFIG.clear()
        SEARCH_CONFIG.update(search_config_before)

print("Fixed-size wrapper defined.")
print("Target model size:", FIXED_SIZE_CONFIG["base_channels"])
print("Seeds:", FIXED_SIZE_CONFIG["seeds"])
print("Folds:", FIXED_SIZE_CONFIG["folds"])
print("Trials per optimizer per seed:", FIXED_SIZE_CONFIG["trials_per_seed"])
print("Epochs per fold:", FIXED_SIZE_CONFIG["epochs_per_fold"])

assert COMPARISON_CONFIG["search_space"]["base_channels"] == [8, 16]
assert SEARCH_CONFIG["base_channels"] == [8, 16]
print("Original configurations remain unchanged.")


# Cell 114 — Inspect the result-saving function

In [ ]:

# Cell 114 — Inspect result-saving behavior (no files are written)

import inspect

print("SAVE FUNCTION SIGNATURE")
print(inspect.signature(save_optimizer_run))

print("\nSAVE FUNCTION SOURCE")
print(inspect.getsource(save_optimizer_run))

print("\nRESULT DIRECTORY")
print("results_dir =", results_dir)
print("Resolved path =", results_dir.resolve())

print("\nEXISTING FILES THAT COULD CONFLICT")
for optimizer_prefix in [
    "as_hmm_bo",
    "tpe",
    "random_search",
    "rule_based_adaptive",
    "state_independent_adaptive",
]:
    for seed in FIXED_SIZE_CONFIG["seeds"]:
        expected = results_dir / f"{optimizer_prefix}_seed{seed}_trials.csv"
        print(f"{'EXISTS' if expected.exists() else 'not found':10} {expected.name}")


# Cell 115 — Prepare isolated filenames and verify all five runners

In [ ]:

# Cell 115 — Prepare isolated fixed16 result saving

import inspect

# Preserve the original saving function.
ORIGINAL_SAVE_OPTIMIZER_RUN = save_optimizer_run

# Locate the existing TPE runner without guessing its name.
tpe_runner_candidates = [
    name for name, obj in globals().items()
    if callable(obj)
    and "tpe" in name.lower()
    and "run" in name.lower()
    and "comparison" in name.lower()
]

print("TPE runner candidates:", tpe_runner_candidates)

# The four known runners from the earlier audit.
known_runners = {
    "AS-HMM-BO": "run_as_hmm_bo_comparison",
    "Random Search": "run_random_search_comparison",
    "Rule-Based Adaptive": "run_rule_based_adaptive_comparison",
    "State-Independent Adaptive": "run_state_independent_adaptive_comparison",
}

for label, function_name in known_runners.items():
    function_obj = globals().get(function_name)
    assert callable(function_obj), f"Missing runner: {function_name}"
    print(f"{label}: {function_name}{inspect.signature(function_obj)}")

# Preview isolated filenames; do not write anything.
print("\nExample isolated output filenames:")
for filename in [
    "fixed16_as_hmm_bo_seed42_trials.csv",
    "fixed16_tpe_seed42_trials.csv",
    "fixed16_random_search_seed42_trials.csv",
    "fixed16_rule_based_adaptive_seed42_trials.csv",
    "fixed16_state_independent_adaptive_seed42_trials.csv",
]:
    print(results_dir / filename)

print("\nNo training started. No files written.")


# Cell 116 — Install an isolated result saver

In [ ]:

# Cell 116 — Isolated fixed16 result saver

import json
import pandas as pd

# Keep a reference to the original saver.
ORIGINAL_SAVE_OPTIMIZER_RUN = save_optimizer_run

FIXED16_PREFIXES = {
    "as_hmm_bo": "as_hmm_bo",
    "tpe": "tpe",
    "random_search": "random_search",
    "rule_based_adaptive": "rule_based_adaptive",
    "state_independent_adaptive": "state_independent_adaptive",
}

def save_fixed16_optimizer_run(run, optimizer_name, seed):
    """Save fixed-size experiment results without touching original files."""

    name = str(optimizer_name).lower().replace(" ", "_")
    if name not in FIXED16_PREFIXES:
        raise ValueError(
            f"Unrecognized optimizer name {optimizer_name!r}; "
            "refusing to save to an unexpected path."
        )

    prefix = f"fixed16_{FIXED16_PREFIXES[name]}_seed{seed}"

    # Refuse to overwrite any existing fixed16 result.
    output_paths = [
        results_dir / f"{prefix}_trials.csv",
        results_dir / f"{prefix}_folds.csv",
        results_dir / f"{prefix}_summary.json",
    ]
    if "states" in run:
        output_paths.append(results_dir / f"{prefix}_states.csv")

    existing = [path.name for path in output_paths if path.exists()]
    if existing:
        raise FileExistsError(
            "Refusing to overwrite existing fixed16 results: "
            + ", ".join(existing)
        )

    run["trials"].to_csv(results_dir / f"{prefix}_trials.csv", index=False)
    run["folds"].to_csv(results_dir / f"{prefix}_folds.csv", index=False)

    if "states" in run:
        run["states"].to_csv(
            results_dir / f"{prefix}_states.csv", index=False
        )

    with open(
        results_dir / f"{prefix}_summary.json",
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(run["summary"], file, indent=2)

    print(f"Saved isolated results: {prefix}")

print("Isolated saver defined; not installed globally.")
print("Original saver preserved:", callable(ORIGINAL_SAVE_OPTIMIZER_RUN))
print("Fixed-size output directory:", results_dir.resolve())
print("No training started; no files written.")


# Cell 117 — Run a two-trial isolation smoke test

In [ ]:

# Cell 117 — Two-trial fixed16 smoke test

smoke_seed = 9876

# Refuse to overwrite previous smoke-test outputs.
smoke_paths = [
    results_dir / f"fixed16_as_hmm_bo_seed{smoke_seed}_trials.csv",
    results_dir / f"fixed16_as_hmm_bo_seed{smoke_seed}_folds.csv",
    results_dir / f"fixed16_as_hmm_bo_seed{smoke_seed}_states.csv",
    results_dir / f"fixed16_as_hmm_bo_seed{smoke_seed}_summary.json",
]
existing_smoke_paths = [p.name for p in smoke_paths if p.exists()]
assert not existing_smoke_paths, (
    "Smoke-test files already exist; refusing to overwrite: "
    + ", ".join(existing_smoke_paths)
)

# Preserve both configuration objects and the active saver.
comparison_space_before = deepcopy(COMPARISON_CONFIG["search_space"])
search_config_before = deepcopy(SEARCH_CONFIG)
saver_before = save_optimizer_run

try:
    COMPARISON_CONFIG["search_space"]["base_channels"] = [16]
    SEARCH_CONFIG["base_channels"] = [16]
    save_optimizer_run = save_fixed16_optimizer_run

    fixed16_smoke_run = run_as_hmm_bo_comparison(
        trials_per_search=2,
        seed=smoke_seed,
        folds=[3, 4],
        epochs=3,
    )
finally:
    COMPARISON_CONFIG["search_space"] = comparison_space_before
    SEARCH_CONFIG.clear()
    SEARCH_CONFIG.update(search_config_before)
    save_optimizer_run = saver_before

# Validate returned trials and the saved artifacts.
smoke_trials = fixed16_smoke_run["trials"]
assert len(smoke_trials) == 2, f"Expected 2 trials, got {len(smoke_trials)}"
assert set(smoke_trials["base_channels"].astype(int)) == {16}

saved_trials_path = (
    results_dir / f"fixed16_as_hmm_bo_seed{smoke_seed}_trials.csv"
)
saved_folds_path = (
    results_dir / f"fixed16_as_hmm_bo_seed{smoke_seed}_folds.csv"
)
assert saved_trials_path.exists(), "Isolated trial CSV was not created"
assert saved_folds_path.exists(), "Isolated fold CSV was not created"

saved_trials = pd.read_csv(saved_trials_path)
saved_folds = pd.read_csv(saved_folds_path)

assert len(saved_trials) == 2
assert set(saved_trials["base_channels"].astype(int)) == {16}
assert len(saved_folds) == 4, f"Expected 4 fold records, got {len(saved_folds)}"
assert set(saved_folds["base_channels"].astype(int)) == {16}

print("\nSMOKE TEST PASSED")
print("Trials:", len(saved_trials))
print("Fold records:", len(saved_folds))
print("Model sizes:", sorted(saved_trials["base_channels"].unique().tolist()))
print("Trial file:", saved_trials_path.name)
print("Fold file:", saved_folds_path.name)
print("Original saver restored:", save_optimizer_run is ORIGINAL_SAVE_OPTIMIZER_RUN)
print("Original search spaces restored:",
      COMPARISON_CONFIG["search_space"]["base_channels"] == [8, 16]
      and SEARCH_CONFIG["base_channels"] == [8, 16])


# Cell 118 — Diagnose where the smoke-test results went

In [ ]:

# Cell 118 — Diagnose smoke-test saving

import inspect

print("1. SAVER CALLS INSIDE AS-HMM-BO RUNNER")
runner_source = inspect.getsource(run_as_hmm_bo_comparison)
for line_number, line in enumerate(runner_source.splitlines(), start=1):
    if any(term in line.lower() for term in [
        "save_optimizer_run", "save_", "optimizer_name", "return run"
    ]):
        print(f"{line_number:>3}: {line}")

print("\n2. SMOKE-TEST RUN OBJECT")
print("Run object exists:", "fixed16_smoke_run" in globals())
if "fixed16_smoke_run" in globals():
    print("Returned keys:", list(fixed16_smoke_run.keys()))
    print("Trial rows:", len(fixed16_smoke_run.get("trials", [])))
    print("Fold rows:", len(fixed16_smoke_run.get("folds", [])))

print("\n3. MATCHING FILES IN RESULTS DIRECTORY")
matches = sorted(results_dir.glob("*9876*"))
if matches:
    for path in matches:
        print(path.name, "| bytes:", path.stat().st_size)
else:
    print("No files matching seed 9876.")

print("\n4. SAVER REFERENCES")
print("Current saver is original:",
      save_optimizer_run is ORIGINAL_SAVE_OPTIMIZER_RUN)
print("Isolated saver callable:", callable(save_fixed16_optimizer_run))

print("\n5. CONFIGURATION RESTORATION")
print("COMPARISON_CONFIG:", COMPARISON_CONFIG["search_space"]["base_channels"])
print("SEARCH_CONFIG:", SEARCH_CONFIG["base_channels"])


# Cell 119 — Save the completed smoke test safely

In [ ]:

# Cell 119 — Save existing smoke-test results without retraining

assert "fixed16_smoke_run" in globals(), "Smoke-test run is not in memory."

smoke_seed = 9876

assert len(fixed16_smoke_run["trials"]) == 2
assert len(fixed16_smoke_run["folds"]) == 4
assert set(
    fixed16_smoke_run["trials"]["base_channels"].astype(int)
) == {16}
assert set(
    fixed16_smoke_run["folds"]["base_channels"].astype(int)
) == {16}

expected_paths = [
    results_dir / f"fixed16_as_hmm_bo_seed{smoke_seed}_trials.csv",
    results_dir / f"fixed16_as_hmm_bo_seed{smoke_seed}_folds.csv",
    results_dir / f"fixed16_as_hmm_bo_seed{smoke_seed}_states.csv",
    results_dir / f"fixed16_as_hmm_bo_seed{smoke_seed}_summary.json",
]

existing_paths = [p.name for p in expected_paths if p.exists()]
assert not existing_paths, (
    "Refusing to overwrite existing files: " + ", ".join(existing_paths)
)

save_fixed16_optimizer_run(
    fixed16_smoke_run,
    optimizer_name="as_hmm_bo",
    seed=smoke_seed,
)

# Verify saved trial and fold files.
saved_trials = pd.read_csv(expected_paths[0])
saved_folds = pd.read_csv(expected_paths[1])

assert len(saved_trials) == 2
assert len(saved_folds) == 4
assert set(saved_trials["base_channels"].astype(int)) == {16}
assert set(saved_folds["base_channels"].astype(int)) == {16}

print("\nSaved and verified the existing smoke-test results.")
print("Trial rows:", len(saved_trials))
print("Fold rows:", len(saved_folds))
print("All saved configurations use base_channels=16.")
print("No retraining performed.")


# Cell 120 — Inspect saving behavior for all five runners

In [ ]:

# Cell 120 — Inspect all optimizer runners and save-call paths

import inspect

runner_names = [
    "run_as_hmm_bo_comparison",
    "run_tpe_comparison",
    "run_random_search_comparison",
    "run_rule_based_adaptive_comparison",
    "run_state_independent_adaptive_comparison",
]

for name in runner_names:
    runner = globals().get(name)

    print("\n" + "=" * 90)
    print("RUNNER:", name)

    if not callable(runner):
        print("NOT FOUND")
        continue

    print("Signature:", inspect.signature(runner))

    try:
        source = inspect.getsource(runner)
        print("Source file:", inspect.getsourcefile(runner))

        for line_number, line in enumerate(source.splitlines(), start=1):
            if any(term in line.lower() for term in [
                "save", "return run", "return {", "optimizer_name",
                "trials", "folds", "summary"
            ]):
                print(f"{line_number:>3}: {line}")

    except (OSError, TypeError) as exc:
        print("Source inspection unavailable:", exc)

print("\nCurrent saver:", save_optimizer_run.__name__)
print("Original saver preserved:",
      save_optimizer_run is ORIGINAL_SAVE_OPTIMIZER_RUN)


# Cell 121 — Define the guarded runner for all five optimizers

In [ ]:

# Cell 121 — Guarded runner and isolated saving for fixed16 experiments

FIXED16_RUNNERS = {
    "as_hmm_bo": run_as_hmm_bo_comparison,
    "tpe": run_tpe_comparison,
    "random_search": run_random_search_comparison,
    "rule_based_adaptive": run_rule_based_adaptive_comparison,
    "state_independent_adaptive": run_state_independent_adaptive_comparison,
}

def run_fixed16_and_save(optimizer_key, seed):
    """
    Run one optimizer with base_channels=16, validate the returned run,
    and save only to isolated fixed16 filenames.
    """
    if optimizer_key not in FIXED16_RUNNERS:
        raise ValueError(f"Unknown optimizer key: {optimizer_key}")

    trials_count = FIXED_SIZE_CONFIG["trials_per_seed"]
    folds = FIXED_SIZE_CONFIG["folds"]
    epochs = FIXED_SIZE_CONFIG["epochs_per_fold"]

    # Refuse to overwrite any results from this optimizer/seed.
    prefix = f"fixed16_{optimizer_key}_seed{seed}"
    expected_files = [
        results_dir / f"{prefix}_trials.csv",
        results_dir / f"{prefix}_folds.csv",
        results_dir / f"{prefix}_summary.json",
    ]
    if optimizer_key in {"as_hmm_bo", "rule_based_adaptive"}:
        expected_files.append(results_dir / f"{prefix}_states.csv")

    existing = [path.name for path in expected_files if path.exists()]
    if existing:
        raise FileExistsError(
            "Refusing to overwrite existing results: " + ", ".join(existing)
        )

    comparison_before = deepcopy(COMPARISON_CONFIG["search_space"])
    search_before = deepcopy(SEARCH_CONFIG)
    saver_before = save_optimizer_run

    try:
        COMPARISON_CONFIG["search_space"]["base_channels"] = [16]
        SEARCH_CONFIG["base_channels"] = [16]

        # Suppress runners' internal saves; we save once after validation.
        save_optimizer_run = lambda *args, **kwargs: None

        run = FIXED16_RUNNERS[optimizer_key](
            trials_per_search=trials_count,
            seed=seed,
            folds=folds,
            epochs=epochs,
        )
    finally:
        COMPARISON_CONFIG["search_space"] = comparison_before
        SEARCH_CONFIG.clear()
        SEARCH_CONFIG.update(search_before)
        save_optimizer_run = saver_before

    # Validate results before saving anything.
    trials = run["trials"]
    fold_records = run["folds"]

    assert len(trials) == trials_count, (
        f"Expected {trials_count} trials; got {len(trials)}"
    )
    assert len(fold_records) == trials_count * len(folds), (
        f"Expected {trials_count * len(folds)} fold records; "
        f"got {len(fold_records)}"
    )
    assert set(trials["base_channels"].astype(int)) == {16}
    assert set(fold_records["base_channels"].astype(int)) == {16}
    assert set(fold_records["fold"].astype(int)).issubset(set(folds))

    # Save to the isolated namespace, never to original filenames.
    save_fixed16_optimizer_run(
        run,
        optimizer_name=optimizer_key,
        seed=seed,
    )

    print(f"\nValidated: {optimizer_key}, seed={seed}")
    print(f"Trials: {len(trials)} | Fold records: {len(fold_records)}")
    print("Model size: base_channels=16")
    print("Development folds:", folds)
    print("Original saver and search spaces restored.")

    return run


# Validate the wrapper setup without starting training.
assert set(FIXED16_RUNNERS) == set(FIXED16_PREFIXES)
assert save_optimizer_run is ORIGINAL_SAVE_OPTIMIZER_RUN
assert COMPARISON_CONFIG["search_space"]["base_channels"] == [8, 16]
assert SEARCH_CONFIG["base_channels"] == [8, 16]

print("Guarded runner defined for all five optimizers.")
print("No training started; no experiment files written.")


In [ ]:

# Cell 123 — Corrected guarded runner

def run_fixed16_and_save(optimizer_key, seed):
    global save_optimizer_run

    if optimizer_key not in FIXED16_RUNNERS:
        raise ValueError(f"Unknown optimizer key: {optimizer_key}")

    trials_count = FIXED_SIZE_CONFIG["trials_per_seed"]
    folds = FIXED_SIZE_CONFIG["folds"]
    epochs = FIXED_SIZE_CONFIG["epochs_per_fold"]

    prefix = f"fixed16_{optimizer_key}_seed{seed}"
    expected_files = [
        results_dir / f"{prefix}_trials.csv",
        results_dir / f"{prefix}_folds.csv",
        results_dir / f"{prefix}_summary.json",
    ]
    if optimizer_key in {"as_hmm_bo", "rule_based_adaptive"}:
        expected_files.append(results_dir / f"{prefix}_states.csv")

    existing = [path.name for path in expected_files if path.exists()]
    if existing:
        raise FileExistsError(
            "Refusing to overwrite existing results: " + ", ".join(existing)
        )

    comparison_before = deepcopy(COMPARISON_CONFIG["search_space"])
    search_before = deepcopy(SEARCH_CONFIG)
    saver_before = save_optimizer_run

    try:
        COMPARISON_CONFIG["search_space"]["base_channels"] = [16]
        SEARCH_CONFIG["base_channels"] = [16]

        # Disable internal saves; save the validated result once afterward.
        save_optimizer_run = lambda *args, **kwargs: None

        run = FIXED16_RUNNERS[optimizer_key](
            trials_per_search=trials_count,
            seed=seed,
            folds=folds,
            epochs=epochs,
        )
    finally:
        COMPARISON_CONFIG["search_space"] = comparison_before
        SEARCH_CONFIG.clear()
        SEARCH_CONFIG.update(search_before)
        save_optimizer_run = saver_before

    trials = run["trials"]
    fold_records = run["folds"]

    assert len(trials) == trials_count
    assert len(fold_records) == trials_count * len(folds)
    assert set(trials["base_channels"].astype(int)) == {16}
    assert set(fold_records["base_channels"].astype(int)) == {16}
    assert set(fold_records["fold"].astype(int)).issubset(set(folds))

    save_fixed16_optimizer_run(
        run,
        optimizer_name=optimizer_key,
        seed=seed,
    )

    print(f"\nValidated: {optimizer_key}, seed={seed}")
    print(f"Trials: {len(trials)} | Fold records: {len(fold_records)}")
    print("Model size: base_channels=16")
    print("Development folds:", folds)
    print("Original saver and search spaces restored.")

    return run


# Confirm the fix without launching training.
assert save_optimizer_run is ORIGINAL_SAVE_OPTIMIZER_RUN
assert COMPARISON_CONFIG["search_space"]["base_channels"] == [8, 16]
assert SEARCH_CONFIG["base_channels"] == [8, 16]

print("Scoping fix applied. No training started.")


# Cell 122 — Run AS-HMM-BO, seed 42

In [ ]:
fixed16_as_hmm_seed42 = run_fixed16_and_save("as_hmm_bo", 42)

# Next: Cell 124 — AS-HMM-BO, seed 123

In [ ]:

# Cell 124 — Fixed16 AS-HMM-BO, seed 123

fixed16_as_hmm_seed123 = run_fixed16_and_save("as_hmm_bo", 123)


# Cell 125 — Fixed16 AS-HMM-BO, seed 2026

In [ ]:

# Cell 125 — Fixed16 AS-HMM-BO, seed 2026

fixed16_as_hmm_seed2026 = run_fixed16_and_save("as_hmm_bo", 2026)


# Cell 126 — Fixed16 Optuna TPE, seed 42

In [ ]:

# Cell 126 — Fixed16 Optuna TPE, seed 42

fixed16_tpe_seed42 = run_fixed16_and_save("tpe", 42)

# Cell 127 — Fixed16 Optuna TPE, seed 123

In [ ]:

# Cell 127 — Fixed16 Optuna TPE, seed 123

fixed16_tpe_seed123 = run_fixed16_and_save("tpe", 123)

# Cell 128 — Fixed16 Optuna TPE, seed 2026

In [ ]:
# Cell 128 — Fixed16 Optuna TPE, seed 2026

fixed16_tpe_seed2026 = run_fixed16_and_save("tpe", 2026)

# Cell 129 — Run all remaining fixed16 experiments

In [ ]:

# Cell 129 — Complete all remaining fixed16 experiments
# Expected: 9 runs, 16 trials per run, 3 epochs per fold.
# Existing AS-HMM-BO and TPE results are not rerun.

FIXED16_REMAINING_RUNS = [
    ("random_search", 42),
    ("random_search", 123),
    ("random_search", 2026),
    ("rule_based_adaptive", 42),
    ("rule_based_adaptive", 123),
    ("rule_based_adaptive", 2026),
    ("state_independent_adaptive", 42),
    ("state_independent_adaptive", 123),
    ("state_independent_adaptive", 2026),
]

fixed16_remaining_results = {}

for optimizer_key, seed in FIXED16_REMAINING_RUNS:
    run_key = f"{optimizer_key}_seed{seed}"
    print("\n" + "=" * 76)
    print(f"Starting fixed16 run: {run_key}")
    print("=" * 76)

    # Skip runs whose output files already exist.
    prefix = f"fixed16_{optimizer_key}_seed{seed}"
    required_files = [
        results_dir / f"{prefix}_trials.csv",
        results_dir / f"{prefix}_folds.csv",
        results_dir / f"{prefix}_summary.json",
    ]
    if all(path.exists() for path in required_files):
        print(f"SKIPPED: saved results already exist for {run_key}")
        continue

    result = run_fixed16_and_save(optimizer_key, seed)
    fixed16_remaining_results[run_key] = result

print("\nRemaining fixed16 experiment batch finished.")
print("Runs executed in this session:", len(fixed16_remaining_results))
print("Existing runs were skipped rather than overwritten.")


# Cell 130 — Aggregate all fixed16 optimizer results

In [ ]:

# Cell 130 — Aggregate all fixed16 optimizer results

import json
import pandas as pd
from pathlib import Path

fixed16_methods = {
    "AS-HMM-BO": "as_hmm_bo",
    "Optuna TPE": "tpe",
    "Random Search": "random_search",
    "Rule-Based Adaptive": "rule_based_adaptive",
    "State-Independent Adaptive": "state_independent_adaptive",
}
fixed16_seeds = [42, 123, 2026]

# Locate the output directory used by the earlier experiment cells.
fixed16_dir = Path(results_dir)
fixed16_rows = []
fixed16_missing = []

def find_summary_value(obj, key_options):
    """Find a summary field recursively, tolerating naming differences."""
    if isinstance(obj, dict):
        normalized = {
            str(k).lower().replace(" ", "_"): v
            for k, v in obj.items()
        }
        for key in key_options:
            if key in normalized and not isinstance(normalized[key], (dict, list)):
                try:
                    return float(normalized[key])
                except (TypeError, ValueError):
                    pass
        for value in obj.values():
            result = find_summary_value(value, key_options)
            if result is not None:
                return result
    elif isinstance(obj, list):
        for value in obj:
            result = find_summary_value(value, key_options)
            if result is not None:
                return result
    return None

for method_label, method_key in fixed16_methods.items():
    for seed in fixed16_seeds:
        prefix = f"fixed16_{method_key}_seed{seed}"
        summary_path = fixed16_dir / f"{prefix}_summary.json"

        if not summary_path.exists():
            fixed16_missing.append(str(summary_path))
            continue

        with open(summary_path, "r", encoding="utf-8") as f:
            summary = json.load(f)

        best_dice = find_summary_value(
            summary, ["best_validation_dice", "best_dice"]
        )
        mean_dice = find_summary_value(
            summary, ["mean_trial_dice", "mean_dice"]
        )
        runtime = find_summary_value(
            summary, ["runtime_seconds", "total_runtime_seconds"]
        )

        if best_dice is None or runtime is None:
            print(f"Could not identify required fields in {summary_path.name}")
            print("Available summary content:", summary)
            continue

        fixed16_rows.append({
            "Optimizer": method_label,
            "Seed": seed,
            "Best validation Dice": best_dice,
            "Mean trial Dice": mean_dice,
            "Runtime (seconds)": runtime,
            "Runtime (minutes)": runtime / 60,
        })

fixed16_per_seed = pd.DataFrame(fixed16_rows)

if fixed16_missing:
    print("Missing summary files:")
    for path in fixed16_missing:
        print(" -", path)

if fixed16_per_seed.empty:
    raise RuntimeError(
        "No usable fixed16 summaries were found. Check results_dir and "
        "the summary JSON field names printed above."
    )

print("\nPER-SEED RESULTS")
display(
    fixed16_per_seed.sort_values(["Optimizer", "Seed"])
    .round(4)
    .reset_index(drop=True)
)

fixed16_aggregate = (
    fixed16_per_seed.groupby("Optimizer")
    .agg(
        Seeds=("Seed", "nunique"),
        Best_Dice_Mean=("Best validation Dice", "mean"),
        Best_Dice_SD=("Best validation Dice", "std"),
        Mean_Trial_Dice=("Mean trial Dice", "mean"),
        Mean_Trial_Dice_SD=("Mean trial Dice", "std"),
        Runtime_Minutes_Mean=("Runtime (minutes)", "mean"),
        Runtime_Minutes_SD=("Runtime (minutes)", "std"),
    )
    .reset_index()
)

print("\nAGGREGATE RESULTS — MEAN ± STANDARD DEVIATION ACROSS SEEDS")
display(fixed16_aggregate.round(4))

aggregate_path = fixed16_dir / "fixed16_aggregate_summary.csv"
per_seed_path = fixed16_dir / "fixed16_per_seed_summary.csv"

fixed16_aggregate.to_csv(aggregate_path, index=False)
fixed16_per_seed.to_csv(per_seed_path, index=False)

print("\nSaved aggregate table:", aggregate_path)
print("Saved per-seed table:", per_seed_path)

if fixed16_per_seed.groupby("Optimizer")["Seed"].nunique().min() < 3:
    print(
        "\nWARNING: At least one optimizer is missing a seed. "
        "Do not treat the comparison as complete until all three are present."
    )
else:
    print("\nAll five optimizers have three seeds represented.")


# Cell 131 — Corrected fixed16 aggregation

In [ ]:

# Cell 131 — Corrected fixed16 aggregation
# Reads existing summaries only; does not retrain or overwrite experiment results.

import json
import pandas as pd
from pathlib import Path

fixed16_dir = Path(results_dir)

fixed16_methods = {
    "AS-HMM-BO": "as_hmm_bo",
    "Optuna TPE": "tpe",
    "Random Search": "random_search",
    "Rule-Based Adaptive": "rule_based_adaptive",
    "State-Independent Adaptive": "state_independent_adaptive",
}
fixed16_seeds = [42, 123, 2026]

rows = []
missing = []
invalid = []

for label, key in fixed16_methods.items():
    for seed in fixed16_seeds:
        prefix = f"fixed16_{key}_seed{seed}"
        path = fixed16_dir / f"{prefix}_summary.json"

        if not path.exists():
            missing.append(str(path))
            continue

        with open(path, "r", encoding="utf-8") as f:
            s = json.load(f)

        # Actual summary keys used by the experiment runners.
        best = s.get("best_mean_validation_dice",
                     s.get("best_validation_dice"))
        mean_trial = s.get("mean_trial_dice")
        runtime = s.get("total_wall_clock_seconds",
                        s.get("runtime_seconds"))

        if best is None or mean_trial is None or runtime is None:
            invalid.append((path.name, s))
            continue

        rows.append({
            "Optimizer": label,
            "Seed": seed,
            "Best validation Dice": float(best),
            "Mean trial Dice": float(mean_trial),
            "Runtime (seconds)": float(runtime),
            "Runtime (minutes)": float(runtime) / 60,
            "Trials completed": int(s.get("trials_completed", 0)),
            "Test sets used": s.get("test_sets_used", None),
        })

per_seed = pd.DataFrame(rows)

print("Valid summary records:", len(per_seed), "/ 15")
if missing:
    print("\nMissing files:")
    for p in missing:
        print(" -", p)
if invalid:
    print("\nSummaries with unrecognized fields:")
    for name, content in invalid:
        print(name, content)

if not missing and not invalid:
    counts = per_seed.groupby("Optimizer")["Seed"].nunique()
    assert len(per_seed) == 15, "Expected exactly 15 summary records."
    assert (counts == 3).all(), "Each optimizer must have all three seeds."
    assert (per_seed["Trials completed"] == 16).all(), (
        "At least one run did not complete 16 trials."
    )
    assert not per_seed["Test sets used"].eq(True).any(), (
        "A summary indicates that a test set was used."
    )

print("\nPER-SEED RESULTS")
display(
    per_seed.sort_values(["Optimizer", "Seed"])
    .round(4)
    .reset_index(drop=True)
)

aggregate = (
    per_seed.groupby("Optimizer", sort=False)
    .agg(
        Seeds=("Seed", "nunique"),
        Best_Dice_Mean=("Best validation Dice", "mean"),
        Best_Dice_SD=("Best validation Dice", "std"),
        Mean_Trial_Dice=("Mean trial Dice", "mean"),
        Mean_Trial_Dice_SD=("Mean trial Dice", "std"),
        Runtime_Minutes_Mean=("Runtime (minutes)", "mean"),
        Runtime_Minutes_SD=("Runtime (minutes)", "std"),
    )
    .reset_index()
)

print("\nAGGREGATE RESULTS — MEAN ± SAMPLE SD ACROSS SEEDS")
display(aggregate.round(4))

if not missing and not invalid and len(per_seed) == 15:
    aggregate.to_csv(
        fixed16_dir / "fixed16_aggregate_summary.csv", index=False
    )
    per_seed.to_csv(
        fixed16_dir / "fixed16_per_seed_summary.csv", index=False
    )
    print("\nCorrected aggregate and per-seed CSV files saved.")
else:
    print(
        "\nWARNING: completeness checks failed. "
        "CSV outputs were not replaced."
    )


# Cell 138 — Inspect trial counts and evaluation metadata

In [ ]:

# Cell 138 — Inspect trial counts and evaluation metadata

display(
    per_seed[
        [
            "Optimizer",
            "Seed",
            "Trials completed",
            "Test sets used",
            "Best validation Dice",
            "Runtime (minutes)",
        ]
    ].sort_values(["Optimizer", "Seed"]).reset_index(drop=True)
)

print("\nRaw metadata from each summary:")
for _, row in per_seed.iterrows():
    key = fixed16_methods[row["Optimizer"]]
    path = fixed16_dir / f"fixed16_{key}_seed{int(row['Seed'])}_summary.json"

    with open(path, "r", encoding="utf-8") as f:
        summary = json.load(f)

    print(
        f"{row['Optimizer']} | seed={int(row['Seed'])} | "
        f"trials_completed={summary.get('trials_completed', '<missing>')} | "
        f"test_sets_used={summary.get('test_sets_used', '<missing>')}"
    )


# Cell 139 — Validate fixed16 results using summaries and saved records

In [ ]:

# Cell 139 — Validate fixed16 results using summaries and saved records

import json
import pandas as pd

validation_rows = []
validation_errors = []

for _, row in per_seed.iterrows():
    optimizer = row["Optimizer"]
    seed = int(row["Seed"])
    key = fixed16_methods[optimizer]

    prefix = f"fixed16_{key}_seed{seed}"
    summary_path = fixed16_dir / f"{prefix}_summary.json"

    with open(summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)

    # Find saved trial and fold records.
    trial_path = fixed16_dir / f"{prefix}_trials.csv"
    fold_path = fixed16_dir / f"{prefix}_folds.csv"

    if not trial_path.exists():
        validation_errors.append(f"{optimizer}, seed {seed}: missing trials CSV")
        continue

    trials_df = pd.read_csv(trial_path)
    observed_trials = len(trials_df)

    observed_folds = None
    if fold_path.exists():
        folds_df = pd.read_csv(fold_path)
        observed_folds = len(folds_df)

    # Use metadata when present; otherwise rely on saved records.
    metadata_trials = summary.get("trials_completed")
    trials_valid = (
        metadata_trials == 16
        if metadata_trials is not None
        else observed_trials == 16
    )

    metadata_test_use = summary.get("test_sets_used")
    test_use_valid = (
        metadata_test_use is False
        if metadata_test_use is not None
        else None
    )

    folds_valid = observed_folds == 32 if observed_folds is not None else False

    validation_rows.append({
        "Optimizer": optimizer,
        "Seed": seed,
        "Summary trials field": metadata_trials,
        "Saved trial rows": observed_trials,
        "Saved fold rows": observed_folds,
        "Trials valid": trials_valid,
        "Folds valid": folds_valid,
        "Test metadata": metadata_test_use,
    })

    if not trials_valid:
        validation_errors.append(
            f"{optimizer}, seed {seed}: expected 16 trials, found {observed_trials}"
        )

    if not folds_valid:
        validation_errors.append(
            f"{optimizer}, seed {seed}: expected 32 fold rows, found {observed_folds}"
        )

display(
    pd.DataFrame(validation_rows).sort_values(
        ["Optimizer", "Seed"]
    ).reset_index(drop=True)
)

if validation_errors:
    print("\nVALIDATION ERRORS:")
    for error in validation_errors:
        print("-", error)
else:
    print(
        "\nAll 15 runs have 16 saved trial rows and 32 saved fold rows."
    )
    print(
        "For State-Independent Adaptive, test-set non-use must be confirmed "
        "from its runner/configuration or execution logs because its summary "
        "does not record that metadata."
    )


# Cell 140 — Aggregate fixed16 results after record validation

In [ ]:

# Cell 140 — Aggregate fixed16 results after record validation

import numpy as np
import pandas as pd
import json

methods_order = [
    "AS-HMM-BO",
    "Optuna TPE",
    "Random Search",
    "Rule-Based Adaptive",
    "State-Independent Adaptive",
]

# Rebuild per-seed metrics from the validated summary records.
aggregate_rows = []

for _, row in per_seed.iterrows():
    optimizer = row["Optimizer"]
    seed = int(row["Seed"])
    key = fixed16_methods[optimizer]
    prefix = f"fixed16_{key}_seed{seed}"

    with open(fixed16_dir / f"{prefix}_summary.json", "r", encoding="utf-8") as f:
        summary = json.load(f)

    trials_df = pd.read_csv(fixed16_dir / f"{prefix}_trials.csv")
    folds_df = pd.read_csv(fixed16_dir / f"{prefix}_folds.csv")

    # Read summary metrics using the known schema variants.
    best_dice = summary.get(
        "best_mean_validation_dice",
        summary.get("best_validation_dice")
    )
    mean_trial_dice = summary.get("mean_trial_dice")

    runtime = summary.get(
        "total_wall_clock_seconds",
        summary.get("runtime_seconds")
    )

    # Fallback to saved trial records only if a summary metric is absent.
    if best_dice is None:
        possible_cols = [
            c for c in trials_df.columns
            if "dice" in c.lower() and "best" in c.lower()
        ]
        if possible_cols:
            best_dice = pd.to_numeric(
                trials_df[possible_cols[0]], errors="coerce"
            ).max()

    if mean_trial_dice is None:
        dice_cols = [
            c for c in trials_df.columns
            if "dice" in c.lower()
            and ("mean" in c.lower() or "validation" in c.lower())
        ]
        if dice_cols:
            mean_trial_dice = pd.to_numeric(
                trials_df[dice_cols[0]], errors="coerce"
            ).mean()

    aggregate_rows.append({
        "Optimizer": optimizer,
        "Seed": seed,
        "Trials completed": len(trials_df),
        "Fold evaluations": len(folds_df),
        "Best validation Dice": best_dice,
        "Mean trial Dice": mean_trial_dice,
        "Runtime (seconds)": runtime,
        "Test metadata": summary.get("test_sets_used", None),
    })

fixed16_per_seed = pd.DataFrame(aggregate_rows)

# Require complete, finite metrics before producing the aggregate.
metric_cols = [
    "Best validation Dice",
    "Mean trial Dice",
    "Runtime (seconds)",
]

assert len(fixed16_per_seed) == 15
assert fixed16_per_seed.groupby("Optimizer")["Seed"].nunique().eq(3).all()
assert fixed16_per_seed["Trials completed"].eq(16).all()
assert fixed16_per_seed["Fold evaluations"].eq(32).all()
assert np.isfinite(fixed16_per_seed[metric_cols].to_numpy(dtype=float)).all()

fixed16_per_seed["Runtime (minutes)"] = (
    fixed16_per_seed["Runtime (seconds)"] / 60
)

summary_rows = []

for optimizer in methods_order:
    group = fixed16_per_seed[
        fixed16_per_seed["Optimizer"] == optimizer
    ]

    result = {"Optimizer": optimizer}

    for metric in [
        "Best validation Dice",
        "Mean trial Dice",
        "Runtime (minutes)",
    ]:
        values = group[metric].astype(float)
        result[f"{metric} mean"] = values.mean()
        result[f"{metric} SD"] = values.std(ddof=1)

    summary_rows.append(result)

fixed16_aggregate = pd.DataFrame(summary_rows)

display(
    fixed16_per_seed[
        [
            "Optimizer", "Seed", "Trials completed",
            "Fold evaluations", "Best validation Dice",
            "Mean trial Dice", "Runtime (minutes)", "Test metadata",
        ]
    ].sort_values(["Optimizer", "Seed"]).reset_index(drop=True)
)

display(fixed16_aggregate.round(6))

# Save only after every assertion succeeds.
fixed16_per_seed.to_csv(
    fixed16_dir / "fixed16_per_seed_summary.csv", index=False
)
fixed16_aggregate.to_csv(
    fixed16_dir / "fixed16_aggregate_summary.csv", index=False
)

print("Saved fixed16_per_seed_summary.csv")
print("Saved fixed16_aggregate_summary.csv")
print("All 15 runs passed trial/fold-count and metric-completeness checks.")
print(
    "Note: State-Independent Adaptive test-set metadata remains missing "
    "until verified from its runner/configuration."
)


# Cell 141 — Per-seed paired differences against AS-HMM-BO

In [ ]:

# Cell 141 — Per-seed paired differences against AS-HMM-BO

metric = "Best validation Dice"

paired = fixed16_per_seed.pivot(
    index="Seed",
    columns="Optimizer",
    values=metric,
)

baselines = [
    "Optuna TPE",
    "Random Search",
    "Rule-Based Adaptive",
    "State-Independent Adaptive",
]

difference_rows = []

for baseline in baselines:
    differences = paired["AS-HMM-BO"] - paired[baseline]

    difference_rows.append({
        "Comparison": f"AS-HMM-BO minus {baseline}",
        "Seed 42 difference": differences.loc[42],
        "Seed 123 difference": differences.loc[123],
        "Seed 2026 difference": differences.loc[2026],
        "Mean paired difference": differences.mean(),
        "Seeds AS-HMM-BO wins": int((differences > 0).sum()),
        "Seeds tied": int((differences == 0).sum()),
    })

paired_comparison = pd.DataFrame(difference_rows)

display(paired_comparison.round(6))

paired_comparison.to_csv(
    fixed16_dir / "fixed16_paired_comparison.csv",
    index=False,
)

print("Saved fixed16_paired_comparison.csv")


# Cell 142 — Inspect fixed16 trial-record schemas

In [ ]:
# Cell 142 — Inspect fixed16 trial-record schemas

for optimizer, key in fixed16_methods.items():
    path = fixed16_dir / f"fixed16_{key}_seed42_trials.csv"
    df = pd.read_csv(path)

    print(f"\n{optimizer}")
    print("Columns:", df.columns.tolist())
    print(df.head(2).to_string(index=False))

# Cell 143 — Search-efficiency trajectories and AUC

In [ ]:

# Cell 143 — Fixed16 search-efficiency analysis
# Uses saved trial records only. Does not rerun experiments.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

methods_order = [
    "AS-HMM-BO",
    "Optuna TPE",
    "Random Search",
    "Rule-Based Adaptive",
    "State-Independent Adaptive",
]

display_names = {
    "as_hmm_bo": "AS-HMM-BO",
    "optuna_tpe": "Optuna TPE",
    "random_search": "Random Search",
    "rule_based_adaptive": "Rule-Based Adaptive",
    "state_independent_adaptive": "State-Independent Adaptive",
}

trajectory_frames = []

for optimizer, key in fixed16_methods.items():
    display_name = display_names.get(key, optimizer)

    for seed in [42, 123, 2026]:
        path = fixed16_dir / f"fixed16_{key}_seed{seed}_trials.csv"

        if not path.exists():
            raise FileNotFoundError(f"Missing trial file: {path}")

        df = pd.read_csv(path).sort_values("trial").reset_index(drop=True)

        assert len(df) == 16, f"{display_name}, seed {seed}: expected 16 trials"
        assert df["mean_validation_dice"].notna().all()

        # Derive best-so-far values for every method consistently.
        df["best_so_far_dice"] = df["mean_validation_dice"].cummax()

        # Derive cumulative time if it was not saved.
        if (
            "cumulative_wall_clock_seconds" not in df.columns
            or df["cumulative_wall_clock_seconds"].isna().any()
        ):
            df["cumulative_wall_clock_seconds"] = (
                df["wall_clock_seconds"].cumsum()
            )

        df["Optimizer"] = display_name
        df["Seed"] = seed
        trajectory_frames.append(df)

fixed16_trajectories = pd.concat(
    trajectory_frames, ignore_index=True
)

# Validate consistent budgets and increasing elapsed time.
assert fixed16_trajectories.groupby(
    ["Optimizer", "Seed"]
).size().eq(16).all()

for (optimizer, seed), group in fixed16_trajectories.groupby(
    ["Optimizer", "Seed"]
):
    times = group["cumulative_wall_clock_seconds"].to_numpy()
    assert np.all(np.diff(times) >= 0), (
        f"Non-monotonic cumulative time: {optimizer}, seed {seed}"
    )

# Save the unified trajectories.
fixed16_trajectories.to_csv(
    fixed16_dir / "fixed16_search_trajectories.csv",
    index=False,
)

# Per-run AUC: normalize by trial budget or total elapsed time.
auc_rows = []

for (optimizer, seed), group in fixed16_trajectories.groupby(
    ["Optimizer", "Seed"]
):
    group = group.sort_values("trial")

    trials = group["trial"].to_numpy(dtype=float)
    best_dice = group["best_so_far_dice"].to_numpy(dtype=float)
    seconds = group["cumulative_wall_clock_seconds"].to_numpy(dtype=float)

    trial_auc = np.trapezoid(best_dice, trials) / (
        trials[-1] - trials[0]
    )

    time_auc = np.trapezoid(best_dice, seconds) / (
        seconds[-1] - seconds[0]
    )

    auc_rows.append({
        "Optimizer": optimizer,
        "Seed": seed,
        "Trial AUC normalized": trial_auc,
        "Time AUC normalized": time_auc,
        "Final best validation Dice": best_dice[-1],
        "Seconds to final trial": seconds[-1],
    })

fixed16_auc_per_seed = pd.DataFrame(auc_rows)

fixed16_auc_summary = (
    fixed16_auc_per_seed.groupby("Optimizer")
    .agg(
        Trial_AUC_mean=("Trial AUC normalized", "mean"),
        Trial_AUC_SD=("Trial AUC normalized", "std"),
        Time_AUC_mean=("Time AUC normalized", "mean"),
        Time_AUC_SD=("Time AUC normalized", "std"),
        Final_best_Dice_mean=("Final best validation Dice", "mean"),
        Final_best_Dice_SD=("Final best validation Dice", "std"),
        Runtime_seconds_mean=("Seconds to final trial", "mean"),
        Runtime_seconds_SD=("Seconds to final trial", "std"),
    )
    .reindex(methods_order)
    .reset_index()
    .rename(columns={"index": "Optimizer"})
)

display(fixed16_auc_per_seed.round(6))
display(fixed16_auc_summary.round(6))

fixed16_auc_per_seed.to_csv(
    fixed16_dir / "fixed16_search_efficiency_per_seed.csv",
    index=False,
)
fixed16_auc_summary.to_csv(
    fixed16_dir / "fixed16_search_efficiency_summary.csv",
    index=False,
)

# Plot 1: best-so-far Dice versus trial count, mean ± 1 SD across seeds.
plt.figure(figsize=(10, 6))

for optimizer in methods_order:
    subset = fixed16_trajectories[
        fixed16_trajectories["Optimizer"] == optimizer
    ]

    stats = subset.groupby("trial")["best_so_far_dice"].agg(
        ["mean", "std"]
    )

    x = stats.index.to_numpy()
    y = stats["mean"].to_numpy()
    sd = stats["std"].fillna(0).to_numpy()

    plt.plot(x, y, marker="o", markersize=3, label=optimizer)
    plt.fill_between(x, y - sd, y + sd, alpha=0.12)

plt.xlabel("Trial")
plt.ylabel("Best-so-far mean validation Dice")
plt.title("Fixed16 search progress by trial")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    fixed16_dir / "fixed16_best_so_far_by_trial.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

# Plot 2: best-so-far Dice versus elapsed time.
# Interpolate each run onto a common 0–100% runtime grid.
plt.figure(figsize=(10, 6))
time_grid = np.linspace(0, 1, 101)

for optimizer in methods_order:
    normalized_curves = []

    for seed in [42, 123, 2026]:
        group = fixed16_trajectories[
            (fixed16_trajectories["Optimizer"] == optimizer)
            & (fixed16_trajectories["Seed"] == seed)
        ].sort_values("trial")

        seconds = group["cumulative_wall_clock_seconds"].to_numpy()
        scores = group["best_so_far_dice"].to_numpy()

        # Add the initial point: no completed trials, no score yet.
        relative_time = np.r_[0.0, seconds / seconds[-1]]
        curve = np.r_[scores[0], scores]

        normalized_curves.append(
            np.interp(time_grid, relative_time, curve)
        )

    curves = np.vstack(normalized_curves)
    mean_curve = curves.mean(axis=0)
    sd_curve = curves.std(axis=0, ddof=1)

    plt.plot(
        time_grid * 100,
        mean_curve,
        label=optimizer,
    )
    plt.fill_between(
        time_grid * 100,
        mean_curve - sd_curve,
        mean_curve + sd_curve,
        alpha=0.12,
    )

plt.xlabel("Elapsed runtime (% of each run's total)")
plt.ylabel("Best-so-far mean validation Dice")
plt.title("Fixed16 search progress by normalized elapsed time")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(
    fixed16_dir / "fixed16_best_so_far_by_time.png",
    dpi=160,
    bbox_inches="tight",
)
plt.show()

print("Saved:")
print("- fixed16_search_trajectories.csv")
print("- fixed16_search_efficiency_per_seed.csv")
print("- fixed16_search_efficiency_summary.csv")
print("- fixed16_best_so_far_by_trial.png")
print("- fixed16_best_so_far_by_time.png")


# Cell 144 — Paired search-efficiency comparisons across seeds

In [ ]:
# Cell 144 — Paired search-efficiency comparisons across seeds

from pathlib import Path
import pandas as pd

# Locate the project root from the notebook's current working directory.
current_dir = Path.cwd().resolve()
fixed16_dir = None

for directory in [current_dir, *current_dir.parents]:
    candidate = directory / "outputs" / "cross_validation"

    if (candidate / "fixed16_search_trajectories.csv").exists():
        fixed16_dir = candidate
        break

if fixed16_dir is None:
    raise FileNotFoundError(
        "Could not locate the experiment results.\n"
        f"Current working directory: {current_dir}\n"
        "Expected to find outputs/cross_validation/"
        "fixed16_search_trajectories.csv in the project or a parent directory."
    )

print("Results directory:", fixed16_dir)

# Load the per-seed efficiency results generated by Cell 143.
efficiency_file = fixed16_dir / "fixed16_search_efficiency_per_seed.csv"

if not efficiency_file.exists():
    raise FileNotFoundError(
        f"Missing efficiency CSV: {efficiency_file}\n"
        "The project root was located, but the efficiency CSV is missing. "
        "Check whether Cell 143 completed successfully."
    )

fixed16_auc_per_seed = pd.read_csv(efficiency_file)

auc_metrics = [
    "Trial AUC normalized",
    "Time AUC normalized",
    "Final best validation Dice",
    "Seconds to final trial",
]

baselines = [
    "Optuna TPE",
    "Random Search",
    "Rule-Based Adaptive",
    "State-Independent Adaptive",
]

required_columns = ["Seed", "Optimizer"] + auc_metrics
missing_columns = [
    col for col in required_columns
    if col not in fixed16_auc_per_seed.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}\n"
        f"Available columns: {fixed16_auc_per_seed.columns.tolist()}"
    )

paired_efficiency_rows = []

for metric in auc_metrics:
    pivot = fixed16_auc_per_seed.pivot(
        index="Seed",
        columns="Optimizer",
        values=metric,
    )

    for baseline in baselines:
        differences = pivot["AS-HMM-BO"] - pivot[baseline]

        paired_efficiency_rows.append({
            "Metric": metric,
            "Comparison": f"AS-HMM-BO minus {baseline}",
            "Seed 42": differences.loc[42],
            "Seed 123": differences.loc[123],
            "Seed 2026": differences.loc[2026],
            "Mean difference": differences.mean(),
            "AS-HMM-BO higher": int((differences > 0).sum()),
            "Equal": int((differences == 0).sum()),
        })

paired_efficiency = pd.DataFrame(paired_efficiency_rows)

display(paired_efficiency.round(6))

output_file = fixed16_dir / "fixed16_paired_efficiency_comparison.csv"
paired_efficiency.to_csv(output_file, index=False)

print("\nSaved:", output_file)
print(
    "Runtime: lower is better. "
    "AUC and Dice: higher is better."
)

# Diagnostic: inspect the current segmentation and training setup

In [ ]:
# Diagnostic: inspect the current segmentation and training setup

import inspect

for name in [
    "UNet",
    "train_one_epoch",
    "evaluate_model",
    "evaluate_candidate_across_folds",
]:
    obj = globals().get(name)

    print(f"\n{'=' * 20} {name} {'=' * 20}")

    if obj is None:
        print("Not found under this name.")
    else:
        try:
            print(inspect.getsource(obj))
        except (OSError, TypeError):
            print("Source unavailable:", obj)

In [ ]:
# Diagnostic — locate the actual model and training implementation

import inspect
import re

# 1. Inspect the candidate-training function
for name in [
    "evaluate_fixed_budget_candidate",
    "evaluate_candidate_across_folds",
]:
    obj = globals().get(name)
    print(f"\n{'=' * 25} {name} {'=' * 25}")

    if obj is not None:
        try:
            print(inspect.getsource(obj))
        except (OSError, TypeError):
            print("Python source unavailable.")

# 2. Find model, training, and loss definitions in notebook history
print("\n" + "=" * 25 + " NOTEBOOK DEFINITIONS " + "=" * 25)

patterns = re.compile(
    r"^\s*(class\s+\w*(UNet|Dataset|Loss|Model)\w*|"
    r"def\s+\w*(train|eval|loss|dice|dataset|model|candidate)\w*)",
    re.IGNORECASE | re.MULTILINE,
)

for cell_number, source in enumerate(In):
    if not source:
        continue

    matches = patterns.findall(source)
    if matches:
        definitions = [
            line.strip()
            for line in source.splitlines()
            if patterns.match(line)
        ]
        if definitions:
            print(f"\nCell {cell_number}:")
            for definition in definitions:
                print(" ", definition)

# 3. Show relevant configuration values
print("\n" + "=" * 25 + " CURRENT CONFIGURATION " + "=" * 25)

for name in [
    "COMPARISON_CONFIG",
    "SEARCH_CONFIG",
    "FIXED_SIZE_CONFIG",
]:
    if name in globals():
        print(f"\n{name}:")
        print(globals()[name])

# Diagnostic: inspect the existing segmentation implementation

In [ ]:

# Diagnostic: inspect the existing segmentation implementation
# No training is performed by this cell.

from pathlib import Path

cells_to_inspect = [13, 16, 19, 27, 35, 38]

for cell_number in cells_to_inspect:
    print("\n" + "=" * 30)
    print(f"NOTEBOOK INPUT CELL {cell_number}")
    print("=" * 30)

    if cell_number < len(In) and In[cell_number]:
        print(In[cell_number])
    else:
        print("No source available in notebook input history.")

print("\n" + "=" * 30)
print("DATA AND RUNTIME DETAILS")
print("=" * 30)

print("Project directory:", Path.cwd())
print("PyTorch version:", torch.__version__)
print("Device:", device if "device" in globals() else "not identified")

for name in [
    "BATCH_SIZE",
    "IMAGE_SIZE",
    "IMG_SIZE",
    "DATA_ROOT",
    "DATA_DIR",
    "TRAIN_DIR",
    "COMPARISON_CONFIG",
]:
    if name in globals():
        print(f"\n{name}:")
        print(globals()[name])


# Diagnostic — 30-epoch training curve on development folds only

In [ ]:
# Diagnostic — 30-epoch training curve on development folds only
# Does not modify previous optimizer-comparison results.

from pathlib import Path
import pandas as pd
import numpy as np

diagnostic_params = {
    "learning_rate": 2e-4,
    "weight_decay": 1e-4,
    "base_channels": 16,
    "loss_bce_weight": 0.5,
}

diagnostic_epochs = 30
diagnostic_rows = []

diagnostic_dir = (
    PROJECT_ROOT / "outputs" / "cross_validation"
)
diagnostic_dir.mkdir(parents=True, exist_ok=True)

for fold_id in [3, 4]:
    print(f"\nStarting 30-epoch diagnostic for fold {fold_id}...")

    result = train_search_trial(
        params=diagnostic_params,
        fold_id=fold_id,
        epochs=diagnostic_epochs,
        seed=42 + fold_id,
        return_history=True,
    )

    history = result["history"]

    if len(history) != diagnostic_epochs:
        raise RuntimeError(
            f"Fold {fold_id}: expected {diagnostic_epochs} epochs, "
            f"received {len(history)}."
        )

    for record in history:
        diagnostic_rows.append({
            "fold": fold_id,
            **diagnostic_params,
            **record,
        })

    curve = pd.DataFrame(history)

    print(f"Fold {fold_id} completed.")
    print(
        f"  Epoch 3 Dice:  "
        f"{curve.loc[curve['epoch'] == 3, 'mean_image_dice'].iloc[0]:.4f}"
    )
    print(
        f"  Epoch 10 Dice: "
        f"{curve.loc[curve['epoch'] == 10, 'mean_image_dice'].iloc[0]:.4f}"
    )
    print(
        f"  Epoch 20 Dice: "
        f"{curve.loc[curve['epoch'] == 20, 'mean_image_dice'].iloc[0]:.4f}"
    )
    print(
        f"  Epoch 30 Dice: "
        f"{curve.loc[curve['epoch'] == 30, 'mean_image_dice'].iloc[0]:.4f}"
    )
    print(
        f"  Best Dice: {result['mean_validation_image_dice']:.4f} "
        f"at epoch {result['best_epoch']}"
    )

diagnostic_df = pd.DataFrame(diagnostic_rows)

csv_path = diagnostic_dir / "training_curve_diagnostic_30_epochs.csv"
diagnostic_df.to_csv(csv_path, index=False)

display(
    diagnostic_df[
        diagnostic_df["epoch"].isin([1, 3, 5, 10, 15, 20, 25, 30])
    ][["fold", "epoch", "train_loss", "validation_loss",
       "mean_image_dice", "mean_image_iou",
       "mean_precision", "mean_recall"]]
)

print(f"\nSaved training curves to: {csv_path}")


In [ ]:

import inspect

print("Function signature:")
print(inspect.signature(train_search_trial))

print("\nCurrent implementation:")
print(inspect.getsource(train_search_trial))


In [ ]:

import random
import time
import copy
import numpy as np
import pandas as pd
import torch

from torch.utils.data import DataLoader
from pathlib import Path


def train_search_trial_early_stopping(
    params,
    fold_id,
    max_epochs=40,
    patience=6,
    min_delta=0.001,
    seed=42,
    checkpoint_dir=None,
):
    """
    Experimental early-stopping trainer.

    Uses development folds only.
    Selects checkpoints by validation mean image Dice.
    Stops after `patience` epochs without a sufficient improvement.
    Does not modify the original train_search_trial function.
    """

    if fold_id not in SEARCH_CONFIG["search_fold_ids"]:
        raise ValueError(
            f"Fold {fold_id} is not an authorized development fold."
        )

    required_params = {
        "learning_rate",
        "weight_decay",
        "base_channels",
        "loss_bce_weight",
    }
    missing = required_params - set(params)
    if missing:
        raise ValueError(f"Missing hyperparameters: {sorted(missing)}")

    if max_epochs < 1 or patience < 1:
        raise ValueError("max_epochs and patience must be positive.")

    if min_delta < 0:
        raise ValueError("min_delta must be non-negative.")

    if params["learning_rate"] <= 0 or params["weight_decay"] < 0:
        raise ValueError("Invalid learning rate or weight decay.")

    if params["base_channels"] not in SEARCH_CONFIG["base_channels"]:
        raise ValueError("Unsupported base_channels value.")

    if not 0.0 <= params["loss_bce_weight"] <= 1.0:
        raise ValueError("loss_bce_weight must be between 0 and 1.")

    fold_info = next(
        fold for fold in cv_folds if fold["fold"] == fold_id
    )

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    train_ds = GlaSSegmentationDataset(fold_info["train_manifest"])
    val_ds = GlaSSegmentationDataset(fold_info["val_manifest"])

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_dl = DataLoader(
        train_ds,
        batch_size=SEARCH_CONFIG["batch_size"],
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        generator=generator,
    )

    val_dl = DataLoader(
        val_ds,
        batch_size=SEARCH_CONFIG["batch_size"],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    positive_weight = calculate_positive_weight(
        train_ds,
        SEARCH_CONFIG["batch_size"],
    )

    model = UNet(
        base_channels=int(params["base_channels"])
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(params["learning_rate"]),
        weight_decay=float(params["weight_decay"]),
    )

    best_dice = -1.0
    best_epoch = None
    best_val_loss = None
    best_state = None
    epochs_without_improvement = 0
    history = []
    start_time = time.time()

    for epoch in range(1, max_epochs + 1):
        # Training
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        for batch in train_dl:
            images = batch["image"].to(DEVICE)
            targets = batch["mask"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)

            loss = mixed_segmentation_loss(
                logits,
                targets,
                positive_weight,
                float(params["loss_bce_weight"]),
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite training loss at fold={fold_id}, "
                    f"epoch={epoch}"
                )

            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * images.size(0)
            train_count += images.size(0)

        train_loss = train_loss_sum / max(train_count, 1)

        # Validation
        model.eval()
        val_loss_sum = 0.0
        val_count = 0
        dice_values = []
        iou_values = []
        precision_values = []
        recall_values = []

        with torch.no_grad():
            for batch in val_dl:
                images = batch["image"].to(DEVICE)
                targets = batch["mask"].to(DEVICE)

                logits = model(images)
                loss = mixed_segmentation_loss(
                    logits,
                    targets,
                    positive_weight,
                    float(params["loss_bce_weight"]),
                )

                val_loss_sum += loss.item() * images.size(0)
                val_count += images.size(0)

                predictions = torch.sigmoid(logits) >= 0.5
                truths = targets >= 0.5

                for i in range(images.size(0)):
                    pred = predictions[i]
                    truth = truths[i]

                    intersection = (pred & truth).sum().item()
                    pred_count = pred.sum().item()
                    truth_count = truth.sum().item()

                    dice = (
                        2.0 * intersection + 1e-7
                    ) / (pred_count + truth_count + 1e-7)

                    union = pred_count + truth_count - intersection

                    iou = (
                        intersection + 1e-7
                    ) / (union + 1e-7)

                    precision = (
                        intersection + 1e-7
                    ) / (pred_count + 1e-7)

                    recall = (
                        intersection + 1e-7
                    ) / (truth_count + 1e-7)

                    dice_values.append(float(dice))
                    iou_values.append(float(iou))
                    precision_values.append(float(precision))
                    recall_values.append(float(recall))

        if not dice_values:
            raise RuntimeError("Validation fold contains no images.")

        val_loss = val_loss_sum / max(val_count, 1)
        mean_dice = float(np.mean(dice_values))

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": val_loss,
            "mean_image_dice": mean_dice,
            "mean_image_iou": float(np.mean(iou_values)),
            "mean_precision": float(np.mean(precision_values)),
            "mean_recall": float(np.mean(recall_values)),
        }
        history.append(row)

        improved = mean_dice > best_dice + min_delta

        if improved:
            best_dice = mean_dice
            best_epoch = epoch
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"Fold {fold_id} | Epoch {epoch:02d}/{max_epochs} "
            f"| Dice {mean_dice:.4f} "
            f"| Best {best_dice:.4f} (epoch {best_epoch}) "
            f"| No-improvement epochs "
            f"{epochs_without_improvement}/{patience}"
        )

        if epochs_without_improvement >= patience:
            print(
                f"Early stopping at epoch {epoch}; "
                f"best checkpoint is epoch {best_epoch}."
            )
            break

    duration = time.time() - start_time

    # Save experimental outputs separately.
    if checkpoint_dir is None:
        checkpoint_dir = (
            PROJECT_ROOT
            / "outputs"
            / "cross_validation"
            / "early_stopping_experiment"
        )
    else:
        checkpoint_dir = Path(checkpoint_dir)

    checkpoint_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = checkpoint_dir / (
        f"fold_{fold_id}_seed_{seed}_best.pt"
    )
    history_path = checkpoint_dir / (
        f"fold_{fold_id}_seed_{seed}_history.csv"
    )

    torch.save(
        {
            "fold": int(fold_id),
            "seed": int(seed),
            "best_epoch": int(best_epoch),
            "best_validation_dice": float(best_dice),
            "best_validation_loss": float(best_val_loss),
            "params": dict(params),
            "model_state_dict": best_state,
        },
        checkpoint_path,
    )

    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)

    return {
        "fold": int(fold_id),
        "seed": int(seed),
        "best_epoch": int(best_epoch),
        "best_validation_dice": float(best_dice),
        "best_validation_loss": float(best_val_loss),
        "epochs_completed": len(history),
        "stopped_early": len(history) < max_epochs,
        "duration_seconds": float(duration),
        "checkpoint_path": str(checkpoint_path),
        "history_path": str(history_path),
        "history": history,
    }


In [ ]:

diagnostic_params = {
    "learning_rate": 2e-4,
    "weight_decay": 1e-4,
    "base_channels": 16,
    "loss_bce_weight": 0.5,
}

early_stop_test = train_search_trial_early_stopping(
    params=diagnostic_params,
    fold_id=3,
    max_epochs=40,
    patience=6,
    min_delta=0.001,
    seed=42,
)

print("\nOne-fold early-stopping test summary")
print({
    key: value
    for key, value in early_stop_test.items()
    if key != "history"
})


In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

early_stop_dir = (
    PROJECT_ROOT
    / "outputs"
    / "cross_validation"
    / "early_stopping_experiment"
)

history_path = early_stop_dir / "fold_3_seed_42_history.csv"
history_df = pd.read_csv(history_path)

display(
    history_df.loc[
        history_df["mean_image_dice"].idxmax()
    ].to_frame("Best epoch").T
)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(
    history_df["epoch"],
    history_df["mean_image_dice"],
    marker="o",
    markersize=3,
    label="Validation Dice",
)

ax.plot(
    history_df["epoch"],
    history_df["validation_loss"],
    marker=".",
    markersize=3,
    label="Validation loss",
)

ax.axvline(
    early_stop_test["best_epoch"],
    linestyle="--",
    label=f'Best Dice epoch: {early_stop_test["best_epoch"]}',
)

ax.set_xlabel("Epoch")
ax.set_title("Fold 3 — Early-stopping diagnostic")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print("Epochs completed:", len(history_df))
print(
    "Validation Dice standard deviation:",
    round(history_df["mean_image_dice"].std(), 4),
)
print(
    "Best-to-final Dice difference:",
    round(
        history_df["mean_image_dice"].max()
        - history_df["mean_image_dice"].iloc[-1],
        4,
    ),
)


In [ ]:

import pandas as pd
from pathlib import Path

history_path = (
    PROJECT_ROOT
    / "outputs"
    / "cross_validation"
    / "early_stopping_experiment"
    / "fold_3_seed_42_history.csv"
)

curve_df = pd.read_csv(history_path)

columns = [
    "epoch",
    "train_loss",
    "validation_loss",
    "mean_image_dice",
    "mean_image_iou",
    "mean_precision",
    "mean_recall",
]

display(
    curve_df.loc[
        curve_df["epoch"].isin([1, 3, 7, 9, 14, 16, 19, 22]),
        columns,
    ].round(4)
)

print("\nValidation Dice changes:")
curve_df["dice_change"] = curve_df["mean_image_dice"].diff()
display(
    curve_df.loc[
        curve_df["dice_change"].abs() >= 0.05,
        ["epoch", "mean_image_dice", "dice_change",
         "validation_loss", "mean_precision", "mean_recall"],
    ].round(4)
)


In [ ]:

history_df = pd.read_csv(
    PROJECT_ROOT
    / "outputs"
    / "cross_validation"
    / "early_stopping_experiment"
    / "fold_3_seed_42_history.csv"
)

coverage_df = history_df[
    [
        "epoch",
        "mean_image_dice",
        "mean_precision",
        "mean_recall",
        "validation_loss",
    ]
].copy()

# Precision/recall balance: a simple indicator of whether
# the model is becoming conservative or overpredicting foreground.
coverage_df["precision_recall_gap"] = (
    coverage_df["mean_precision"] - coverage_df["mean_recall"]
)

display(
    coverage_df.sort_values(
        "mean_image_dice",
        ascending=True,
    ).head(8).round(4)
)

print("\nLargest precision-recall gaps:")
display(
    coverage_df.reindex(
        coverage_df["precision_recall_gap"]
        .abs()
        .sort_values(ascending=False)
        .index
    ).head(8).round(4)
)


In [ ]:

import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

checkpoint_path = (
    PROJECT_ROOT
    / "outputs"
    / "cross_validation"
    / "early_stopping_experiment"
    / "fold_3_seed_42_best.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False,
)

fold_info = next(
    fold for fold in cv_folds if fold["fold"] == 3
)

val_ds = GlaSSegmentationDataset(fold_info["val_manifest"])
val_dl = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

model = UNet(
    base_channels=int(checkpoint["params"]["base_channels"])
).to(DEVICE)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

diagnostic_images = []

with torch.no_grad():
    for batch_index, batch in enumerate(val_dl):
        image = batch["image"].to(DEVICE)
        target = batch["mask"].to(DEVICE)

        logits = model(image)
        probabilities = torch.sigmoid(logits)
        prediction = probabilities >= 0.5
        truth = target >= 0.5

        intersection = (prediction & truth).sum().item()
        pred_pixels = prediction.sum().item()
        true_pixels = truth.sum().item()

        dice = (
            2 * intersection + 1e-7
        ) / (
            pred_pixels + true_pixels + 1e-7
        )

        diagnostic_images.append({
            "index": batch_index,
            "dice": float(dice),
            "predicted_foreground_fraction": (
                pred_pixels / prediction.numel()
            ),
            "true_foreground_fraction": (
                true_pixels / truth.numel()
            ),
            "image": batch["image"][0].cpu().permute(1, 2, 0).numpy(),
            "mask": batch["mask"][0, 0].cpu().numpy(),
            "prediction": prediction[0, 0].cpu().numpy(),
        })

coverage_summary = pd.DataFrame([
    {k: v for k, v in row.items()
     if k not in ["image", "mask", "prediction"]}
    for row in diagnostic_images
])

print("Checkpoint best epoch:", checkpoint["best_epoch"])
print("Checkpoint validation Dice:", checkpoint["best_validation_dice"])
display(coverage_summary.describe().round(4))

# Show the three lowest-Dice validation images.
worst_images = sorted(
    diagnostic_images,
    key=lambda row: row["dice"],
)[:3]

fig, axes = plt.subplots(
    len(worst_images), 3,
    figsize=(10, 3.5 * len(worst_images)),
)

if len(worst_images) == 1:
    axes = np.expand_dims(axes, axis=0)

for row_index, item in enumerate(worst_images):
    axes[row_index, 0].imshow(item["image"])
    axes[row_index, 0].set_title(f'Image {item["index"]}')

    axes[row_index, 1].imshow(item["mask"], cmap="gray", vmin=0, vmax=1)
    axes[row_index, 1].set_title("Ground truth")

    axes[row_index, 2].imshow(
        item["prediction"], cmap="gray", vmin=0, vmax=1
    )
    axes[row_index, 2].set_title(
        f'Prediction | Dice {item["dice"]:.3f}'
    )

    for ax in axes[row_index]:
        ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

coverage_details = pd.DataFrame([
    {
        "index": row["index"],
        "dice": row["dice"],
        "predicted_foreground_fraction": row[
            "predicted_foreground_fraction"
        ],
        "true_foreground_fraction": row[
            "true_foreground_fraction"
        ],
        "foreground_fraction_difference": (
            row["predicted_foreground_fraction"]
            - row["true_foreground_fraction"]
        ),
    }
    for row in diagnostic_images
])

print("Lowest-Dice images:")
display(
    coverage_details.sort_values("dice").head(6).round(4)
)

print("Largest foreground coverage mismatches:")
display(
    coverage_details.assign(
        absolute_coverage_difference=lambda df:
            df["foreground_fraction_difference"].abs()
    ).sort_values(
        "absolute_coverage_difference", ascending=False
    ).head(6).round(4)
)


In [ ]:

worst_images = sorted(
    diagnostic_images, key=lambda row: row["dice"]
)[:3]

fig, axes = plt.subplots(3, 3, figsize=(12, 12))

for r, item in enumerate(worst_images):
    axes[r, 0].imshow(item["image"])
    axes[r, 0].set_title(f'Image {item["index"]}')

    axes[r, 1].imshow(item["mask"], cmap="gray", vmin=0, vmax=1)
    axes[r, 1].set_title("Ground truth")

    axes[r, 2].imshow(
        item["prediction"], cmap="gray", vmin=0, vmax=1
    )
    axes[r, 2].set_title(f'Prediction, Dice={item["dice"]:.3f}')

    for ax in axes[r]:
        ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

import numpy as np
import pandas as pd
import torch

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
threshold_rows = []

model.eval()

with torch.no_grad():
    for threshold in thresholds:
        image_dice = []

        for batch in val_dl:
            images = batch["image"].to(DEVICE)
            targets = batch["mask"].to(DEVICE) >= 0.5

            probabilities = torch.sigmoid(model(images))
            predictions = probabilities >= threshold

            for i in range(images.size(0)):
                pred = predictions[i]
                truth = targets[i]

                intersection = (pred & truth).sum().item()
                pred_count = pred.sum().item()
                true_count = truth.sum().item()

                dice = (
                    2.0 * intersection + 1e-7
                ) / (
                    pred_count + true_count + 1e-7
                )

                image_dice.append(float(dice))

        threshold_rows.append({
            "threshold": threshold,
            "mean_image_dice": float(np.mean(image_dice)),
        })

threshold_df = pd.DataFrame(threshold_rows)

display(
    threshold_df.sort_values(
        "mean_image_dice", ascending=False
    ).round(4)
)

best_threshold_row = threshold_df.loc[
    threshold_df["mean_image_dice"].idxmax()
]

print(
    "\nBest diagnostic threshold:",
    best_threshold_row["threshold"],
)
print(
    "Best diagnostic Dice:",
    round(best_threshold_row["mean_image_dice"], 4),
)


In [ ]:

import inspect

objects_to_check = {
    "GlaSSegmentationDataset": GlaSSegmentationDataset,
    "train_search_trial_early_stopping": train_search_trial_early_stopping,
}

for name, obj in objects_to_check.items():
    print(f"\n{'=' * 65}")
    print(name)
    print("=" * 65)
    print("Type:", type(obj))
    print("Module:", getattr(obj, "__module__", None))
    print("Signature:", inspect.signature(obj))

    try:
        print(inspect.getsource(obj))
    except (OSError, TypeError) as exc:
        print("Source unavailable:", repr(exc))
        print("Code names:", getattr(
            getattr(obj, "__code__", None),
            "co_names",
            "Not available"
        ))
        print("First line:", getattr(
            getattr(obj, "__code__", None),
            "co_firstlineno",
            "Not available"
        ))


In [ ]:

# Inspect the existing dataset without changing it.

fold_info = next(
    fold for fold in cv_folds if fold["fold"] == 3
)

inspection_ds = GlaSSegmentationDataset(
    fold_info["train_manifest"]
)

sample = inspection_ds[0]

print("Sample type:", type(sample))
print("Sample keys:", list(sample.keys()))

for key, value in sample.items():
    if torch.is_tensor(value):
        print(
            f"{key}: shape={tuple(value.shape)}, "
            f"dtype={value.dtype}, "
            f"min={value.min().item():.4f}, "
            f"max={value.max().item():.4f}"
        )
    else:
        print(f"{key}: type={type(value).__name__}, value={value}")

print("\nTraining manifest columns:")
print(list(fold_info["train_manifest"].columns))

print("\nValidation sample count:")
print(len(fold_info["val_manifest"]))


In [ ]:

import torch
from torch.utils.data import Dataset
import random

class GeometricAugmentationDataset(Dataset):
    """
    Applies synchronized geometric augmentation to image, binary mask,
    and instance mask. The underlying dataset remains unchanged.
    """

    def __init__(self, base_dataset, augment=False, seed=42):
        self.base_dataset = base_dataset
        self.augment = augment
        self.seed = seed

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        sample = self.base_dataset[index]

        # Clone tensors so the original sample cannot be modified in-place.
        result = {
            key: value.clone() if torch.is_tensor(value) else value
            for key, value in sample.items()
        }

        if not self.augment:
            return result

        # Deterministic per-sample transformations for reproducibility.
        rng = random.Random(self.seed + index)

        # Horizontal flip.
        if rng.random() < 0.5:
            for key in ("image", "mask", "instance_mask"):
                result[key] = torch.flip(result[key], dims=[-1])

        # Vertical flip.
        if rng.random() < 0.5:
            for key in ("image", "mask", "instance_mask"):
                result[key] = torch.flip(result[key], dims=[-2])

        # Rotation by 0, 90, 180, or 270 degrees.
        k = rng.randrange(4)
        if k:
            for key in ("image", "mask", "instance_mask"):
                result[key] = torch.rot90(
                    result[key], k=k, dims=[-2, -1]
                )

        return result


# Build a separate augmented view of the training data.
augmentation_base_ds = GlaSSegmentationDataset(
    fold_info["train_manifest"]
)

augmented_check_ds = GeometricAugmentationDataset(
    augmentation_base_ds,
    augment=True,
    seed=42,
)

original_sample = augmentation_base_ds[0]
augmented_sample = augmented_check_ds[0]

print("Original image shape:", tuple(original_sample["image"].shape))
print("Augmented image shape:", tuple(augmented_sample["image"].shape))
print("Original mask shape:", tuple(original_sample["mask"].shape))
print("Augmented mask shape:", tuple(augmented_sample["mask"].shape))

print(
    "Binary mask values preserved:",
    set(torch.unique(augmented_sample["mask"]).tolist()).issubset({0.0, 1.0})
)
print(
    "Instance labels preserved:",
    torch.equal(
        torch.sort(torch.unique(original_sample["instance_mask"])).values,
        torch.sort(torch.unique(augmented_sample["instance_mask"])).values,
    )
)

print("\nOriginal dataset length:", len(augmentation_base_ds))
print("Augmented dataset length:", len(augmented_check_ds))


In [ ]:

import copy
import random
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader, Dataset


class RandomGeometricAugmentationDataset(Dataset):
    """Random synchronized geometric augmentation on every sample access."""

    def __init__(self, base_dataset, seed=42):
        self.base_dataset = base_dataset
        self.rng = random.Random(seed)

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        sample = self.base_dataset[index]

        result = {
            key: value.clone() if torch.is_tensor(value) else value
            for key, value in sample.items()
        }

        if self.rng.random() < 0.5:
            for key in ("image", "mask", "instance_mask"):
                result[key] = torch.flip(result[key], dims=[-1])

        if self.rng.random() < 0.5:
            for key in ("image", "mask", "instance_mask"):
                result[key] = torch.flip(result[key], dims=[-2])

        k = self.rng.randrange(4)
        if k:
            for key in ("image", "mask", "instance_mask"):
                result[key] = torch.rot90(
                    result[key], k=k, dims=[-2, -1]
                )

        return result


def train_search_trial_augmented(
    params,
    fold_id,
    max_epochs=40,
    patience=6,
    min_delta=0.001,
    seed=42,
    checkpoint_dir=None,
):
    """Separate experimental trainer; validates on unaugmented images."""

    if fold_id not in SEARCH_CONFIG["search_fold_ids"]:
        raise ValueError(f"Unauthorized development fold: {fold_id}")

    required = {
        "learning_rate", "weight_decay",
        "base_channels", "loss_bce_weight"
    }
    missing = required - set(params)
    if missing:
        raise ValueError(f"Missing parameters: {sorted(missing)}")

    if max_epochs < 1 or patience < 1 or min_delta < 0:
        raise ValueError("Invalid early-stopping configuration.")

    if params["learning_rate"] <= 0 or params["weight_decay"] < 0:
        raise ValueError("Invalid learning rate or weight decay.")

    if params["base_channels"] not in SEARCH_CONFIG["base_channels"]:
        raise ValueError("Unsupported base_channels.")

    if not 0 <= params["loss_bce_weight"] <= 1:
        raise ValueError("loss_bce_weight must be between 0 and 1.")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    fold_info = next(f for f in cv_folds if f["fold"] == fold_id)

    base_train_ds = GlaSSegmentationDataset(fold_info["train_manifest"])
    train_ds = RandomGeometricAugmentationDataset(
        base_train_ds, seed=seed
    )
    val_ds = GlaSSegmentationDataset(fold_info["val_manifest"])

    generator = torch.Generator().manual_seed(seed)

    train_dl = DataLoader(
        train_ds,
        batch_size=SEARCH_CONFIG["batch_size"],
        shuffle=True,
        num_workers=0,
        pin_memory=PIN_MEMORY,
        generator=generator,
    )
    val_dl = DataLoader(
        val_ds,
        batch_size=SEARCH_CONFIG["batch_size"],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    positive_weight = calculate_positive_weight(
        base_train_ds, SEARCH_CONFIG["batch_size"]
    )

    model = UNet(base_channels=int(params["base_channels"])).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(params["learning_rate"]),
        weight_decay=float(params["weight_decay"]),
    )

    best_dice = -1.0
    best_epoch = None
    best_val_loss = None
    best_state = None
    no_improvement = 0
    history = []
    start_time = time.time()

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        for batch in train_dl:
            images = batch["image"].to(DEVICE)
            targets = batch["mask"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = mixed_segmentation_loss(
                logits,
                targets,
                positive_weight,
                float(params["loss_bce_weight"]),
            )

            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite loss at epoch {epoch}")

            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * images.size(0)
            train_count += images.size(0)

        train_loss = train_loss_sum / max(train_count, 1)

        model.eval()
        val_loss_sum = 0.0
        val_count = 0
        dice_values, iou_values = [], []
        precision_values, recall_values = [], []

        with torch.no_grad():
            for batch in val_dl:
                images = batch["image"].to(DEVICE)
                targets = batch["mask"].to(DEVICE)
                logits = model(images)

                loss = mixed_segmentation_loss(
                    logits,
                    targets,
                    positive_weight,
                    float(params["loss_bce_weight"]),
                )
                val_loss_sum += loss.item() * images.size(0)
                val_count += images.size(0)

                predictions = torch.sigmoid(logits) >= 0.5
                truths = targets >= 0.5

                for i in range(images.size(0)):
                    pred, truth = predictions[i], truths[i]
                    intersection = (pred & truth).sum().item()
                    pred_count = pred.sum().item()
                    true_count = truth.sum().item()
                    union = pred_count + true_count - intersection

                    dice_values.append(
                        (2 * intersection + 1e-7)
                        / (pred_count + true_count + 1e-7)
                    )
                    iou_values.append(
                        (intersection + 1e-7) / (union + 1e-7)
                    )
                    precision_values.append(
                        (intersection + 1e-7) / (pred_count + 1e-7)
                    )
                    recall_values.append(
                        (intersection + 1e-7) / (true_count + 1e-7)
                    )

        if not dice_values:
            raise RuntimeError("Validation fold contains no images.")

        val_loss = val_loss_sum / max(val_count, 1)
        mean_dice = float(np.mean(dice_values))

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": val_loss,
            "mean_image_dice": mean_dice,
            "mean_image_iou": float(np.mean(iou_values)),
            "mean_precision": float(np.mean(precision_values)),
            "mean_recall": float(np.mean(recall_values)),
        })

        if mean_dice > best_dice + min_delta:
            best_dice = mean_dice
            best_epoch = epoch
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            no_improvement = 0
        else:
            no_improvement += 1

        print(
            f"Augmented | Fold {fold_id} | Epoch {epoch:02d}/{max_epochs} "
            f"| Dice {mean_dice:.4f} | Best {best_dice:.4f} "
            f"(epoch {best_epoch}) | Patience {no_improvement}/{patience}"
        )

        if no_improvement >= patience:
            print(f"Early stopping; best epoch = {best_epoch}")
            break

    duration = time.time() - start_time

    if checkpoint_dir is None:
        checkpoint_dir = (
            PROJECT_ROOT / "outputs" / "cross_validation"
            / "augmentation_experiment"
        )
    else:
        checkpoint_dir = Path(checkpoint_dir)

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / f"fold_{fold_id}_seed_{seed}_best.pt"
    history_path = checkpoint_dir / f"fold_{fold_id}_seed_{seed}_history.csv"

    torch.save({
        "fold": int(fold_id),
        "seed": int(seed),
        "best_epoch": int(best_epoch),
        "best_validation_dice": float(best_dice),
        "best_validation_loss": float(best_val_loss),
        "params": dict(params),
        "augmentation": "synchronized random flips and 90-degree rotations",
        "model_state_dict": best_state,
    }, checkpoint_path)

    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)

    return {
        "fold": int(fold_id),
        "seed": int(seed),
        "best_epoch": int(best_epoch),
        "best_validation_dice": float(best_dice),
        "best_validation_loss": float(best_val_loss),
        "epochs_completed": len(history),
        "stopped_early": len(history) < max_epochs,
        "duration_seconds": float(duration),
        "checkpoint_path": str(checkpoint_path),
        "history_path": str(history_path),
        "history": history,
    }


print("Augmented trainer defined successfully.")
print("Original trainer remains:", train_search_trial_early_stopping.__name__)


In [ ]:

augmentation_params = {
    "learning_rate": 2e-4,
    "weight_decay": 1e-4,
    "base_channels": 16,
    "loss_bce_weight": 0.5,
}

augmented_fold3_seed42 = train_search_trial_augmented(
    params=augmentation_params,
    fold_id=3,
    max_epochs=40,
    patience=6,
    min_delta=0.001,
    seed=42,
)

print("\nAugmentation experiment result:")
for key, value in augmented_fold3_seed42.items():
    if key != "history":
        print(f"{key}: {value}")


In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

cv_dir = PROJECT_ROOT / "outputs" / "cross_validation"

baseline_path = (
    cv_dir / "early_stopping_experiment"
    / "fold_3_seed_42_history.csv"
)
augmented_path = (
    cv_dir / "augmentation_experiment"
    / "fold_3_seed_42_history.csv"
)

baseline_df = pd.read_csv(baseline_path)
augmented_df = pd.read_csv(augmented_path)

print("Baseline history:", baseline_df.shape)
print("Augmented history:", augmented_df.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(
    baseline_df["epoch"],
    baseline_df["mean_image_dice"],
    marker="o",
    markersize=3,
    label="Original",
)
axes[0].plot(
    augmented_df["epoch"],
    augmented_df["mean_image_dice"],
    marker="o",
    markersize=3,
    label="Geometric augmentation",
)
axes[0].set_title("Validation Dice by epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Mean image Dice")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(
    baseline_df["epoch"],
    baseline_df["validation_loss"],
    marker="o",
    markersize=3,
    label="Original",
)
axes[1].plot(
    augmented_df["epoch"],
    augmented_df["validation_loss"],
    marker="o",
    markersize=3,
    label="Geometric augmentation",
)
axes[1].set_title("Validation loss by epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nBaseline best epoch:")
print(baseline_df.loc[baseline_df["mean_image_dice"].idxmax()].round(4))

print("\nAugmented best epoch:")
print(augmented_df.loc[augmented_df["mean_image_dice"].idxmax()].round(4))


In [ ]:

import copy
import random
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from torch.utils.data import DataLoader


def train_search_trial_scheduler(
    params,
    fold_id,
    max_epochs=40,
    patience=6,
    min_delta=0.001,
    seed=42,
    scheduler_patience=3,
    scheduler_factor=0.5,
    min_learning_rate=1e-6,
    checkpoint_dir=None,
):
    """
    Unaugmented training with ReduceLROnPlateau.

    Scheduler monitors validation loss.
    Early stopping and checkpoint selection monitor validation Dice.
    Existing trainers and outputs remain unchanged.
    """

    if fold_id not in SEARCH_CONFIG["search_fold_ids"]:
        raise ValueError(f"Unauthorized development fold: {fold_id}")

    required = {
        "learning_rate", "weight_decay",
        "base_channels", "loss_bce_weight"
    }
    missing = required - set(params)
    if missing:
        raise ValueError(f"Missing parameters: {sorted(missing)}")

    if max_epochs < 1 or patience < 1 or scheduler_patience < 0:
        raise ValueError("Invalid epoch or patience settings.")
    if min_delta < 0:
        raise ValueError("min_delta must be non-negative.")
    if not 0 < scheduler_factor < 1:
        raise ValueError("scheduler_factor must be between 0 and 1.")
    if min_learning_rate <= 0:
        raise ValueError("min_learning_rate must be positive.")
    if params["learning_rate"] <= 0 or params["weight_decay"] < 0:
        raise ValueError("Invalid learning rate or weight decay.")
    if params["base_channels"] not in SEARCH_CONFIG["base_channels"]:
        raise ValueError("Unsupported base_channels.")
    if not 0 <= params["loss_bce_weight"] <= 1:
        raise ValueError("loss_bce_weight must be between 0 and 1.")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    fold_info = next(f for f in cv_folds if f["fold"] == fold_id)

    train_ds = GlaSSegmentationDataset(fold_info["train_manifest"])
    val_ds = GlaSSegmentationDataset(fold_info["val_manifest"])

    generator = torch.Generator().manual_seed(seed)

    train_dl = DataLoader(
        train_ds,
        batch_size=SEARCH_CONFIG["batch_size"],
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        generator=generator,
    )
    val_dl = DataLoader(
        val_ds,
        batch_size=SEARCH_CONFIG["batch_size"],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    positive_weight = calculate_positive_weight(
        train_ds, SEARCH_CONFIG["batch_size"]
    )

    model = UNet(base_channels=int(params["base_channels"])).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=float(params["learning_rate"]),
        weight_decay=float(params["weight_decay"]),
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=scheduler_factor,
        patience=scheduler_patience,
        threshold=1e-4,
        threshold_mode="rel",
        min_lr=min_learning_rate,
    )

    best_dice = -1.0
    best_epoch = None
    best_val_loss = None
    best_state = None
    no_improvement = 0
    history = []
    start_time = time.time()

    for epoch in range(1, max_epochs + 1):
        # Training: original dataset, no augmentation.
        model.train()
        train_loss_sum = 0.0
        train_count = 0

        for batch in train_dl:
            images = batch["image"].to(DEVICE)
            targets = batch["mask"].to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)

            loss = mixed_segmentation_loss(
                logits,
                targets,
                positive_weight,
                float(params["loss_bce_weight"]),
            )

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite training loss at epoch {epoch}"
                )

            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * images.size(0)
            train_count += images.size(0)

        train_loss = train_loss_sum / max(train_count, 1)

        # Validation: no augmentation and no gradient updates.
        model.eval()
        val_loss_sum = 0.0
        val_count = 0
        dice_values = []
        iou_values = []
        precision_values = []
        recall_values = []

        with torch.no_grad():
            for batch in val_dl:
                images = batch["image"].to(DEVICE)
                targets = batch["mask"].to(DEVICE)

                logits = model(images)
                loss = mixed_segmentation_loss(
                    logits,
                    targets,
                    positive_weight,
                    float(params["loss_bce_weight"]),
                )

                val_loss_sum += loss.item() * images.size(0)
                val_count += images.size(0)

                predictions = torch.sigmoid(logits) >= 0.5
                truths = targets >= 0.5

                for i in range(images.size(0)):
                    pred = predictions[i]
                    truth = truths[i]

                    intersection = (pred & truth).sum().item()
                    pred_count = pred.sum().item()
                    true_count = truth.sum().item()
                    union = pred_count + true_count - intersection

                    dice_values.append(
                        (2 * intersection + 1e-7)
                        / (pred_count + true_count + 1e-7)
                    )
                    iou_values.append(
                        (intersection + 1e-7) / (union + 1e-7)
                    )
                    precision_values.append(
                        (intersection + 1e-7) / (pred_count + 1e-7)
                    )
                    recall_values.append(
                        (intersection + 1e-7) / (true_count + 1e-7)
                    )

        if not dice_values:
            raise RuntimeError("Validation fold contains no images.")

        val_loss = val_loss_sum / max(val_count, 1)

        # Scheduler responds to validation loss.
        scheduler.step(val_loss)
        current_lr = float(optimizer.param_groups[0]["lr"])

        mean_dice = float(np.mean(dice_values))

        history.append({
            "epoch": epoch,
            "learning_rate": current_lr,
            "train_loss": train_loss,
            "validation_loss": val_loss,
            "mean_image_dice": mean_dice,
            "mean_image_iou": float(np.mean(iou_values)),
            "mean_precision": float(np.mean(precision_values)),
            "mean_recall": float(np.mean(recall_values)),
        })

        if mean_dice > best_dice + min_delta:
            best_dice = mean_dice
            best_epoch = epoch
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            no_improvement = 0
        else:
            no_improvement += 1

        print(
            f"Scheduler | Fold {fold_id} | Epoch {epoch:02d}/{max_epochs} "
            f"| Dice {mean_dice:.4f} | Val loss {val_loss:.4f} "
            f"| LR {current_lr:.2e} | Best {best_dice:.4f} "
            f"(epoch {best_epoch}) | Patience {no_improvement}/{patience}"
        )

        if no_improvement >= patience:
            print(f"Early stopping; best epoch = {best_epoch}")
            break

    duration = time.time() - start_time

    if checkpoint_dir is None:
        checkpoint_dir = (
            PROJECT_ROOT / "outputs" / "cross_validation"
            / "scheduler_experiment"
        )
    else:
        checkpoint_dir = Path(checkpoint_dir)

    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_dir / f"fold_{fold_id}_seed_{seed}_best.pt"
    history_path = checkpoint_dir / f"fold_{fold_id}_seed_{seed}_history.csv"

    torch.save({
        "fold": int(fold_id),
        "seed": int(seed),
        "best_epoch": int(best_epoch),
        "best_validation_dice": float(best_dice),
        "best_validation_loss": float(best_val_loss),
        "params": dict(params),
        "scheduler": {
            "name": "ReduceLROnPlateau",
            "monitor": "validation_loss",
            "factor": scheduler_factor,
            "patience": scheduler_patience,
            "minimum_learning_rate": min_learning_rate,
        },
        "model_state_dict": best_state,
    }, checkpoint_path)

    history_df = pd.DataFrame(history)
    history_df.to_csv(history_path, index=False)

    return {
        "fold": int(fold_id),
        "seed": int(seed),
        "best_epoch": int(best_epoch),
        "best_validation_dice": float(best_dice),
        "best_validation_loss": float(best_val_loss),
        "epochs_completed": len(history),
        "stopped_early": len(history) < max_epochs,
        "duration_seconds": float(duration),
        "checkpoint_path": str(checkpoint_path),
        "history_path": str(history_path),
        "history": history,
    }


print("Scheduler trainer defined successfully.")


In [ ]:

scheduler_params = {
    "learning_rate": 2e-4,
    "weight_decay": 1e-4,
    "base_channels": 16,
    "loss_bce_weight": 0.5,
}

scheduler_fold3_seed42 = train_search_trial_scheduler(
    params=scheduler_params,
    fold_id=3,
    max_epochs=40,
    patience=6,
    min_delta=0.001,
    seed=42,
    scheduler_patience=3,
    scheduler_factor=0.5,
)

print("\nScheduler experiment result:")
for key, value in scheduler_fold3_seed42.items():
    if key != "history":
        print(f"{key}: {value}")


In [ ]:

baseline_fold4_seed42 = train_search_trial_early_stopping(
    params={
        "learning_rate": 2e-4,
        "weight_decay": 1e-4,
        "base_channels": 16,
        "loss_bce_weight": 0.5,
    },
    fold_id=4,
    max_epochs=40,
    patience=6,
    min_delta=0.001,
    seed=42,
)

print("\nFold 4 baseline result:")
for key, value in baseline_fold4_seed42.items():
    if key != "history":
        print(f"{key}: {value}")


In [ ]:
from pathlib import Path
import pandas as pd
import torch

experiment_dir = (
    PROJECT_ROOT
    / "outputs"
    / "cross_validation"
    / "early_stopping_experiment"
)

for fold_id in (3, 4):
    history_path = experiment_dir / f"fold_{fold_id}_seed_42_history.csv"
    checkpoint_path = experiment_dir / f"fold_{fold_id}_seed_42_best.pt"

    print(f"\n{'=' * 60}")
    print(f"FOLD {fold_id}")
    print("=" * 60)

    history = pd.read_csv(history_path)
    print("History columns:", history.columns.tolist())
    print("\nLast 8 epochs:")
    print(history.tail(8).to_string(index=False))

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    print("\nCheckpoint type:", type(checkpoint).__name__)

    if isinstance(checkpoint, dict):
        print("Checkpoint keys:", list(checkpoint.keys()))

        for key, value in checkpoint.items():
            if isinstance(value, dict):
                print(f"  {key}: dictionary with {len(value)} entries")
            elif isinstance(value, torch.Tensor):
                print(f"  {key}: tensor with shape {tuple(value.shape)}")
            else:
                print(f"  {key}: {value}")

In [ ]:
import inspect
from pathlib import Path

print("Dataset constructor:")
print(inspect.signature(GlaSSegmentationDataset))

print("\nU-Net constructor:")
print(inspect.signature(UNet))

print("\nDevice:", DEVICE)
print("Project root:", PROJECT_ROOT)

for fold_id in (3, 4):
    print(f"\n{'=' * 55}")
    print(f"FOLD {fold_id}")
    print("=" * 55)

    fold_info = cv_folds[fold_id]

    if isinstance(fold_info, dict):
        print("Metadata keys:", list(fold_info.keys()))

        for key, value in fold_info.items():
            if isinstance(value, (list, tuple)):
                print(f"{key}: length={len(value)}")
                if value and isinstance(value[0], dict):
                    print("  First item keys:", list(value[0].keys()))
            elif isinstance(value, pd.DataFrame):
                print(f"{key}: DataFrame shape={value.shape}")
                print("  Columns:", value.columns.tolist())
            else:
                print(f"{key}: {value}")
    else:
        print("Metadata type:", type(fold_info).__name__)
        print(fold_info)

    checkpoint_path = (
        PROJECT_ROOT
        / "outputs"
        / "cross_validation"
        / "early_stopping_experiment"
        / f"fold_{fold_id}_seed_42_best.pt"
    )

    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    print("\nSaved model parameters:", checkpoint["params"])
    print("Saved best epoch:", checkpoint["best_epoch"])
    print("Saved best Dice:", checkpoint["best_validation_dice"])

In [ ]:
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import torch

diagnostic_rows = []

experiment_dir = (
    PROJECT_ROOT
    / "outputs"
    / "cross_validation"
    / "early_stopping_experiment"
)

for fold_id in (3, 4):
    fold_info = cv_folds[fold_id]
    val_manifest = fold_info["val_manifest"].reset_index(drop=True)

    checkpoint_path = (
        experiment_dir / f"fold_{fold_id}_seed_42_best.pt"
    )
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model = UNet(
        in_channels=3,
        out_channels=1,
        base_channels=checkpoint["params"]["base_channels"],
    ).to(DEVICE)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    val_dataset = GlaSSegmentationDataset(val_manifest)
    val_loader = DataLoader(
        val_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
    )

    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(DEVICE)
            masks = batch["mask"].to(DEVICE)

            logits = model(images)
            probabilities = torch.sigmoid(logits)
            predictions = (probabilities >= 0.5).float()

            pred = predictions[0, 0].cpu().numpy().astype(bool)
            truth = masks[0, 0].cpu().numpy() >= 0.5

            intersection = np.logical_and(pred, truth).sum()
            pred_area = pred.sum()
            true_area = truth.sum()
            union = np.logical_or(pred, truth).sum()

            dice = (
                2.0 * intersection / (pred_area + true_area)
                if pred_area + true_area else 1.0
            )
            iou = intersection / union if union else 1.0
            precision = (
                intersection / pred_area if pred_area else 0.0
            )
            recall = (
                intersection / true_area if true_area else 0.0
            )

            sample_id = batch["sample_id"][0]

            diagnostic_rows.append({
                "fold": fold_id,
                "sample_id": sample_id,
                "dice": dice,
                "iou": iou,
                "precision": precision,
                "recall": recall,
                "pred_foreground_fraction": pred.mean(),
                "true_foreground_fraction": truth.mean(),
            })

diagnostics_df = pd.DataFrame(diagnostic_rows)

print("Per-image results:")
print(
    diagnostics_df.sort_values(["fold", "dice"]).to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)

print("\nSummary by fold:")
print(
    diagnostics_df.groupby("fold")[
        [
            "dice",
            "iou",
            "precision",
            "recall",
            "pred_foreground_fraction",
            "true_foreground_fraction",
        ]
    ].agg(["mean", "std", "min", "median", "max"]).round(4)
)

In [ ]:
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import torch

checkpoint_dir = (
    PROJECT_ROOT / "outputs" / "cross_validation"
    / "early_stopping_experiment"
)

cross_check_rows = []

for checkpoint_fold in (3, 4):
    checkpoint_path = (
        checkpoint_dir / f"fold_{checkpoint_fold}_seed_42_best.pt"
    )
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model = UNet(
        in_channels=3,
        out_channels=1,
        base_channels=checkpoint["params"]["base_channels"],
    ).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    for manifest_fold in (3, 4):
        val_manifest = cv_folds[manifest_fold]["val_manifest"].reset_index(
            drop=True
        )
        dataset = GlaSSegmentationDataset(val_manifest)
        loader = DataLoader(
            dataset, batch_size=1, shuffle=False, num_workers=0
        )

        image_dices = []

        with torch.no_grad():
            for batch in loader:
                image = batch["image"].to(DEVICE)
                mask = batch["mask"].to(DEVICE)

                probability = torch.sigmoid(model(image))
                prediction = probability >= 0.5
                truth = mask >= 0.5

                intersection = (prediction & truth).sum().item()
                pred_sum = prediction.sum().item()
                truth_sum = truth.sum().item()

                dice = (
                    2 * intersection / (pred_sum + truth_sum)
                    if pred_sum + truth_sum > 0 else 1.0
                )
                image_dices.append(dice)

        cross_check_rows.append({
            "checkpoint_fold": checkpoint_fold,
            "checkpoint_recorded_best_dice":
                checkpoint["best_validation_dice"],
            "manifest_key": manifest_fold,
            "manifest_metadata_fold":
                cv_folds[manifest_fold]["fold"],
            "num_validation_images": len(image_dices),
            "recomputed_mean_dice": np.mean(image_dices),
        })

cross_check_df = pd.DataFrame(cross_check_rows)
print(cross_check_df.round(5).to_string(index=False))

In [ ]:
for fold_id in (3, 4):
    info = cv_folds[fold_id]
    val_manifest = info["val_manifest"]

    print(f"\n{'=' * 55}")
    print(f"cv_folds key: {fold_id}")
    print(f"Metadata fold: {info['fold']}")
    print(f"Validation images: {len(val_manifest)}")
    print("Sample IDs:")
    print(val_manifest["sample_id"].tolist())

    if "patient_id" in val_manifest.columns:
        print("\nPatient IDs:")
        print(sorted(val_manifest["patient_id"].dropna().unique().tolist()))

In [ ]:
import inspect

print(inspect.getsource(train_search_trial_early_stopping))
print("Trainer name:", train_search_trial_early_stopping.__name__)
print("Trainer module:", train_search_trial_early_stopping.__module__)
print("Available global names used by trainer:")

code = train_search_trial_early_stopping.__code__
print(code.co_names)

In [ ]:
fold_info = next(
    fold for fold in cv_folds if fold["fold"] == fold_id
)

In [ ]:
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import torch

checkpoint_dir = (
    PROJECT_ROOT
    / "outputs"
    / "cross_validation"
    / "early_stopping_experiment"
)

verification_rows = []

for fold_id in (3, 4):
    # Match the trainer's exact lookup logic.
    fold_info = next(
        fold for fold in cv_folds if fold["fold"] == fold_id
    )
    val_manifest = fold_info["val_manifest"].reset_index(drop=True)

    checkpoint_path = checkpoint_dir / f"fold_{fold_id}_seed_42_best.pt"
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model = UNet(
        in_channels=3,
        out_channels=1,
        base_channels=checkpoint["params"]["base_channels"],
    ).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    loader = DataLoader(
        GlaSSegmentationDataset(val_manifest),
        batch_size=SEARCH_CONFIG["batch_size"],
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )

    dice_values = []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(DEVICE)
            targets = batch["mask"].to(DEVICE)

            predictions = torch.sigmoid(model(images)) >= 0.5
            truths = targets >= 0.5

            for i in range(images.size(0)):
                pred = predictions[i]
                truth = truths[i]
                intersection = (pred & truth).sum().item()
                pred_count = pred.sum().item()
                truth_count = truth.sum().item()

                dice = (
                    2.0 * intersection + 1e-7
                ) / (pred_count + truth_count + 1e-7)

                dice_values.append(float(dice))

    recomputed = float(np.mean(dice_values))

    verification_rows.append({
        "checkpoint_fold": fold_id,
        "metadata_fold": fold_info["fold"],
        "validation_images": len(dice_values),
        "saved_best_dice": checkpoint["best_validation_dice"],
        "recomputed_dice": recomputed,
        "absolute_difference": abs(
            recomputed - checkpoint["best_validation_dice"]
        ),
    })

print(pd.DataFrame(verification_rows).round(6).to_string(index=False))

In [ ]:
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import torch

checkpoint_dir = (
    PROJECT_ROOT / "outputs" / "cross_validation"
    / "early_stopping_experiment"
)

corrected_rows = []

for fold_id in (3, 4):
    fold_info = next(
        fold for fold in cv_folds if fold["fold"] == fold_id
    )
    val_manifest = fold_info["val_manifest"].reset_index(drop=True)

    checkpoint = torch.load(
        checkpoint_dir / f"fold_{fold_id}_seed_42_best.pt",
        map_location=DEVICE,
        weights_only=False,
    )

    model = UNet(
        in_channels=3,
        out_channels=1,
        base_channels=checkpoint["params"]["base_channels"],
    ).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    loader = DataLoader(
        GlaSSegmentationDataset(val_manifest),
        batch_size=1,
        shuffle=False,
        num_workers=0,
    )

    with torch.no_grad():
        for batch in loader:
            image = batch["image"].to(DEVICE)
            target = batch["mask"].to(DEVICE)

            pred = torch.sigmoid(model(image))[0, 0] >= 0.5
            truth = target[0, 0] >= 0.5

            intersection = (pred & truth).sum().item()
            pred_count = pred.sum().item()
            truth_count = truth.sum().item()

            dice = (
                2 * intersection / (pred_count + truth_count)
                if pred_count + truth_count else 1.0
            )

            corrected_rows.append({
                "fold": fold_id,
                "sample_id": batch["sample_id"][0],
                "dice": dice,
                "precision": (
                    intersection / pred_count if pred_count else 0.0
                ),
                "recall": (
                    intersection / truth_count if truth_count else 0.0
                ),
                "pred_foreground_fraction": pred.float().mean().item(),
                "true_foreground_fraction": truth.float().mean().item(),
            })

corrected_df = pd.DataFrame(corrected_rows)

for fold_id in (3, 4):
    print(f"\nFold {fold_id}: five lowest-Dice samples")
    print(
        corrected_df[corrected_df["fold"] == fold_id]
        .nsmallest(5, "dice")
        .round(4)
        .to_string(index=False)
    )

In [ ]:
# Diagnostic cell: visualize the five worst validation samples per fold

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader

checkpoint_dir = (
    PROJECT_ROOT / "outputs" / "cross_validation"
    / "early_stopping_experiment"
)

for fold_id in (3, 4):
    fold_info = next(
        fold for fold in cv_folds if fold["fold"] == fold_id
    )
    val_manifest = fold_info["val_manifest"].reset_index(drop=True)

    checkpoint = torch.load(
        checkpoint_dir / f"fold_{fold_id}_seed_42_best.pt",
        map_location=DEVICE,
        weights_only=False,
    )

    model = UNet(
        in_channels=3,
        out_channels=1,
        base_channels=checkpoint["params"]["base_channels"],
    ).to(DEVICE)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    worst_ids = (
        corrected_df[corrected_df["fold"] == fold_id]
        .nsmallest(5, "dice")["sample_id"]
        .tolist()
    )

    dataset = GlaSSegmentationDataset(val_manifest)
    loader = DataLoader(dataset, batch_size=1, shuffle=False)

    samples = {}

    with torch.no_grad():
        for batch in loader:
            sample_id = batch["sample_id"][0]

            if sample_id not in worst_ids:
                continue

            image = batch["image"].to(DEVICE)
            target = batch["mask"][0, 0].cpu().numpy()
            prediction = (
                torch.sigmoid(model(image))[0, 0] >= 0.5
            ).cpu().numpy()

            image_np = batch["image"][0].permute(1, 2, 0).numpy()

            samples[sample_id] = (
                image_np, target, prediction
            )

    fig, axes = plt.subplots(
        len(worst_ids), 3,
        figsize=(12, 4 * len(worst_ids)),
        squeeze=False,
    )
    fig.suptitle(
        f"Fold {fold_id}: worst validation predictions",
        fontsize=14,
    )

    for row, sample_id in enumerate(worst_ids):
        image_np, target, prediction = samples[sample_id]

        axes[row, 0].imshow(np.clip(image_np, 0, 1))
        axes[row, 0].set_title(f"{sample_id}: image")

        axes[row, 1].imshow(target, cmap="gray", vmin=0, vmax=1)
        axes[row, 1].set_title("Ground truth")

        axes[row, 2].imshow(prediction, cmap="gray", vmin=0, vmax=1)
        axes[row, 2].set_title(
            f"Prediction (Dice "
            f"{corrected_df.loc[corrected_df['sample_id'].eq(sample_id) & corrected_df['fold'].eq(fold_id), 'dice'].iloc[0]:.3f})"
        )

        for ax in axes[row]:
            ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

diagnostic_rows = []

checkpoint_dir = (
    PROJECT_ROOT / "outputs" / "cross_validation"
    / "early_stopping_experiment"
)

for fold_id in (3, 4):
    fold_info = next(
        f for f in cv_folds if f["fold"] == fold_id
    )
    val_manifest = fold_info["val_manifest"].reset_index(drop=True)

    checkpoint = torch.load(
        checkpoint_dir / f"fold_{fold_id}_seed_42_best.pt",
        map_location=DEVICE,
        weights_only=False,
    )

    model = UNet(
        in_channels=3,
        out_channels=1,
        base_channels=checkpoint["params"]["base_channels"],
    ).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    loader = DataLoader(
        GlaSSegmentationDataset(val_manifest),
        batch_size=1,
        shuffle=False,
        num_workers=0,
    )

    with torch.no_grad():
        for batch in loader:
            sample_id = batch["sample_id"][0]
            image = batch["image"].to(DEVICE)
            target = batch["mask"][0, 0].cpu().numpy() >= 0.5
            prob = torch.sigmoid(model(image))[0, 0].cpu().numpy()
            pred = prob >= 0.5

            tp = int(np.logical_and(pred, target).sum())
            fp = int(np.logical_and(pred, ~target).sum())
            fn = int(np.logical_and(~pred, target).sum())
            tn = int(np.logical_and(~pred, ~target).sum())

            dice = 2 * tp / max(2 * tp + fp + fn, 1)
            precision = tp / max(tp + fp, 1)
            recall = tp / max(tp + fn, 1)

            if precision < 0.60 and recall >= 0.70:
                pattern = "Over-segmentation: many false positives"
            elif recall < 0.60 and precision >= 0.70:
                pattern = "Under-segmentation: many missed target pixels"
            elif precision < 0.60 and recall < 0.60:
                pattern = "Mixed errors: false positives and missed target"
            elif recall > precision + 0.15:
                pattern = "Likely over-segmentation"
            elif precision > recall + 0.15:
                pattern = "Likely under-segmentation"
            else:
                pattern = "Mixed or comparatively balanced errors"

            diagnostic_rows.append({
                "fold": fold_id,
                "sample": sample_id,
                "dice": round(dice, 4),
                "precision": round(precision, 4),
                "recall": round(recall, 4),
                "true_fg_pct": round(100 * target.mean(), 2),
                "pred_fg_pct": round(100 * pred.mean(), 2),
                "false_positive_pct": round(100 * fp / pred.size, 2),
                "false_negative_pct": round(100 * fn / target.size, 2),
                "probability_min": round(float(prob.min()), 4),
                "probability_max": round(float(prob.max()), 4),
                "pattern": pattern,
            })

diagnostic_report = pd.DataFrame(diagnostic_rows)

for fold_id in (3, 4):
    print(f"\n{'=' * 90}")
    print(f"FOLD {fold_id}: FIVE WORST VALIDATION IMAGES")
    print("=" * 90)

    worst = (
        diagnostic_report[diagnostic_report["fold"] == fold_id]
        .nsmallest(5, "dice")
    )
    print(worst.to_string(index=False))

print("\nDiagnostic complete. No model weights were changed.")


In [ ]:
import inspect

print("=== CURRENT SEGMENTATION LOSS ===")
print(inspect.getsource(mixed_segmentation_loss))

print("\n=== CURRENT EARLY-STOPPING TRAINER ===")
print(inspect.getsource(train_search_trial_early_stopping))

In [ ]:
import torch
import torch.nn.functional as F

def make_tversky_mixed_loss(alpha=0.55, beta=0.45):
    """
    Weighted BCE + soft Tversky loss.
    alpha weights false positives.
    beta weights false negatives.
    """
    if alpha <= 0 or beta <= 0:
        raise ValueError("alpha and beta must be positive.")

    def loss_fn(logits, targets, positive_weight, bce_weight):
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=positive_weight,
        )

        probabilities = torch.sigmoid(logits)

        dims = (1, 2, 3)
        tp = (probabilities * targets).sum(dim=dims)
        fp = (probabilities * (1 - targets)).sum(dim=dims)
        fn = ((1 - probabilities) * targets).sum(dim=dims)

        tversky = (tp + 1e-7) / (
            tp + alpha * fp + beta * fn + 1e-7
        )

        tversky_loss = 1 - tversky.mean()

        return (
            bce_weight * bce
            + (1 - bce_weight) * tversky_loss
        )

    return loss_fn

print("Tversky loss factory defined.")
print("Baseline loss function remains:", mixed_segmentation_loss.__name__)

In [ ]:
from pathlib import Path
import pandas as pd

baseline_loss_fn = mixed_segmentation_loss
tversky_results = []

try:
    for setting_name, alpha, beta in [
        ("fp_weighted", 0.55, 0.45),
        ("fn_weighted", 0.45, 0.55),
    ]:
        mixed_segmentation_loss = make_tversky_mixed_loss(
            alpha=alpha,
            beta=beta,
        )

        for fold_id in (3, 4):
            output_dir = (
                PROJECT_ROOT / "outputs" / "cross_validation"
                / "tversky_experiment" / setting_name
                / f"fold_{fold_id}"
            )

            result = train_search_trial_early_stopping(
                params={
                    "learning_rate": 2e-4,
                    "weight_decay": 1e-4,
                    "base_channels": 16,
                    "loss_bce_weight": 0.5,
                },
                fold_id=fold_id,
                max_epochs=40,
                patience=6,
                min_delta=0.001,
                seed=42,
                checkpoint_dir=output_dir,
            )

            tversky_results.append({
                "setting": setting_name,
                "alpha": alpha,
                "beta": beta,
                "fold": fold_id,
                "best_dice": result["best_validation_dice"],
                "best_epoch": result["best_epoch"],
                "best_val_loss": result["best_validation_loss"],
                "epochs": result["epochs_completed"],
                "duration_seconds": result["duration_seconds"],
                "checkpoint": result["checkpoint_path"],
            })

finally:
    mixed_segmentation_loss = baseline_loss_fn

tversky_df = pd.DataFrame(tversky_results)

print("\n=== TVERSKY EXPERIMENT RESULTS ===")
print(tversky_df.round(4).to_string(index=False))

print("\n=== ORIGINAL LOSS RESTORED ===")
print(mixed_segmentation_loss.__name__)

In [ ]:

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from pathlib import Path

baseline_dir = (
    PROJECT_ROOT / "outputs" / "cross_validation"
    / "early_stopping_experiment"
)


experiment_dirs = {
    "BCE_Dice": (
        PROJECT_ROOT / "outputs" / "cross_validation"
        / "early_stopping_experiment"
    ),
    "Tversky_FP": (
        PROJECT_ROOT / "outputs" / "cross_validation"
        / "tversky_experiment" / "fp_weighted"
    ),
    "Tversky_FN": (
        PROJECT_ROOT / "outputs" / "cross_validation"
        / "tversky_experiment" / "fn_weighted"
    ),
}

difficult_samples = {
    3: ["train_24", "train_7", "train_68", "train_37", "train_14"],
    4: ["train_47", "train_61", "train_34", "train_30", "train_33"],
}

comparison_rows = []

for fold_id in (3, 4):
    fold_info = next(
        f for f in cv_folds if f["fold"] == fold_id
    )
    val_manifest = fold_info["val_manifest"].reset_index(drop=True)

    dataset = GlaSSegmentationDataset(val_manifest)
    loader = DataLoader(
        dataset, batch_size=1, shuffle=False, num_workers=0
    )

    for method, root in experiment_dirs.items():
        if method == "BCE_Dice":
            checkpoint_path = root / f"fold_{fold_id}_seed_42_best.pt"
        else:
            checkpoint_path = (
                root / f"fold_{fold_id}"
                / f"fold_{fold_id}_seed_42_best.pt"
            )
        if not checkpoint_path.exists():
            raise FileNotFoundError(
                f"Missing checkpoint: {checkpoint_path}"
            )

        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False,
        )

        if int(checkpoint["fold"]) != fold_id:
            raise ValueError(f"Fold mismatch in {checkpoint_path}")

        model = UNet(
            in_channels=3,
            out_channels=1,
            base_channels=int(checkpoint["params"]["base_channels"]),
        ).to(DEVICE)
        model.load_state_dict(checkpoint["model_state_dict"])
        model.eval()

        with torch.no_grad():
            for batch in loader:
                sample_id = batch["sample_id"][0]
                image = batch["image"].to(DEVICE)
                target = batch["mask"][0, 0].to(DEVICE) >= 0.5

                prediction = (
                    torch.sigmoid(model(image))[0, 0] >= 0.5
                )

                tp = int((prediction & target).sum().item())
                fp = int((prediction & ~target).sum().item())
                fn = int((~prediction & target).sum().item())

                dice = 2 * tp / max(2 * tp + fp + fn, 1)
                precision = tp / max(tp + fp, 1)
                recall = tp / max(tp + fn, 1)

                comparison_rows.append({
                    "fold": fold_id,
                    "method": method,
                    "sample": sample_id,
                    "dice": dice,
                    "precision": precision,
                    "recall": recall,
                    "pred_fg_pct": (
                        100 * prediction.float().mean().item()
                    ),
                    "true_fg_pct": (
                        100 * target.float().mean().item()
                    ),
                })

comparison_df = pd.DataFrame(comparison_rows)

# Overall results on the same validation images
print("=== OVERALL VALIDATION COMPARISON ===")
summary = (
    comparison_df.groupby(["fold", "method"])
    .agg(
        mean_dice=("dice", "mean"),
        mean_precision=("precision", "mean"),
        mean_recall=("recall", "mean"),
        mean_pred_fg_pct=("pred_fg_pct", "mean"),
    )
    .reset_index()
)
print(summary.round(4).to_string(index=False))

# Direct comparison on the previously difficult images
print("\n=== DIFFICULT IMAGE COMPARISON ===")
hard = comparison_df[
    comparison_df.apply(
        lambda r: r["sample"] in difficult_samples[int(r["fold"])],
        axis=1,
    )
].copy()

print(
    hard.sort_values(["fold", "sample", "method"])
    .round(4)
    .to_string(index=False)
)

# Best method per difficult image
print("\n=== BEST DICE METHOD PER DIFFICULT IMAGE ===")
best_per_image = hard.loc[
    hard.groupby(["fold", "sample"])["dice"].idxmax(),
    ["fold", "sample", "method", "dice"],
].sort_values(["fold", "sample"])

print(best_per_image.round(4).to_string(index=False))

print("\nInference comparison complete. No checkpoints were modified.")


In [ ]:

# Compare per-image Dice differences against the original loss.

wide = comparison_df.pivot_table(
    index=["fold", "sample"],
    columns="method",
    values="dice",
).reset_index()

for method in ("Tversky_FP", "Tversky_FN"):
    wide[f"{method}_delta"] = (
        wide[method] - wide["BCE_Dice"]
    )

print("=== PER-FOLD PAIRED COMPARISON ===")

for fold_id in (3, 4):
    part = wide[wide["fold"] == fold_id]

    print(f"\nFold {fold_id}, n={len(part)} images")

    for method in ("Tversky_FP", "Tversky_FN"):
        delta = part[f"{method}_delta"]

        print(
            f"{method}: "
            f"mean Dice change={delta.mean():+.4f}, "
            f"median={delta.median():+.4f}, "
            f"improved={(delta > 0).sum()}, "
            f"unchanged={(delta == 0).sum()}, "
            f"worsened={(delta < 0).sum()}"
        )

        print("  Biggest improvements:")
        print(
            part.nlargest(3, f"{method}_delta")[
                ["sample", "BCE_Dice", method, f"{method}_delta"]
            ].round(4).to_string(index=False)
        )

        print("  Biggest regressions:")
        print(
            part.nsmallest(3, f"{method}_delta")[
                ["sample", "BCE_Dice", method, f"{method}_delta"]
            ].round(4).to_string(index=False)
        )

print("\nAnalysis complete; no model weights were changed.")


In [ ]:

# Cell: Inspect saved training histories (read-only)

from pathlib import Path
import pandas as pd

PROJECT_ROOT = PROJECT_ROOT

history_files = {
    "BCE_Dice": {
        3: PROJECT_ROOT / "outputs/cross_validation/early_stopping_experiment/fold_3_seed_42_history.csv",
        4: PROJECT_ROOT / "outputs/cross_validation/early_stopping_experiment/fold_4_seed_42_history.csv",
    },
    "Tversky_FP": {
        3: PROJECT_ROOT / "outputs/cross_validation/tversky_experiment/fp_weighted/fold_3/fold_3_seed_42_history.csv",
        4: PROJECT_ROOT / "outputs/cross_validation/tversky_experiment/fp_weighted/fold_4/fold_4_seed_42_history.csv",
    },
    "Tversky_FN": {
        3: PROJECT_ROOT / "outputs/cross_validation/tversky_experiment/fn_weighted/fold_3/fold_3_seed_42_history.csv",
        4: PROJECT_ROOT / "outputs/cross_validation/tversky_experiment/fn_weighted/fold_4/fold_4_seed_42_history.csv",
    },
}

history_data = {}

for method, fold_files in history_files.items():
    for fold_id, path in fold_files.items():
        key = (method, fold_id)

        print("\n" + "=" * 75)
        print(f"Method: {method} | Fold: {fold_id}")
        print(f"File: {path}")

        if not path.exists():
            print("WARNING: File not found.")
            continue

        df = pd.read_csv(path)
        history_data[key] = df

        print("Rows:", len(df))
        print("Columns:", df.columns.tolist())
        print("\nFirst 3 rows:")
        display(df.head(3))

print("\nInspection complete. No models were trained or modified.")


In [ ]:

# Cell: Compare convergence, validation stability, and best epochs
# Read-only: no training, checkpoint writes, or test-set access.

import numpy as np
import pandas as pd

summary_rows = []
epoch_rows = []

for (method, fold_id), df in sorted(history_data.items()):
    df = df.sort_values("epoch").reset_index(drop=True)

    best_dice_idx = df["mean_image_dice"].idxmax()
    best_loss_idx = df["validation_loss"].idxmin()

    dice = df["mean_image_dice"].astype(float)
    val_loss = df["validation_loss"].astype(float)
    train_loss = df["train_loss"].astype(float)

    # Epoch-to-epoch changes measure how much validation performance fluctuates.
    dice_changes = dice.diff().dropna()

    summary_rows.append({
        "method": method,
        "fold": fold_id,
        "epochs_completed": len(df),
        "best_dice": dice.max(),
        "best_dice_epoch": int(df.loc[best_dice_idx, "epoch"]),
        "dice_at_last_epoch": dice.iloc[-1],
        "best_val_loss": val_loss.min(),
        "best_val_loss_epoch": int(df.loc[best_loss_idx, "epoch"]),
        "last_train_loss": train_loss.iloc[-1],
        "last_val_loss": val_loss.iloc[-1],
        "dice_epoch_std": dice.std(),
        "mean_abs_dice_change": dice_changes.abs().mean(),
        "largest_dice_drop": dice_changes.min(),
        "dice_improvements": int((dice_changes > 0).sum()),
        "dice_declines": int((dice_changes < 0).sum()),
    })

    for _, row in df.iterrows():
        epoch_rows.append({
            "method": method,
            "fold": fold_id,
            "epoch": int(row["epoch"]),
            "train_loss": row["train_loss"],
            "validation_loss": row["validation_loss"],
            "mean_image_dice": row["mean_image_dice"],
        })

summary_df = pd.DataFrame(summary_rows).sort_values(["fold", "method"])
epoch_comparison_df = pd.DataFrame(epoch_rows)

print("=== CONVERGENCE AND STABILITY SUMMARY ===")
display(
    summary_df.round(4).reset_index(drop=True)
)

print("\n=== VALIDATION DICE BY EPOCH ===")
dice_by_epoch = epoch_comparison_df.pivot(
    index="epoch", columns=["fold", "method"], values="mean_image_dice"
)
display(dice_by_epoch.round(4))

print("\n=== FINAL-EPOCH LOSS CHECK ===")
print(
    "A lower training loss alongside worsening validation Dice can be "
    "consistent with overfitting, but does not prove it by itself."
)
display(
    summary_df[
        [
            "method", "fold", "last_train_loss", "last_val_loss",
            "best_dice", "best_dice_epoch", "dice_at_last_epoch"
        ]
    ].round(4).reset_index(drop=True)
)

print("\nAnalysis complete. No model weights or files were changed.")


In [ ]:

# Cell: Visualize training and validation behavior
# Uses the six already-loaded histories. Does not train or save anything.

import matplotlib.pyplot as plt

for fold_id in (3, 4):
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

    for (method, current_fold), df in history_data.items():
        if current_fold != fold_id:
            continue

        df = df.sort_values("epoch")
        epochs = df["epoch"]

        axes[0].plot(
            epochs, df["mean_image_dice"],
            marker="o", markersize=3, label=method
        )
        axes[1].plot(
            epochs, df["train_loss"],
            marker="o", markersize=3, label=method
        )
        axes[2].plot(
            epochs, df["validation_loss"],
            marker="o", markersize=3, label=method
        )

    axes[0].set_title(f"Fold {fold_id}: Validation Dice")
    axes[1].set_title(f"Fold {fold_id}: Training Loss")
    axes[2].set_title(f"Fold {fold_id}: Validation Loss")

    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    axes[0].set_ylabel("Mean image Dice")
    axes[1].set_ylabel("Loss")
    axes[2].set_ylabel("Loss")

    plt.tight_layout()
    plt.show()

print("Plots generated in memory only; no files were saved.")


In [ ]:

# Read-only diagnostic: per-image validation Dice distribution

baseline = comparison_df[
    comparison_df["method"] == "BCE_Dice"
].copy()

for fold_id in (3, 4):
    values = baseline.loc[
        baseline["fold"] == fold_id, "dice"
    ].dropna()

    print(f"\n=== BASELINE FOLD {fold_id} ===")
    print(f"Images: {len(values)}")
    print(f"Mean Dice:   {values.mean():.4f}")
    print(f"Median Dice: {values.median():.4f}")
    print(f"Std Dice:    {values.std():.4f}")
    print(f"Minimum:     {values.min():.4f}")
    print(f"25th pct:    {values.quantile(0.25):.4f}")
    print(f"75th pct:    {values.quantile(0.75):.4f}")
    print(f"Maximum:     {values.max():.4f}")
    print(f"Images below 0.60: {(values < 0.60).sum()}")
    print(f"Images below 0.70: {(values < 0.70).sum()}")

print("\nRead-only analysis complete.")


In [ ]:

# Cell: Inspect the 8 hardest baseline validation images per fold
# Reuses comparison_df; no training or file modifications.

baseline = comparison_df[
    comparison_df["method"] == "BCE_Dice"
].copy()

columns = [
    "sample", "dice", "precision", "recall",
    "pred_fg_pct", "true_fg_pct"
]

for fold_id in (3, 4):
    part = (
        baseline[baseline["fold"] == fold_id]
        .sort_values("dice", ascending=True)
    )

    print(f"\n=== FOLD {fold_id}: LOWEST-DICE IMAGES ===")
    display(part[columns].head(8).round(4).reset_index(drop=True))

    print(f"\n=== FOLD {fold_id}: HIGHEST-DICE IMAGES ===")
    display(
        part[columns]
        .sort_values("dice", ascending=False)
        .head(3)
        .round(4)
        .reset_index(drop=True)
    )

print("\nRead-only analysis complete.")


In [ ]:

# Cell: Numerical error analysis only
# No images displayed, uploaded, or saved.

import pandas as pd

target_samples = ["train_47", "train_61", "train_34", "train_12"]

baseline = comparison_df[
    (comparison_df["method"] == "BCE_Dice")
    & (comparison_df["fold"] == 4)
    & (comparison_df["sample"].isin(target_samples))
].copy()

if baseline.empty:
    print("No matching rows found. Check comparison_df and the sample names.")
else:
    baseline["estimated_pixels"] = 256 * 256

    baseline["predicted_fg_pixels"] = (
        baseline["pred_fg_pct"] / 100
        * baseline["estimated_pixels"]
    ).round().astype(int)

    baseline["true_fg_pixels"] = (
        baseline["true_fg_pct"] / 100
        * baseline["estimated_pixels"]
    ).round().astype(int)

    baseline["foreground_area_difference_pct"] = (
        baseline["pred_fg_pct"] - baseline["true_fg_pct"]
    )

    baseline["false_positive_pixels_approx"] = (
        baseline["precision"].apply(
            lambda p: None if pd.isna(p) or p <= 0 else p
        )
    )

    print("=== FOLD 4: LOCAL NUMERICAL DIAGNOSTICS ===")
    display(
        baseline[
            [
                "sample", "dice", "precision", "recall",
                "predicted_fg_pixels", "true_fg_pixels",
                "foreground_area_difference_pct"
            ]
        ].sort_values("dice").round(4).reset_index(drop=True)
    )

    print(
        "\nNote: foreground pixel counts are estimates from rounded "
        "percentages. Exact FP/FN counts require the original local "
        "prediction and ground-truth mask arrays."
    )

print("\nNo images were opened, displayed, uploaded, or saved.")


In [ ]:

# Read-only notebook state check
from pathlib import Path

project_root = PROJECT_ROOT

checkpoint = (
    project_root
    / "outputs/cross_validation/early_stopping_experiment/"
      "fold_4_seed_42_best.pt"
)

print("=== EXISTING CHECKPOINT ===")
print("Exists:", checkpoint.exists())
if checkpoint.exists():
    print(f"Size: {checkpoint.stat().st_size / (1024**2):.2f} MB")

print("\n=== RELEVANT VARIABLES IN MEMORY ===")
keywords = ("model", "loader", "dataset", "fold", "device", "transform")

found = []
for name, obj in list(globals().items()):
    if name.startswith("_") or name in {"In", "Out"}:
        continue
    if any(word in name.lower() for word in keywords):
        try:
            description = type(obj).__name__
            if isinstance(obj, (list, tuple, dict)):
                description += f" (length={len(obj)})"
            found.append((name, description))
        except Exception:
            pass

for name, description in sorted(found):
    print(f"{name}: {description}")

print("\nState check complete. Nothing was loaded or modified.")


In [ ]:

# Inspect existing objects without changing them or loading files.

print("=== BASELINE FOLD 4 RESULT KEYS ===")
print(list(baseline_fold4_seed42.keys()))

print("\n=== VALIDATION DATASET ===")
print("Type:", type(validation_dataset).__name__)
print("Length:", len(validation_dataset))

if len(validation_dataset) > 0:
    sample = validation_dataset[0]
    print("Sample keys:", list(sample.keys()))
    print("Sample ID:", sample.get("sample_id"))
    print("Image tensor shape:", tuple(sample["image"].shape))
    print("Mask tensor shape:", tuple(sample["mask"].shape))

print("\n=== VALIDATION LOADER ===")
print("Number of batches:", len(validation_loader))

print("\n=== CHECKPOINT FOLD METADATA ===")
print("Expected fold: 4")
print("Expected seed: 42")
print("Checkpoint exists:", checkpoint.exists())

print("\nInspection complete. No images were displayed or saved.")


In [ ]:

# Read-only: verify Fold 4 validation membership
# No training, image display, or file writes.

print("=== FOLD METADATA STRUCTURE ===")
fold4_info = next(f for f in cv_folds if f["fold"] == 4)
print("Fold 4 keys:", list(fold4_info.keys()))

for key, value in fold4_info.items():
    if hasattr(value, "columns"):
        print(f"\n{key}: DataFrame, shape={value.shape}")
        if "sample_id" in value.columns:
            print("Sample IDs:", value["sample_id"].astype(str).tolist())
        elif "image_id" in value.columns:
            print("Image IDs:", value["image_id"].astype(str).tolist())
    elif isinstance(value, (list, tuple, set)):
        print(f"\n{key}: {type(value).__name__}, length={len(value)}")
        print("First entries:", list(value)[:8])
    elif key != "fold":
        print(f"\n{key}: {type(value).__name__}")

print("\n=== CURRENT VALIDATION DATASET IDS ===")
current_ids = [
    str(validation_dataset[i]["sample_id"])
    for i in range(len(validation_dataset))
]
print("Count:", len(current_ids))
print("First 10:", current_ids[:10])

print("\n=== FOLD 4 CHECKPOINT METADATA ===")
print("Result fold:", baseline_fold4_seed42["fold"])
print("Result seed:", baseline_fold4_seed42["seed"])
print("Best epoch:", baseline_fold4_seed42["best_epoch"])
print("Recorded best validation Dice:",
      baseline_fold4_seed42["best_validation_dice"])

print("\nRead-only inspection complete.")


In [ ]:

# Build the correct Fold 4 validation dataset — read-only evaluation setup

import torch
from torch.utils.data import DataLoader

fold4_info = next(f for f in cv_folds if f["fold"] == 4)
fold4_val_manifest = fold4_info["val_manifest"].copy()

fold4_val_dataset = GlaSSegmentationDataset(fold4_val_manifest)
fold4_val_loader = DataLoader(
    fold4_val_dataset,
    batch_size=SEARCH_CONFIG["batch_size"],
    shuffle=False,
    num_workers=0
)

fold4_val_ids = [
    str(fold4_val_dataset[i]["sample_id"])
    for i in range(len(fold4_val_dataset))
]

print("=== FOLD 4 VALIDATION VERIFICATION ===")
print("Expected manifest samples:", len(fold4_val_manifest))
print("Dataset samples:", len(fold4_val_dataset))
print("Validation batches:", len(fold4_val_loader))
print("IDs match manifest:",
      set(fold4_val_ids) == set(fold4_val_manifest["sample_id"].astype(str)))
print("IDs:", fold4_val_ids)

print("\n=== SPLIT SAFETY CHECK ===")
fold4_train_ids = set(fold4_info["train_manifest"]["sample_id"].astype(str))
fold4_val_id_set = set(fold4_val_ids)
print("Train/validation overlap:", sorted(fold4_train_ids & fold4_val_id_set))
print("TestA/TestB accessed: No")
print("Files written: No")


In [ ]:

# Fold 4 baseline: local threshold-sensitivity analysis
# Inference only. No training, image display, or file writes.

from pathlib import Path
import numpy as np
import pandas as pd
import torch

project_root = PROJECT_ROOT
checkpoint_path = (
    project_root
    / "outputs"
    / "cross_validation"
    / "early_stopping_experiment"
    / "fold_4_seed_42_best.pt"
)

assert checkpoint_path.exists(), f"Checkpoint not found: {checkpoint_path}"

# Recreate the model using the same architecture.
threshold_model = UNet(
    in_channels=3,
    out_channels=1,
    base_channels=16
).to(DEVICE)

try:
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=True
    )
except TypeError:
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

threshold_model.load_state_dict(checkpoint["model_state_dict"])
threshold_model.eval()

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
rows = []

with torch.inference_mode():
    for batch in fold4_val_loader:
        images = batch["image"].to(DEVICE)
        masks = batch["mask"].to(DEVICE)
        sample_ids = batch["sample_id"]

        probabilities = torch.sigmoid(threshold_model(images))

        for i, sample_id in enumerate(sample_ids):
            target = masks[i] >= 0.5
            true_pixels = target.sum().item()

            for threshold in thresholds:
                prediction = probabilities[i] >= threshold
                pred_pixels = prediction.sum().item()
                intersection = (prediction & target).sum().item()

                dice = (
                    2.0 * intersection / (pred_pixels + true_pixels)
                    if pred_pixels + true_pixels > 0
                    else 1.0
                )

                rows.append({
                    "sample_id": str(sample_id),
                    "threshold": threshold,
                    "dice": dice,
                    "pred_fg_pct": 100.0 * pred_pixels / target.numel(),
                    "true_fg_pct": 100.0 * true_pixels / target.numel()
                })

threshold_df = pd.DataFrame(rows)

summary = (
    threshold_df.groupby("threshold")
    .agg(
        mean_dice=("dice", "mean"),
        std_dice=("dice", "std"),
        median_dice=("dice", "median"),
        min_dice=("dice", "min"),
        mean_pred_fg_pct=("pred_fg_pct", "mean")
    )
    .reset_index()
)

print("=== CHECKPOINT METADATA ===")
print("Fold:", checkpoint.get("fold"))
print("Seed:", checkpoint.get("seed"))
print("Best epoch:", checkpoint.get("best_epoch"))
print("Recorded best validation Dice:",
      checkpoint.get("best_validation_dice"))

print("\n=== FOLD 4 THRESHOLD SUMMARY ===")
print(summary.round(4).to_string(index=False))

print("\n=== BEST THRESHOLD ON THIS VALIDATION FOLD ===")
best_row = summary.loc[summary["mean_dice"].idxmax()]
print(best_row.round(4).to_string())

print("\n=== BASELINE THRESHOLD 0.50 ===")
baseline_row = summary.loc[summary["threshold"] == 0.50].iloc[0]
print(baseline_row.round(4).to_string())

print("\n=== PER-IMAGE DICE AT BEST THRESHOLD ===")
best_threshold = float(best_row["threshold"])
per_image = (
    threshold_df[threshold_df["threshold"] == best_threshold]
    .sort_values("dice")
)
print(per_image[["sample_id", "dice", "pred_fg_pct", "true_fg_pct"]]
      .round(4).to_string(index=False))

print("\nInference complete. No images displayed; no files written.")
